# Eligible fork elig0 · Variant M core + midx final layer, without Engine 2, upstream datasets only

> **Eligibility build elig0** (scripts/make_pub_variant.py with ELIG=0; the audit and fixes F1-F8 are in
> docs/fork_eligibility.md of our repository). Changed against casmi26-pubm:
> - **F1, upstream datasets.** The CC BY-NC-SA container dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets, the
>   natural-product knowledge table, the panel benchmark and sanpier/catalogue-experiment-saner are no longer attached;
>   our bryantyu/casmi26-code and casmi26-models are not attached either: with OURS = False no cell reads them (PF-syn, cell 11, is off, and so is ISO, cell 17).
>   The pipeline reads the original datasets: ahmedberatozer casmi26-v4b-models, casmi26-v3-models, casmi26-fpnet-full1,
>   casmi26-v2-pool, casmi26-iceberg and casmi26-glacier; dmitriigluzdov casmi26-pubchem-popularity-prior and
>   casmi26-fold-safe-fpnet; sanpier/casmi-spectrum-model-v1; metric/rdkit-2026-3-3-wheel. The frozen pipeline's display
>   cells 9, 12, 14, 16 and 18 are skipped. Before the pipeline runs, cell 8 checks the mounted files against the hashes
>   of the casmi26-pubm run (files up to 32 MiB in full, larger ones by size and their first and last 4 MiB) and hashes
>   the larger files of the re-mounted datasets in full (log only). The pipeline's receipt carries an `elig` field.
> - **F2a, null Engine 2.** Engine 2 (prvsiyan, megayak, sergiomalv and franciscoangulo inputs) is not run. Every reader
>   of its lists gets the pipeline's own: Engine 1's top 40 until the PubChem merge (read by the ICEBERG / GLACIER
>   input union), then the main list (Engine 1 + the gated PubChem channel) followed by Engine 1's tail (frozen cells 32,
>   39 and 45; null_engine.py).
> - **F3.** No PubChem depositor-source bonus (SRC_BITS 0: that bit groups the NIST Mass Spectrometry Data Center's
>   deposits with MassBank / MoNA) and SPEC_W 0 in the final layer, which then does not load the flag column. The
>   PubChem channel's delta / growth supplement stores still memory-map their source-mask arrays, which no score reads.
>   NP_W / NP_BONUS stay: NPAtlas is on the host's list.
> - **F4.** The knowledge table's external reference spectra are not added to the library (frozen cell 36).
> - Sections 6 and 8 name the datasets attached here. Not part of the fixes: the ISO cell (cell 17, off with
>   ISO_ALPHA 0 in both builds) and two of its cell-1 knobs (ISO_COS_W, ISO_FREEZE_LEADER) come from a newer generator
>   version than casmi26-pubm's.
>
> **Still open (this build is not eligible until they are closed):** F5a, the ICEBERG / GLACIER runner glue
> (ice_runner.py, fuse.py, gl_runner.py, gl_fuse.py, shim/ in ahmedberatozer casmi26-iceberg / casmi26-glacier) carries
> no OSI licence yet: an owner question to its author. F6, a host question: do reconstructed, not bit-exact retrains with
> public code (Engine 1's ranker_v4b and fe_v4 tables, saner, our own G4 / G5) meet the reproducibility rule, and does a
> PubChem depositor flag naming the NIST MS Data Center count as using NIST? F7, a demonstrated retrain of saner's
> spectrum model. F8, the licence and attribution package: an MIT LICENSE for our code, a NOTICE for the Apache-2.0 and
> MIT sources, the fork's ATTRIBUTION and reproduce files, and the Kaggle licences of our datasets (G19).

> **This version** (made by scripts/make_pub_variant.py; every setting is in cell 1): the final layer (cell 16) is
> cell 15 of baidalinadilzhan/casmi26-midx-exact-dedup-fill (team LB 0.479: documentation prior 0.10 / 0.05 (0 here, F3) / 0.02 on
> ranks 2-25, library bonus only from a best library match of 0.87, the old leader pinned at rank 2 after a
> fragment-check leader move, metric-exact deduplication, empty slots filled with scored outside candidates; no
> formula-gated swaps) plus our PF_MID_W hook; cell 11 scores the candidates with our PF-syn PeakFormer; cell 17 is
> our ISO isomer reranker. The notes below are the forked notebook's and describe its earlier final layer
> (w_pop = 0.14, formula-gated isomer swaps), which this version does not run.

Building directly on our proven personal best **LB 0.478 (Global Rank 53 / 4,399, Top 1.20%)**:
1. **Calibrated Variant M Core ($w_{\text{top}} = 0.25, w_{\text{mid}} = 0.75, \text{rescue} = 0$)**:
   - Preserves the proven optimal weights: library leader preservation at Rank 1 ($w_{\text{top}} = 0.25$) and high-resolution neural discrimination ($w_{\text{mid}} = 0.75$) across ranks 2–25.
   - Strict 5-shield physical fragment re-scorer on V2: `len(spec) >= 3`, `margin >= 0.095`, `votes >= 0.75`, `nmargin >= 0.23`, `depth <= 6`.
   - Guaranteed Rank 1 Lock: Rank 1 is 100% frozen and protected against false-positive demotions.
2. **Calibrated PubChem Documentation & Spectral Prior on Ranks 2–25 ($w_{\text{pop}} = 0.14, w_{\text{spec}} = 0.05$)**:
   - Elevated documentation weighting to bridge 2–3 rank gaps for authentic biological metabolites over synthetic decoys.
   - Fine-tuned on the ground-truth benchmark panel (+0.0086 MRR gain).
3. **Formula-Gated Constitutional Isomer Alignment (Ranks 2–10)**:
   - For adjacent candidates in ranks 2–10 sharing the exact same chemical formula, aligns order with physical fragment explanation evidence.
4. **In-Pool InChIKey14 Skeletal Deduplication**:
   - Reclaims redundant stereoisomer slots within the top 25, backfilling with distinct molecular connectivity skeletons.


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
ELIG = 0                      # eligibility build (docs/fork_eligibility.md): 0 = F1 upstream mounts + F2a null
                              # Engine 2 + F3 no NIST depositor flag + F4 no external spectra; 1 = F2b (reserved)
VARIANT = 'CUSTOM'
#   V0  baseline 0.465 answers, no neural term
#   V1  public 0.466 recipe: neural reciprocal-rank fusion, weight 0.25 everywhere
#   V2  leader-safe consensus: rank 1 as V1, ranks 2-25 at neural weight 0.50   (upstream default)
#   V3  V2 + tail rescue: outside candidates may take the last 3 places by neural score
#   V4  bold consensus: rank 1 as V1, ranks 2-25 at neural weight 1.00
#   'CUSTOM' uses the verified 0.477 optimal weights (w_top=0.25, w_mid=0.75, rescue=0)
CUSTOM = dict(w_top=0.25, w_mid=0.75, rescue=0)

RRF_OFFSET = 3.0              # k in 1/(k + rank), as in the public recipe
COMMIT_MODE = 'smoke'         # 'smoke': commit run validates 12 visible molecules and asserts all stages;
                              #          the scoring rerun on hidden test executes in full
OUTSIDERS_PER_MOLECULE = 20   # candidate pool for the tail rescue (V3, or CUSTOM with rescue > 0)
TIME_GUARD_HOURS = 7.5        # optional stages are skipped once the session has run this long
QUIET_LOGS = True             # one line per pipeline stage here; full logs stay in /kaggle/working/*.log
ENABLE_TAIL_SLOT = True
PHYSICAL_POST_SECONDS = 1500

# Physical switches the fragment check; PROVENANCE the provenance reranker; neural model: Saner Turhaner v1
PHYSICAL = True
PROVENANCE = False
# Variant M: fragment check reads variants/submission_V2.csv;
# merge cell keeps its rank-1 moves unblocked on the CUSTOM order
FRAG_ON_V2 = True
# InChIKey-14 deduplication: removes redundant stereoisomers within candidate pool
ENABLE_K14_DEDUP = True

# Ours: PF-syn PeakFormer (bryantyu/casmi26-models pf_tpu_f.pt) joins the neural term before the fusion
OURS = False                  # False: the unchanged saner-only neural term
PF_W = 0.5                    # per molecule neural_score := (1 - PF_W) * z(saner) + PF_W * z(PF-syn f.z)
OURS_MODE = 'blend'           # 'blend', 'replace' (neural term = z(PF-syn f.z) alone) or 'columns' (neural_score
                              # unchanged; only the columns neural_score_saner / pf_fz, read by PF_MID_W and ISO)
PF_MAX_SECONDS = 1200         # PF-syn scoring budget; it is also skipped past 8.0 h of session time

# ── Final layer (baidalinadilzhan/casmi26-midx-exact-dedup-fill, team LB 0.479) ─────────────────────────────
# Documentation-prior weights on ranks 2-25 (MID: 0.10 / 0.05 / 0.02), spectral-library bonus only for molecules whose
# best library match reaches LIB_HIT_GATE, the pipeline's leader pinned at rank 2 after a fragment-check leader move,
# metric-exact deduplication (the scorer's own key) and empty slots filled with scored outside candidates.
POP_W, SPEC_W, NP_W = 0.10, 0.0, 0.02   # elig0 (F3): SPEC_W 0.0 (was 0.05; spectral-library depositor flag
                                        # includes the NIST MS Data Center); NP_W kept (NPAtlas is host-allowed)
LIB_HIT_GATE = 0.87
PIN_OLD_LEADER = True
OUTSIDER_FILL = True
PF_MID_W = 0.0                # ours: PF-syn in the ranks 2-25 neural rank of the final layer, per molecule
                              # (1 - PF_MID_W) z(neural_score_saner) + PF_MID_W z(pf_fz); 0 = midx exactly

# Ours: ISO isomer reranker on the final answer lists (the cell after the final layer)
ISO_ALPHA = 0.0               # per list z(-log rank) + ISO_ALPHA * z(PF-syn f.z); 0 = off (submission.csv untouched)
ISO_LIB_SKIP = 0.9            # molecules whose best library match reaches this keep their list
ISO_KEEP_TRAIN = True         # candidates whose InChIKey14 is in train.parquet keep their exact positions
ISO_SCOPE = 'formula'         # 'formula': only the rank-1 candidate's formula is re-sorted; 'all': every candidate
ISO_COS_W = 0.0               # > 0: the PF term is z(f.z) + ISO_COS_W * z(PF-syn cosine pf_cos, pf_scores.parquet)
ISO_FREEZE_LEADER = False     # True: rank 1 never moves (ranks 2-25 only; scope 'formula' keeps the leader's formula)


In [ ]:
# ── Session: paths, run mode, design tokens and small display helpers ─────────────────────────
import os, sys, re, io, json, time, math, base64, shutil, hashlib, subprocess, collections, warnings
from pathlib import Path

NB_T0 = time.time()
warnings.filterwarnings('ignore')
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
CODE = ROOT / 'p2s_code'; CODE.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import display, HTML, FileLink

# Design tokens: chart ink and a fixed categorical order (identity never follows rank).
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
GRID, AXIS, SURFACE, CARD = '#e1e0d9', '#c3c2b7', '#fcfcfb', '#f7f8fb'
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    '#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948')
QUIET_GRAY = '#c9c8c1'
ROLE = {'main': BLUE, 'engine2': ORANGE, 'pubchem': AQUA, 'new': VIOLET}
FONT = "system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.size': 10.5, 'axes.titlesize': 12.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.titlecolor': INK, 'axes.titlepad': 10, 'axes.labelcolor': INK2, 'axes.edgecolor': AXIS,
    'axes.linewidth': 0.8, 'xtick.color': AXIS, 'ytick.color': AXIS, 'xtick.labelcolor': INK2,
    'ytick.labelcolor': INK2, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'grid.linestyle': '-', 'axes.axisbelow': True, 'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'legend.fontsize': 10, 'figure.dpi': 110, 'savefig.dpi': 160,
    'savefig.bbox': 'tight'})
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass


def tiles(items):
    """A row of stat tiles: items = [(label, value, note), ...]."""
    cells = ''.join(
        f'<div style="display:inline-block;vertical-align:top;min-width:150px;margin:0 10px 10px 0;'
        f'padding:12px 16px;border-radius:12px;background:#ffffff;border:1px solid rgba(11,11,11,0.10);">'
        f'<div style="font-size:12px;color:{INK2};">{label}</div>'
        f'<div style="font-size:25px;font-weight:650;color:{INK};line-height:1.3;">{value}</div>'
        f'<div style="font-size:11.5px;color:{MUTED};">{note}</div></div>' for label, value, note in items)
    display(HTML(f'<div style="font-family:{FONT};margin:4px 0 2px;">{cells}</div>'))


def note(title, text, color=VIOLET):
    display(HTML(f'<div style="font-family:{FONT};border-left:4px solid {color};background:{CARD};'
                 f'padding:10px 14px;border-radius:8px;margin:8px 0;font-size:13.5px;line-height:1.5;">'
                 f'<b style="color:{INK};">{title}</b> <span style="color:{INK2};">{text}</span></div>'))


def table_html(frame, highlight=None, numeric=()):
    """Plain, readable HTML table; numbers right-aligned; `highlight` = index label of a row to emphasise."""
    align = {c: 'right' if c in set(numeric) else 'left' for c in frame.columns}
    head = ''.join(f'<th style="text-align:{align[c]};padding:7px 12px;border-bottom:1px solid {AXIS};'
                   f'color:{INK2};font-weight:600;background:transparent;">{c}</th>' for c in frame.columns)
    body = []
    for idx, row in frame.iterrows():
        on = highlight is not None and idx == highlight
        style = 'background:#efedfb;font-weight:600;' if on else 'background:transparent;'
        body.append(f'<tr style="{style}">' + ''.join(
            f'<td style="text-align:{align[c]};padding:7px 12px;border-bottom:1px solid {GRID};color:{INK};'
            f'font-variant-numeric:tabular-nums;">{v}</td>' for c, v in zip(frame.columns, row.values)) + '</tr>')
    display(HTML(f'<table style="font-family:{FONT};font-size:13px;border-collapse:collapse;margin:6px 0 10px;">'
                 f'<thead><tr>{head}</tr></thead><tbody>{"".join(body)}</tbody></table>'))


def minutes(seconds):
    return f'{seconds / 60:,.1f} min' if seconds >= 60 else f'{seconds:,.0f} s'


def gpu_name():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=30).stdout.strip()
        return ' + '.join(x.strip() for x in out.splitlines() if x.strip()) or 'none'
    except Exception:
        return 'none'


# Run mode: the commit run sees the visible format example; the scoring rerun sees the hidden test.
COMP = next((p for p in (INPUT / 'competitions/enveda-CASMI26-molecule-id-mass-spectra',
                         INPUT / 'enveda-CASMI26-molecule-id-mass-spectra',
                         INPUT / 'enveda-casmi26-molecule-id-mass-spectra')
             if all((p / f).is_file() for f in ('train.parquet', 'test.parquet', 'sample_submission.csv'))), None)
assert COMP is not None, 'Attach the competition data (enveda-CASMI26-molecule-id-mass-spectra)'
_ids = pd.read_parquet(COMP / 'test.parquet', columns=['spectrum_id']).spectrum_id.astype(str)
TEST_SIGNATURE = hashlib.md5(','.join(sorted(_ids)).encode()).hexdigest()
del _ids
IS_RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN')) or TEST_SIGNATURE != '654e030e4173780e945252b4a5adf70b'
RUN_PIPELINE = IS_RERUN or COMMIT_MODE != 'fast'
SHOW = not IS_RERUN                      # figures are drawn for readers; the scoring rerun skips them
SAMPLE_IDS = pd.read_csv(COMP / 'sample_submission.csv', dtype=str).molecule_id.tolist()
assert VARIANT in ('V0', 'V1', 'V2', 'V3', 'V4', 'CUSTOM'), VARIANT

GPU = gpu_name()
MODE = 'Scoring rerun' if IS_RERUN else ('Commit · smoke run' if RUN_PIPELINE else 'Commit · fast path')
tiles([('Run mode', MODE, 'hidden test' if IS_RERUN else 'visible format example'),
       ('Variant', VARIANT, 'set in the configuration cell'),
       ('Accelerator', GPU.split(' + ')[0] if GPU != 'none' else 'none', GPU if ' + ' in GPU else 'GPU required'),
       ('Python', '.'.join(map(str, sys.version_info[:3])), 'pipeline expects 3.12'),
       ('Answer rows', f'{len(SAMPLE_IDS):,}', 'molecules in sample_submission')])

## 1. The field

On 2026-10-28 07:48 UTC the Code tab listed 974 public notebooks for this competition, 678 of them
with a public score. Of those, 112 score 0.460 or more, and 103 of these 112 attach the
same dataset, Dmitrii Gluzdov's v4g reproduction assets (Figure 1). Most of the top of the Code tab is one pipeline
with different constants.

The leaderboard on 2026-10-27 21:18 UTC had 4,194 teams. Of these, 130 sat at exactly 0.472,
169 at 0.468 and 190 at 0.464, the scores that 10, 7 and 28 public
notebooks display. The board does not record what each team submitted, so this matches scores and does not prove
copying. The gold line was rank 18 at 0.477, and first place was 0.513. No member of the top
ten teams is the author of any of the 974 public notebooks.

In [ ]:
# House style for our own figures (engine/33_style.py at repo commit 4ebb73e9, without its ink and grid
# colours, so the upstream tiles and tables keep theirs). A later cell imports another `apply`; ours stays as style_apply.
ACCENT = "#d9480f"      # the one colour that means "this is the point"
SECOND = "#1c7ed6"      # used only when two things must be told apart
PALETTE = [ACCENT, SECOND, "#495057", "#adb5bd", "#2b8a3e", "#862e9c"]


def apply() -> None:
    """Set matplotlib defaults. Safe to call more than once."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (9, 4.6), "figure.dpi": 110, "savefig.dpi": 160,
        "figure.facecolor": "white", "axes.facecolor": "white",
        "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "semibold", "axes.titlelocation": "left",
        "axes.labelsize": 11, "axes.labelcolor": INK, "text.color": INK,
        "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 10, "ytick.labelsize": 10,
        "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
        "axes.prop_cycle": mpl.cycler(color=PALETTE),
        "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 10,
    })


def label_point(ax, x, y, text: str, color: str = ACCENT) -> None:
    """Print the number that matters on the chart, next to the point it describes."""
    ax.annotate(text, (x, y), xytext=(6, 6), textcoords="offset points", color=color,
                fontsize=11, fontweight="semibold")


def first_screen(claim: str, number: str, beats: str | None = None, runtime: str | None = None,
                 accel: str | None = None) -> str:
    """The markdown every busyaprime notebook opens with. Returns it so a builder can write it into cell 0.

    claim   one sentence: what this notebook shows that the reader did not know
    number  the result, with its metric and where it was measured (public LB, CV on stated folds)
    beats   the public work it beats, with that work's number and a link
    runtime wall clock on the stated accelerator
    """
    rows = [f"| result | **{number}** |"]
    if beats:
        rows.append(f"| beats | {beats} |")
    if runtime:
        rows.append(f"| runtime | {runtime}{' on ' + accel if accel else ''} |")
    return f"{claim}\n\n| | |\n|---|---|\n" + "\n".join(rows) + "\n"
style_apply = apply

In [ ]:
# The field as the Code tab showed it on 2026-10-28 07:48 UTC: every public notebook with a displayed score.
FIELD = {"taken_utc":"2026-10-28 07:48","notebooks_listed":974,"scored":[[0.475,44,"E",1,1,"2026-10-22"],[0.472,92,"M",1,1,"2026-10-23"],[0.472,66,"E",1,1,"2026-10-25"],[0.472,8,"C",1,1,"2026-10-27"],[0.472,1,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.471,0,"C",1,1,"2026-10-27"],[0.471,0,"C",1,1,"2026-10-28"],[0.471,0,"C",1,1,"2026-10-28"],[0.47,0,"C",1,1,"2026-10-26"],[0.47,4,"C",1,1,"2026-10-26"],[0.468,61,"E",1,0,"2026-09-28"],[0.468,98,"M",1,1,"2026-10-13"],[0.468,0,"C",1,1,"2026-10-26"],[0.468,0,"C",1,1,"2026-10-26"],[0.468,0,"C",1,1,"2026-10-26"],[0.468,0,"E",1,1,"2026-10-26"],[0.468,0,"C",1,1,"2026-10-27"],[0.467,59,"E",1,1,"2026-10-25"],[0.467,1,"C",1,1,"2026-10-25"],[0.467,0,"E",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,1,"C",0,0,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,3,"C",1,1,"2026-10-26"],[0.466,2,"E",1,1,"2026-09-17"],[0.466,80,"E",1,1,"2026-10-21"],[0.466,21,"C",1,1,"2026-10-24"],[0.466,6,"E",1,0,"2026-10-24"],[0.466,1,"E",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-25"],[0.466,0,"M",1,1,"2026-10-25"],[0.466,1,"C",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,0,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-27"],[0.465,182,"M",1,0,"2026-10-16"],[0.465,22,"C",1,0,"2026-10-24"],[0.465,0,"C",1,0,"2026-10-24"],[0.465,3,"C",1,0,"2026-10-24"],[0.465,1,"C",1,0,"2026-10-24"],[0.465,0,"C",1,1,"2026-10-25"],[0.465,1,"C",1,1,"2026-10-25"],[0.465,1,"C",1,1,"2026-10-25"],[0.465,0,"M",1,1,"2026-10-26"],[0.465,0,"C",1,0,"2026-10-26"],[0.465,0,"C",1,1,"2026-10-26"],[0.465,0,"E",0,1,"2026-10-28"],[0.464,201,"M",1,0,"2026-09-15"],[0.464,3,"C",1,0,"2026-10-22"],[0.464,2,"C",1,0,"2026-10-22"],[0.464,1,"C",1,0,"2026-10-22"],[0.464,1,"C",1,0,"2026-10-22"],[0.464,0,"C",1,0,"2026-10-22"],[0.464,0,"C",1,0,"2026-10-22"],[0.464,0,"C",1,0,"2026-10-22"],[0.464,2,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,3,"C",1,0,"2026-10-23"],[0.464,5,"C",1,0,"2026-10-23"],[0.464,10,"M",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,2,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-24"],[0.464,1,"C",1,0,"2026-10-25"],[0.464,0,"C",1,0,"2026-10-25"],[0.464,0,"C",1,0,"2026-10-25"],[0.464,0,"C",1,0,"2026-10-26"],[0.464,0,"C",1,0,"2026-10-26"],[0.464,0,"C",1,0,"2026-10-26"],[0.464,1,"C",1,1,"2026-10-26"],[0.464,1,"E",0,1,"2026-10-28"],[0.463,5,"M",0,0,"2026-10-21"],[0.463,7,"M",0,0,"2026-10-22"],[0.463,0,"C",0,0,"2026-10-22"],[0.463,0,"C",0,0,"2026-10-23"],[0.463,0,"C",1,0,"2026-10-24"],[0.463,0,"C",1,0,"2026-10-26"],[0.463,0,"C",1,1,"2026-10-26"],[0.463,0,"E",0,0,"2026-10-27"],[0.462,1,"C",1,0,"2026-10-26"],[0.462,0,"C",1,0,"2026-10-27"],[0.461,0,"C",1,0,"2026-10-26"],[0.461,0,"C",1,0,"2026-10-27"],[0.46,1,"C",0,0,"2026-10-19"],[0.46,0,"C",1,0,"2026-10-24"],[0.46,0,"C",1,0,"2026-10-27"],[0.459,1,"C",0,0,"2026-10-21"],[0.459,0,"C",1,0,"2026-10-22"],[0.459,0,"C",1,0,"2026-10-26"],[0.459,0,"C",1,1,"2026-10-28"],[0.458,0,"C",1,0,"2026-10-23"],[0.457,0,"C",0,0,"2026-10-23"],[0.457,0,"C",1,0,"2026-10-24"],[0.457,0,"C",1,0,"2026-10-25"],[0.457,0,"C",1,0,"2026-10-26"],[0.456,155,"M",0,0,"2026-09-15"],[0.456,9,"E",0,0,"2026-10-17"],[0.456,77,"C",0,0,"2026-10-18"],[0.456,1,"C",0,0,"2026-10-19"],[0.456,2,"C",0,0,"2026-10-19"],[0.456,1,"E",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"E",1,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"C",1,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-22"],[0.456,0,"C",1,0,"2026-10-23"],[0.456,1,"C",1,0,"2026-10-23"],[0.456,0,"C",1,0,"2026-10-23"],[0.456,0,"C",1,1,"2026-10-27"],[0.456,0,"E",0,0,"2026-10-27"],[0.455,166,"M",1,0,"2026-09-15"],[0.455,2,"E",0,0,"2026-10-18"],[0.455,40,"M",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,8,"M",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,1,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,1,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,0,"C",1,0,"2026-10-23"],[0.455,0,"C",1,0,"2026-10-23"],[0.455,0,"C",1,1,"2026-10-27"],[0.455,0,"E",0,0,"2026-10-27"],[0.454,37,"C",0,0,"2026-10-17"],[0.454,15,"C",0,0,"2026-10-17"],[0.454,2,"E",0,0,"2026-10-18"],[0.454,1,"C",0,0,"2026-10-18"],[0.454,0,"M",0,0,"2026-10-19"],[0.454,0,"C",0,0,"2026-10-19"],[0.454,0,"C",0,0,"2026-10-20"],[0.454,0,"E",0,0,"2026-10-20"],[0.454,0,"E",0,0,"2026-10-20"],[0.454,0,"M",0,0,"2026-10-22"],[0.454,1,"C",1,0,"2026-10-24"],[0.452,36,"E",0,0,"2026-10-14"],[0.452,0,"E",0,0,"2026-10-27"],[0.451,8,"E",0,0,"2026-10-02"],[0.451,1,"C",0,0,"2026-10-12"],[0.451,73,"E",0,0,"2026-10-15"],[0.451,48,"E",0,0,"2026-10-15"],[0.451,14,"E",0,0,"2026-10-16"],[0.451,2,"C",0,0,"2026-10-16"],[0.451,2,"C",0,0,"2026-10-16"],[0.451,3,"C",0,0,"2026-10-17"],[0.451,0,"E",0,0,"2026-10-17"],[0.451,4,"M",0,0,"2026-10-17"],[0.451,0,"C",0,0,"2026-10-18"],[0.451,0,"C",0,0,"2026-10-18"],[0.451,0,"C",0,0,"2026-10-18"],[0.451,0,"C",0,0,"2026-10-19"],[0.45,0,"C",0,0,"2026-10-22"],[0.448,2,"E",1,0,"2026-10-17"],[0.448,2,"E",0,0,"2026-10-18"],[0.448,0,"C",0,0,"2026-10-19"],[0.448,0,"C",0,0,"2026-10-19"],[0.448,2,"M",0,0,"2026-10-19"],[0.448,0,"C",0,0,"2026-10-19"],[0.448,0,"C",0,0,"2026-10-20"],[0.448,0,"E",0,0,"2026-10-25"],[0.448,1,"E",0,0,"2026-10-25"],[0.448,0,"C",1,1,"2026-10-27"],[0.447,0,"C",0,0,"2026-10-17"],[0.447,0,"C",0,0,"2026-10-18"],[0.447,1,"M",0,0,"2026-10-20"],[0.447,0,"E",0,0,"2026-10-24"],[0.447,0,"C",1,1,"2026-10-25"],[0.447,0,"C",1,1,"2026-10-25"],[0.447,0,"C",1,1,"2026-10-25"],[0.446,0,"C",0,0,"2026-10-14"],[0.446,0,"C",0,0,"2026-10-14"],[0.446,63,"E",0,0,"2026-10-14"],[0.446,7,"M",0,0,"2026-10-15"],[0.446,2,"E",0,0,"2026-10-15"],[0.446,1,"C",0,0,"2026-10-15"],[0.446,1,"M",0,0,"2026-10-17"],[0.446,0,"E",0,0,"2026-10-20"],[0.446,0,"C",0,0,"2026-10-22"],[0.445,0,"C",0,0,"2026-10-15"],[0.445,0,"C",0,0,"2026-10-17"],[0.445,0,"C",0,0,"2026-10-20"],[0.445,0,"C",0,0,"2026-10-20"],[0.444,2,"C",0,0,"2026-10-13"],[0.444,0,"C",0,0,"2026-10-15"],[0.444,0,"C",0,0,"2026-10-19"],[0.444,0,"C",0,0,"2026-10-19"],[0.444,0,"C",0,0,"2026-10-19"],[0.444,0,"C",0,0,"2026-10-20"],[0.444,0,"C",0,0,"2026-10-20"],[0.443,18,"C",0,0,"2026-10-13"],[0.443,0,"C",0,0,"2026-10-15"],[0.443,3,"C",0,0,"2026-10-15"],[0.443,0,"C",0,0,"2026-10-20"],[0.442,25,"C",0,0,"2026-10-13"],[0.442,1,"C",0,0,"2026-10-17"],[0.442,0,"C",0,0,"2026-10-22"],[0.441,62,"E",0,0,"2026-10-10"],[0.441,69,"M",0,0,"2026-10-12"],[0.441,12,"E",0,0,"2026-10-13"],[0.441,2,"C",0,0,"2026-10-13"],[0.441,3,"C",0,0,"2026-10-13"],[0.441,0,"E",0,0,"2026-10-14"],[0.441,0,"C",0,0,"2026-10-14"],[0.441,1,"C",0,0,"2026-10-15"],[0.44,139,"M",0,0,"2026-09-17"],[0.44,3,"E",0,0,"2026-09-28"],[0.44,125,"M",0,0,"2026-10-10"],[0.44,3,"C",0,0,"2026-10-10"],[0.44,19,"E",0,0,"2026-10-11"],[0.44,26,"E",0,0,"2026-10-11"],[0.44,0,"E",0,0,"2026-10-11"],[0.44,2,"C",0,0,"2026-10-12"],[0.44,1,"C",0,0,"2026-10-12"],[0.44,0,"C",0,0,"2026-10-12"],[0.44,1,"M",0,0,"2026-10-12"],[0.44,1,"E",0,0,"2026-10-12"],[0.44,0,"C",0,0,"2026-10-13"],[0.44,1,"E",0,0,"2026-10-21"],[0.439,1,"M",0,0,"2026-10-17"],[0.438,2,"C",0,0,"2026-10-09"],[0.438,59,"C",0,0,"2026-10-09"],[0.438,0,"C",0,0,"2026-10-11"],[0.438,0,"C",0,0,"2026-10-15"],[0.438,0,"C",0,0,"2026-10-15"],[0.438,2,"C",0,0,"2026-10-16"],[0.438,0,"C",0,0,"2026-10-18"],[0.438,0,"E",0,0,"2026-10-21"],[0.438,0,"C",0,0,"2026-10-21"],[0.436,1,"M",0,0,"2026-10-14"],[0.436,0,"C",0,0,"2026-10-14"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-17"],[0.436,1,"C",0,0,"2026-10-17"],[0.436,0,"C",0,0,"2026-10-18"],[0.436,0,"C",0,0,"2026-10-18"],[0.436,0,"E",0,0,"2026-10-25"],[0.435,25,"E",0,0,"2026-10-09"],[0.435,0,"C",0,0,"2026-10-15"],[0.435,0,"C",0,0,"2026-10-16"],[0.435,0,"C",0,0,"2026-10-20"],[0.435,0,"C",1,0,"2026-10-26"],[0.434,0,"C",0,0,"2026-10-12"],[0.434,0,"C",0,0,"2026-10-16"],[0.434,0,"C",0,0,"2026-10-16"],[0.434,0,"C",0,0,"2026-10-22"],[0.433,42,"E",0,0,"2026-09-17"],[0.433,29,"E",0,0,"2026-10-07"],[0.433,5,"C",0,0,"2026-10-07"],[0.433,48,"C",0,0,"2026-10-08"],[0.433,8,"C",0,0,"2026-10-09"],[0.433,0,"C",0,0,"2026-10-12"],[0.433,0,"M",1,0,"2026-10-13"],[0.433,0,"C",0,0,"2026-10-14"],[0.431,2,"M",0,0,"2026-10-14"],[0.431,0,"C",0,0,"2026-10-14"],[0.431,2,"M",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"M",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-21"],[0.43,0,"C",0,0,"2026-10-12"],[0.43,1,"C",0,0,"2026-10-12"],[0.429,0,"C",0,0,"2026-10-14"],[0.429,0,"E",0,0,"2026-10-25"],[0.428,86,"C",0,0,"2026-10-07"],[0.428,0,"C",0,0,"2026-10-11"],[0.426,2,"C",0,0,"2026-10-13"],[0.425,107,"C",0,0,"2026-10-05"],[0.425,7,"C",0,0,"2026-10-06"],[0.425,23,"E",0,0,"2026-10-06"],[0.425,9,"C",0,0,"2026-10-07"],[0.425,0,"C",0,0,"2026-10-20"],[0.425,0,"C",0,0,"2026-10-21"],[0.424,3,"C",0,0,"2026-10-07"],[0.424,2,"C",0,0,"2026-10-07"],[0.423,1,"E",0,0,"2026-10-07"],[0.423,3,"M",0,0,"2026-10-08"],[0.423,0,"C",0,0,"2026-10-09"],[0.422,0,"C",0,0,"2026-10-09"],[0.422,0,"C",0,0,"2026-10-09"],[0.422,0,"E",0,0,"2026-10-24"],[0.421,56,"C",0,0,"2026-10-02"],[0.421,2,"C",0,0,"2026-10-06"],[0.421,0,"C",0,0,"2026-10-08"],[0.421,0,"C",0,0,"2026-10-08"],[0.42,16,"E",0,0,"2026-09-23"],[0.42,0,"C",0,0,"2026-10-07"],[0.42,0,"C",0,0,"2026-10-18"],[0.419,0,"C",0,0,"2026-10-06"],[0.419,0,"E",0,0,"2026-10-08"],[0.419,0,"E",0,0,"2026-10-08"],[0.419,0,"E",0,0,"2026-10-08"],[0.419,0,"M",0,0,"2026-10-20"],[0.419,0,"C",0,0,"2026-10-21"],[0.419,0,"E",0,0,"2026-10-27"],[0.418,0,"C",0,0,"2026-10-07"],[0.418,0,"C",0,0,"2026-10-08"],[0.418,0,"C",0,0,"2026-10-09"],[0.418,3,"C",0,0,"2026-10-09"],[0.418,0,"E",0,0,"2026-10-09"],[0.418,0,"C",0,0,"2026-10-10"],[0.417,79,"C",0,0,"2026-10-03"],[0.417,6,"E",0,0,"2026-10-03"],[0.417,1,"E",0,0,"2026-10-05"],[0.417,0,"C",0,0,"2026-10-06"],[0.417,0,"E",0,0,"2026-10-06"],[0.417,0,"C",0,0,"2026-10-08"],[0.416,0,"E",0,0,"2026-10-06"],[0.415,93,"C",0,0,"2026-10-01"],[0.415,0,"C",0,0,"2026-10-07"],[0.415,0,"C",0,0,"2026-10-07"],[0.415,0,"C",0,0,"2026-10-08"],[0.415,0,"C",0,0,"2026-10-09"],[0.415,0,"C",1,0,"2026-10-26"],[0.414,0,"C",0,0,"2026-10-07"],[0.413,5,"C",0,0,"2026-10-01"],[0.413,33,"C",0,0,"2026-10-03"],[0.413,2,"C",0,0,"2026-10-04"],[0.413,9,"C",0,0,"2026-10-04"],[0.413,0,"C",0,0,"2026-10-04"],[0.413,8,"C",0,0,"2026-10-05"],[0.413,0,"C",0,0,"2026-10-08"],[0.411,2,"C",0,0,"2026-10-01"],[0.411,2,"C",0,0,"2026-10-03"],[0.411,0,"C",0,0,"2026-10-05"],[0.411,0,"C",0,0,"2026-10-08"],[0.409,1,"C",0,0,"2026-10-01"],[0.409,27,"C",0,0,"2026-10-01"],[0.409,4,"C",0,0,"2026-10-02"],[0.409,2,"C",0,0,"2026-10-02"],[0.409,1,"C",0,0,"2026-10-03"],[0.409,0,"C",0,0,"2026-10-05"],[0.409,0,"E",0,0,"2026-10-06"],[0.409,0,"C",0,0,"2026-10-08"],[0.407,1,"C",0,0,"2026-10-02"],[0.407,0,"C",0,0,"2026-10-04"],[0.407,0,"C",0,0,"2026-10-05"],[0.406,1,"C",0,0,"2026-10-04"],[0.406,0,"C",0,0,"2026-10-04"],[0.406,0,"C",0,0,"2026-10-06"],[0.405,30,"E",0,0,"2026-09-28"],[0.405,0,"C",0,0,"2026-10-04"],[0.404,2,"E",0,0,"2026-10-02"],[0.404,0,"E",0,0,"2026-10-06"],[0.404,0,"C",1,1,"2026-10-24"],[0.403,1,"C",0,0,"2026-10-05"],[0.402,0,"C",0,0,"2026-10-02"],[0.402,0,"E",0,0,"2026-10-05"],[0.401,32,"C",0,0,"2026-09-30"],[0.4,1,"C",0,0,"2026-10-01"],[0.4,2,"E",0,0,"2026-10-02"],[0.4,0,"C",0,0,"2026-10-02"],[0.4,0,"C",0,0,"2026-10-02"],[0.4,0,"C",0,0,"2026-10-16"],[0.4,0,"C",0,0,"2026-10-16"],[0.4,0,"E",0,0,"2026-10-24"],[0.399,56,"E",0,0,"2026-09-30"],[0.399,5,"M",0,0,"2026-09-30"],[0.399,1,"C",0,0,"2026-10-01"],[0.399,0,"C",0,0,"2026-10-01"],[0.399,1,"C",0,0,"2026-10-02"],[0.399,0,"C",0,0,"2026-10-06"],[0.398,1,"C",0,0,"2026-10-12"],[0.397,0,"E",0,0,"2026-10-01"],[0.397,0,"C",0,0,"2026-10-01"],[0.397,1,"C",0,0,"2026-10-02"],[0.396,0,"E",0,0,"2026-10-01"],[0.396,0,"E",0,0,"2026-10-02"],[0.396,0,"C",0,0,"2026-10-02"],[0.395,0,"C",1,0,"2026-10-01"],[0.394,0,"C",0,0,"2026-10-06"],[0.393,1,"C",0,0,"2026-10-02"],[0.393,1,"C",0,0,"2026-10-02"],[0.393,0,"E",0,0,"2026-10-03"],[0.393,0,"M",0,0,"2026-10-03"],[0.393,0,"E",0,0,"2026-10-04"],[0.393,0,"E",0,0,"2026-10-04"],[0.39,0,"C",0,0,"2026-09-30"],[0.389,0,"C",0,0,"2026-10-04"],[0.386,17,"E",0,0,"2026-09-29"],[0.386,0,"M",0,0,"2026-10-02"],[0.386,0,"M",0,0,"2026-10-02"],[0.386,0,"C",0,0,"2026-10-02"],[0.386,0,"C",1,0,"2026-10-09"],[0.384,23,"M",0,0,"2026-09-29"],[0.384,5,"C",0,0,"2026-09-29"],[0.384,1,"C",0,0,"2026-09-30"],[0.384,0,"C",0,0,"2026-09-30"],[0.384,1,"C",0,0,"2026-09-30"],[0.384,0,"C",0,0,"2026-10-01"],[0.384,0,"M",0,0,"2026-10-18"],[0.384,0,"E",0,0,"2026-10-25"],[0.383,0,"C",0,0,"2026-10-02"],[0.383,0,"M",0,0,"2026-10-17"],[0.382,0,"C",0,0,"2026-09-29"],[0.382,0,"C",1,0,"2026-10-02"],[0.382,0,"C",0,0,"2026-10-06"],[0.381,29,"M",0,0,"2026-09-29"],[0.381,0,"C",0,0,"2026-10-02"],[0.381,0,"M",0,0,"2026-10-19"],[0.38,0,"C",0,0,"2026-09-29"],[0.38,1,"C",0,0,"2026-10-11"],[0.379,0,"C",0,0,"2026-09-29"],[0.378,24,"M",0,0,"2026-09-29"],[0.378,1,"C",0,0,"2026-10-01"],[0.378,0,"E",0,0,"2026-10-03"],[0.378,0,"E",0,0,"2026-10-04"],[0.377,0,"C",0,0,"2026-09-30"],[0.377,0,"C",0,0,"2026-10-02"],[0.377,0,"E",0,0,"2026-10-04"],[0.375,0,"C",0,0,"2026-10-02"],[0.375,0,"C",0,0,"2026-10-03"],[0.375,1,"C",0,0,"2026-10-04"],[0.375,0,"E",0,0,"2026-10-05"],[0.374,2,"E",1,0,"2026-09-30"],[0.374,0,"C",0,0,"2026-10-03"],[0.374,0,"E",0,0,"2026-10-05"],[0.373,19,"M",0,0,"2026-09-28"],[0.372,0,"C",0,0,"2026-10-04"],[0.371,0,"C",0,0,"2026-10-06"],[0.37,23,"M",0,0,"2026-09-26"],[0.37,0,"C",0,0,"2026-09-29"],[0.37,0,"C",0,0,"2026-10-05"],[0.368,0,"C",0,0,"2026-09-29"],[0.367,0,"C",0,0,"2026-09-29"],[0.366,0,"C",0,0,"2026-09-28"],[0.365,0,"C",0,0,"2026-09-30"],[0.364,0,"C",1,0,"2026-10-03"],[0.362,20,"M",0,0,"2026-09-26"],[0.359,0,"C",1,1,"2026-10-26"],[0.359,0,"C",1,1,"2026-10-27"],[0.358,30,"M",0,0,"2026-09-21"],[0.356,0,"C",0,0,"2026-09-29"],[0.356,0,"C",0,0,"2026-09-30"],[0.354,45,"M",0,0,"2026-09-21"],[0.348,0,"C",0,0,"2026-09-28"],[0.347,42,"C",0,0,"2026-09-17"],[0.345,0,"C",0,0,"2026-09-28"],[0.344,0,"C",0,0,"2026-09-28"],[0.344,1,"C",0,0,"2026-10-24"],[0.342,16,"M",0,0,"2026-09-18"],[0.341,40,"E",0,0,"2026-09-20"],[0.341,2,"E",0,0,"2026-09-21"],[0.341,2,"C",0,0,"2026-09-22"],[0.341,3,"C",0,0,"2026-09-25"],[0.341,5,"E",0,0,"2026-09-25"],[0.341,1,"C",0,0,"2026-09-25"],[0.341,0,"C",0,0,"2026-09-27"],[0.341,0,"C",0,0,"2026-09-27"],[0.341,7,"E",0,0,"2026-09-29"],[0.339,270,"E",0,0,"2026-09-15"],[0.339,0,"C",0,0,"2026-09-25"],[0.339,1,"C",0,0,"2026-09-26"],[0.339,0,"C",0,0,"2026-10-24"],[0.338,0,"C",0,0,"2026-09-21"],[0.338,0,"C",0,0,"2026-09-26"],[0.337,81,"E",0,0,"2026-09-16"],[0.337,33,"E",0,0,"2026-09-17"],[0.337,0,"E",0,0,"2026-09-20"],[0.337,0,"C",0,0,"2026-09-25"],[0.337,0,"C",0,0,"2026-09-25"],[0.335,0,"E",0,0,"2026-09-20"],[0.334,13,"E",0,0,"2026-09-19"],[0.334,0,"C",0,0,"2026-09-21"],[0.334,0,"C",0,0,"2026-09-21"],[0.334,0,"E",0,0,"2026-09-22"],[0.334,0,"C",0,0,"2026-09-25"],[0.334,0,"C",0,0,"2026-09-25"],[0.334,0,"C",0,0,"2026-10-24"],[0.333,3,"E",0,0,"2026-09-19"],[0.333,0,"E",0,0,"2026-09-20"],[0.333,0,"C",0,0,"2026-09-24"],[0.333,0,"E",0,0,"2026-09-25"],[0.333,3,"E",0,0,"2026-09-25"],[0.333,0,"C",0,0,"2026-09-27"],[0.333,0,"C",0,0,"2026-10-24"],[0.332,2,"E",0,0,"2026-09-20"],[0.332,0,"C",0,0,"2026-09-26"],[0.332,0,"C",0,0,"2026-09-26"],[0.331,3,"C",0,0,"2026-09-18"],[0.331,0,"E",0,0,"2026-09-20"],[0.331,0,"E",0,0,"2026-09-20"],[0.33,0,"C",0,0,"2026-09-20"],[0.33,1,"C",0,0,"2026-09-21"],[0.33,1,"C",0,0,"2026-09-24"],[0.329,0,"C",0,0,"2026-10-24"],[0.328,2,"E",0,0,"2026-09-17"],[0.328,1,"E",0,0,"2026-09-17"],[0.328,0,"C",0,0,"2026-09-18"],[0.328,0,"C",0,0,"2026-09-20"],[0.328,0,"C",0,0,"2026-09-21"],[0.327,0,"C",0,0,"2026-09-21"],[0.327,0,"C",0,0,"2026-09-23"],[0.326,0,"E",0,0,"2026-09-19"],[0.326,0,"E",0,0,"2026-09-19"],[0.326,0,"E",0,0,"2026-09-19"],[0.323,1,"E",0,0,"2026-09-18"],[0.323,2,"E",0,0,"2026-09-19"],[0.323,0,"C",0,0,"2026-09-21"],[0.322,1,"E",0,0,"2026-09-17"],[0.322,0,"C",0,0,"2026-09-25"],[0.32,1,"E",0,0,"2026-09-22"],[0.319,1,"C",0,0,"2026-09-18"],[0.319,1,"E",0,0,"2026-09-19"],[0.317,1,"C",0,0,"2026-09-20"],[0.315,0,"C",0,0,"2026-09-26"],[0.314,0,"C",0,0,"2026-09-25"],[0.311,0,"C",0,0,"2026-09-28"],[0.309,0,"C",0,0,"2026-09-26"],[0.308,1,"E",0,0,"2026-09-21"],[0.308,1,"E",0,0,"2026-09-21"],[0.301,2,"C",0,0,"2026-09-18"],[0.298,0,"C",0,0,"2026-09-25"],[0.297,1,"E",0,0,"2026-09-19"],[0.293,0,"C",0,0,"2026-09-17"],[0.293,1,"E",0,0,"2026-09-19"],[0.291,0,"C",0,0,"2026-10-11"],[0.29,1,"E",0,0,"2026-09-19"],[0.29,1,"E",0,0,"2026-09-19"],[0.288,0,"C",0,0,"2026-10-11"],[0.285,32,"M",0,0,"2026-09-15"],[0.285,1,"E",0,0,"2026-09-20"],[0.285,1,"E",0,0,"2026-09-20"],[0.285,0,"C",0,0,"2026-09-25"],[0.274,0,"C",0,0,"2026-10-26"],[0.274,0,"C",0,0,"2026-10-26"],[0.273,0,"C",0,0,"2026-10-26"],[0.272,2,"C",0,0,"2026-10-14"],[0.269,2,"E",0,0,"2026-09-15"],[0.264,0,"C",0,0,"2026-09-26"],[0.261,0,"C",0,0,"2026-10-03"],[0.255,0,"C",0,0,"2026-09-21"],[0.249,12,"E",0,0,"2026-09-17"],[0.241,8,"E",0,0,"2026-09-15"],[0.241,3,"C",0,0,"2026-09-16"],[0.239,0,"C",0,0,"2026-10-26"],[0.237,0,"C",0,0,"2026-10-22"],[0.236,0,"C",0,0,"2026-09-19"],[0.236,2,"C",0,0,"2026-09-19"],[0.236,0,"C",0,0,"2026-09-21"],[0.233,3,"M",0,0,"2026-09-15"],[0.231,0,"C",0,0,"2026-10-12"],[0.231,0,"C",0,0,"2026-10-12"],[0.231,0,"C",0,0,"2026-10-26"],[0.228,0,"E",0,0,"2026-10-14"],[0.227,4,"C",0,0,"2026-09-16"],[0.226,1,"C",0,0,"2026-10-12"],[0.224,1,"C",0,0,"2026-09-19"],[0.221,2,"C",0,0,"2026-10-11"],[0.217,0,"C",0,0,"2026-10-25"],[0.21,0,"C",0,0,"2026-10-25"],[0.207,0,"C",0,0,"2026-10-11"],[0.204,0,"C",0,0,"2026-10-26"],[0.191,1,"E",0,0,"2026-09-15"],[0.185,0,"C",0,0,"2026-10-17"],[0.182,2,"C",0,0,"2026-09-18"],[0.18,3,"E",0,0,"2026-09-15"],[0.18,2,"C",0,0,"2026-09-21"],[0.179,0,"C",0,0,"2026-10-25"],[0.176,2,"C",0,0,"2026-10-11"],[0.174,3,"C",0,0,"2026-09-16"],[0.174,0,"E",0,0,"2026-09-17"],[0.171,0,"C",0,0,"2026-10-24"],[0.169,1,"E",0,0,"2026-09-18"],[0.166,1,"E",0,0,"2026-09-15"],[0.166,1,"E",0,0,"2026-09-15"],[0.166,1,"E",0,0,"2026-09-15"],[0.165,2,"E",0,0,"2026-09-27"],[0.163,1,"E",0,0,"2026-09-15"],[0.161,0,"C",0,0,"2026-10-25"],[0.16,8,"M",0,0,"2026-09-15"],[0.158,24,"E",0,0,"2026-09-15"],[0.157,8,"E",0,0,"2026-09-15"],[0.156,32,"E",0,0,"2026-09-15"],[0.156,0,"C",0,0,"2026-09-18"],[0.156,0,"E",0,0,"2026-09-21"],[0.156,0,"C",0,0,"2026-10-26"],[0.155,4,"M",0,0,"2026-09-15"],[0.153,2,"E",0,0,"2026-09-15"],[0.153,0,"C",0,0,"2026-09-24"],[0.153,8,"C",0,0,"2026-10-13"],[0.152,0,"C",0,0,"2026-09-17"],[0.151,2,"M",0,0,"2026-09-15"],[0.151,10,"E",0,0,"2026-09-20"],[0.151,9,"E",0,0,"2026-09-20"],[0.15,2,"C",0,0,"2026-10-11"],[0.149,11,"E",0,0,"2026-09-15"],[0.149,2,"C",0,0,"2026-09-16"],[0.149,0,"C",0,0,"2026-10-13"],[0.146,0,"E",0,0,"2026-09-15"],[0.146,1,"E",0,0,"2026-09-15"],[0.145,0,"C",0,0,"2026-09-25"],[0.145,0,"C",0,0,"2026-10-09"],[0.143,3,"E",0,0,"2026-09-15"],[0.142,6,"C",0,0,"2026-09-16"],[0.142,1,"C",0,0,"2026-09-16"],[0.14,24,"E",0,0,"2026-09-15"],[0.14,0,"C",0,0,"2026-10-15"],[0.136,0,"C",0,0,"2026-10-04"],[0.134,2,"C",0,0,"2026-09-21"],[0.134,0,"C",0,0,"2026-10-01"],[0.13,0,"C",0,0,"2026-09-22"],[0.128,0,"E",0,0,"2026-09-15"],[0.115,0,"C",0,0,"2026-10-01"],[0.11,2,"E",0,0,"2026-09-17"],[0.106,0,"C",0,0,"2026-09-20"],[0.103,0,"C",0,0,"2026-09-18"],[0.099,0,"M",0,0,"2026-10-26"],[0.098,0,"C",0,0,"2026-10-04"],[0.096,3,"E",0,0,"2026-09-15"],[0.094,0,"C",0,0,"2026-09-24"],[0.093,0,"C",0,0,"2026-10-13"],[0.09,0,"C",0,0,"2026-09-19"],[0.09,0,"C",0,0,"2026-10-01"],[0.089,26,"E",0,0,"2026-09-21"],[0.088,0,"C",0,0,"2026-09-22"],[0.086,0,"C",0,0,"2026-10-23"],[0.083,0,"E",0,0,"2026-09-17"],[0.082,0,"E",0,0,"2026-09-29"],[0.064,0,"E",0,0,"2026-09-24"],[0.05,0,"C",0,0,"2026-10-18"],[0.046,0,"C",0,0,"2026-10-24"],[0.038,9,"E",0,0,"2026-09-20"],[0.0,419,"S",0,0,"2026-09-14"],[0.0,0,"C",0,0,"2026-09-17"],[0.0,0,"C",0,0,"2026-09-18"],[0.0,0,"E",0,0,"2026-09-18"],[0.0,0,"C",0,0,"2026-09-18"],[0.0,0,"C",0,0,"2026-09-23"],[0.0,0,"C",0,0,"2026-09-25"],[0.0,0,"C",0,0,"2026-09-25"],[0.0,3,"E",0,0,"2026-09-27"],[0.0,0,"C",0,0,"2026-10-22"],[0.0,0,"C",0,0,"2026-10-22"],[0.0,0,"C",1,1,"2026-10-26"]]}
if SHOW:
    import matplotlib.pyplot as plt
    style_apply()
    sc = np.array([r[0] for r in FIELD['scored']])
    v4g = np.array([r[3] for r in FIELD['scored']], bool)
    lo = 0.40
    fig, ax = plt.subplots(figsize=(10, 4.4))
    bins = np.arange(lo - 0.0005, 0.4806, 0.001)   # one bar per printed score (three decimals)
    ax.hist([sc[(sc >= lo) & ~v4g], sc[(sc >= lo) & v4g]], bins=bins, stacked=True, color=['#ced4da', '#495057'],
            label=['other notebooks', "attach Dmitrii Gluzdov's v4g assets"])
    top = ax.get_ylim()[1]
    for x, name in [(0.465, '0.465 frozen pipeline'), (0.468, '0.468 leader-safe V2'), (0.472, '0.472 two ways'), (0.477, 'this notebook 0.477')]:
        ours = name.startswith('this')
        ax.axvline(x, color=ACCENT if ours else MUTED, lw=1.4 if ours else 1.0, ls='-' if ours else ':')
        ax.text(x - 0.0006, top * 0.98, name, rotation=90, va='top', ha='right', fontsize=8.5,
                color=ACCENT if ours else INK, bbox=dict(facecolor='white', edgecolor='none', pad=1.0, alpha=0.85))
    ax.set_xlim(lo, 0.48)
    ax.set_xlabel('public LB shown on the notebook page'); ax.set_ylabel('notebooks')
    ax.set_title(f"{int((sc >= 0.46).sum())} notebooks at 0.460 or more, {int(v4g[sc >= 0.46].sum())} of them on one set of assets")
    ax.legend(loc='upper left')
    print(f"{FIELD['notebooks_listed']} public notebooks, {len(sc)} with a score, {int((sc < lo).sum())} of those below {lo} not drawn")
    plt.tight_layout(); plt.show()

*Figure 1. Public scores as the Code tab shows them (the best over each notebook's versions), 0.40 and above. Dark:
notebooks that attach Dmitrii Gluzdov's v4g assets.*

## 2. One lineage, two branches

| step | notebook | what it changed | public LB shown |
|---|---|---|---:|
| 1 | Dmitrii Gluzdov, [The Data Behind Spectra to Structures](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures) | assembles and documents the shared pipeline: library search, a PubChem channel, GLACIER and ICEBERG forward models, two-engine fusion (its own credits name canqiang's E006 and Ahmed Berat Özer's engines as the base) | 0.465 |
| 2 | Saner Turhaner, [PubChem + Neural025](https://www.kaggle.com/code/sanpier/pubchem-neural025-lb-0-466) | a neural spectrum-to-structure scorer, fused by reciprocal rank at weight 0.25 | 0.466 |
| 3 | Roman Rozen, [From Peaks to Structures](https://www.kaggle.com/code/romanrozen/0-468-casmi-from-peaks-to-structures) | leader-safe consensus V2: rank 1 as in step 2, ranks 2-25 at neural weight 0.50 (Kozykorpesh's [Balanced Acquisition Evidence](https://www.kaggle.com/code/kozykappa/lb-0-468-balanced-acquisition-evidence) ran this V2 in its versions 6-7, the pipeline Spectral Parallax thanks) | 0.468 |
| 4a | alex chilton, [Leader-Safe, Ranks 2-25 at 0.75](https://www.kaggle.com/code/alexchilton/casmi26-lb-0-472-leader-safe-ranks-2-25-at-0-75) | ranks 2-25 at neural weight 0.75 | 0.472 on [a fork that changes only the commit mode](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-lb472); his own page shows 0.467 |
| 4b | Roman Tamrazov, [Spectral Parallax](https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax) | an untrained fragment check on the V2 lists | 0.472 (best of its versions; version 22, whose code runs here, shows no score of its own) |
| 5 | this notebook | 4b's leader moves from the V2 order, ranks 2-25 from 4a's 0.75 order | **0.477** |

Scores are what the Code tab displayed on 2026-10-27, Kaggle's best public score over each notebook's versions. The
current versions of some linked notebooks run something else: Roman Rozen's now stacks a private isomer reranker
(skipped when its dataset is absent), the fragment check and a tail recovery on ranks 2-25 at 1.00, and since
2026-10-28 his page shows 0.472.

The fusion rule in one line: a candidate at pipeline rank b and neural rank n scores 1/(3 + b) + w/(3 + n). Rank 1
uses w = 0.25, so the neural model can take the lead from the pipeline's leader only when that leader sits at neural
rank 18 or worse and the runner-up is the neural favourite. Ranks 2-25 use w_mid; at 0.75 the neural model sorts them
much harder than at 0.50.

**What the fragment check does.** 4b has three parts:

- **Rank 1.** When the leader's match to the reference library is below 0.87 and the molecule has at least three
  usable spectra, a candidate from ranks 2-6 with the leader's formula (an isomer the precursor mass cannot tell
  apart) can take rank 1. It must explain clearly more fragment intensity than the leader and score higher with the
  neural model, and it needs support from the engines' own top-5 lists or from the GLACIER and ICEBERG forward
  models.
- **Swaps.** Up to two adjacent swaps of same-formula candidates in ranks 2-11, with no library gate.
- **Last place.** One same-formula outside candidate may take rank 25.

**Why stacking 4a and 4b scored 0.472, and what scored 0.477.** 4a never touches rank 1, so the two changes looked
independent. Run together (our e1) they scored 0.472, the same as each alone. The rank-1 rule takes its contenders
from ranks 2-6 of the order it is handed. On the 0.75 order that window holds different candidates: on Saner
Turhaner's labelled panel, the same-formula candidates in ranks 2-6 that pass the rule's neural gates change in 17 of
86 eligible lists, each change adds a wrong structure and no right one drops out (the fragment gates need the
spectra and were not replayed). Our reading, which no single submission isolates: on the 0.75 order the rule keeps
its good leader moves and adds wrong ones. The whole fragment check (rank-1 rule, swaps and last place) on the
long-trained neural model's 0.75 order cost 0.008 (e3 0.453 against e4 0.461); that pair does not separate the rank-1
rule from the other two parts. This
notebook runs the rule on the V2 order, keeps only the leader it picks there, and places the old leader second and
the 0.75 order below them: 0.477.

The field already swept the knobs around 4a (displayed scores, knobs read from each notebook's configuration cell):
w_mid 0.70, 0.75 and 0.80 all show 0.472 ([v1220](https://www.kaggle.com/code/rajrajak99/casmi26-apex-v1220-golden-ratio-070), [fork-lb472](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-lb472),
[v1210](https://www.kaggle.com/code/rajrajak99/casmi26-apex-v1210-sweet-spot-080)); rank-1 weight 0 shows 0.472 ([v1230](https://www.kaggle.com/code/rajrajak99/casmi26-apex-v1230-pure-shield-075)), 0.50 shows 0.448 and 1.00 shows
0.359 ([fork-wt05](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-wt05), [fork-wt10](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-wt10)). On the step-3 line (w_mid 0.50), offset 1 instead
of 3 shows 0.466 ([v520](https://www.kaggle.com/code/guhongbin/casmi26-v520-p2s-wmid050off1)). Moving rank 1 with the neural model is where scores are lost.

## 3. The pipeline

The configuration and session cells at the top of this notebook and the cells below are Spectral Parallax's, in his
order. Hidden: the pinned sources (Saner Turhaner's six helper modules from the 0.466 notebooks, Roman Rozen's
`consensus.py` and `outsiders.py`, and the 0.465 pipeline embedded as base64, each checked by SHA-256), the stage
runner, and the fragment check's source. Shown: the run cell (pipeline in one process, the neural scorer on the final
lists and on up to 20 outside candidates per molecule, the leader-safe consensus), the fragment-check cell and the
provenance cell. A failing overlay restores the previous answer file; the scoring rerun refuses to submit without a
real pipeline run and a neural score.

In [ ]:
# ── Pinned sources (generated; hidden in the viewer) ─────────────────────────────────────────
# MODULES: the six helper modules of the public 0.466 notebooks, unchanged, plus this notebook's
# consensus.py and outsiders.py. FROZEN_B64: the public 0.465 pipeline notebook, byte for byte.
MODULES = {}
MODULES['runner.py'] = '"""Execute the pinned corrected baseline once, then export its final candidates."""\nimport os,sys,json,hashlib\nfrom pathlib import Path\nfrom IPython.display import display\nif __name__==\'__main__\':\n    source=Path(sys.argv[1]); expected=sys.argv[2]; root=Path(sys.argv[3]).resolve()\n    assert hashlib.sha256(source.read_bytes()).hexdigest()==expected, \'Source hash differs\'\n    assert not os.environ.get(\'INC_PANEL\'), \'Replay input is not allowed\'\n    root.mkdir(parents=True,exist_ok=True); os.chdir(root)\n    assert not (root/\'work/spec_meta.parquet\').exists(), \'Fresh pipeline directory required\'\n    assert not (root/\'submission.csv\').exists(), \'Fresh pipeline directory required\'\n    scope={\'__name__\':\'__main__\',\'display\':display}\n    for i,c in enumerate(json.loads(source.read_text())[\'cells\']):\n        if c[\'cell_type\']!=\'code\': continue\n        print(f\'PIPELINE CELL {i} START\',flush=True)\n        exec(compile(\'\'.join(c[\'source\']),f\'pipeline_cell_{i}\',\'exec\'),scope)\n        print(f\'PIPELINE CELL {i} DONE\',flush=True)\n    assert \'candidate_promotion_evidence\' in scope, \'Candidate-specific promotion fix missing\'\n    from export_evidence import export\n    export(scope,root)\n    (root/\'execution_receipt.json\').write_text(json.dumps(dict(source_sha256=expected,\n        mode=\'submission\',source_adaptations=[],candidate_promotion_fix=True,\n        is_full_run=bool(scope[\'IS_RERUN\']),processed_molecules=int(scope[\'te\'].molecule_id.nunique())),indent=2))\n'
MODULES['export_evidence.py'] = '"""Executed in the frozen pipeline process AFTER every original final rule."""\nimport math,json,shutil\nfrom pathlib import Path\nimport pandas as pd\n\ndef export(scope, root):\n    root=Path(root);frame=pd.read_csv(root/\'submission.csv\',dtype=str,keep_default_na=False)\n    shutil.copy2(root/\'submission.csv\',root/\'submission_baseline0465.csv\')\n    processed=set(scope[\'te\'].molecule_id.astype(str));identity=scope[\'metric_identity\']\n    def rank_map(smis):\n        out={}\n        for rank,s in enumerate(smis,1):\n            key=identity(s)\n            if key is not None:out.setdefault(key,rank)\n        return out\n    def finite(v):\n        try:return float(v) if v is not None and math.isfinite(float(v)) else None\n        except (ValueError,TypeError):return None\n    rows=[]\n    for mid,answer in frame.itertuples(index=False):\n        candidates=answer.split(\';\')\n        assert 1<=len(candidates)<=25 and len(candidates)==len(set(candidates)),mid\n        active=mid in processed\n        base=scope[\'BASE\'].get(mid,([],[],0.))\n        maps={\n            \'main\':rank_map(base[0]),\n            \'prefusion\':rank_map(scope.get(\'PREFUSION_LISTS\',{}).get(mid,[])),\n            \'engine2\':rank_map(scope.get(\'ENG\',{}).get(mid,{}).get(\'smiles\',[])),\n            \'pubchem\':rank_map(scope.get(\'PC\',{}).get(mid,{}).get(\'pc\',[]))}\n        ice=scope.get(\'ICE_SCORES\',{}).get(mid,{}) or {}\n        gl=scope[\'gl_of\'](mid) or {}\n        for rank,s in enumerate(candidates,1):\n            ident=identity(s);key=scope[\'chem\'].score_key(s)\n            row=dict(molecule_id=mid,smiles=s,metric_key=ident[0] if ident else None,baseline_rank=rank,\n                processed=active,library_max=finite(base[2]) or 0.,proxy_gl=mid in scope.get(\'N1_GL_MIDS\',set()),\n                ice=finite(ice.get(s,ice.get(key))),gl=finite(gl.get(s,gl.get(key))))\n            row.update({name+\'_rank\':mapping.get(ident,0) for name,mapping in maps.items()});rows.append(row)\n    pd.DataFrame(rows).to_parquet(root/\'frozen_candidate_evidence.parquet\',index=False)\n    # Save the effective query spectra used by the pipeline (12 rows\' molecules in visible smoke, all in rerun).\n    scope[\'te\'].to_parquet(root/\'effective_query_spectra.parquet\',index=False)\n    (root/\'frozen_run_info.json\').write_text(json.dumps(dict(processed_molecules=len(processed),\n        output_molecules=len(frame),is_full_run=bool(scope[\'IS_RERUN\']),\n        note=\'Baseline file saved after the original Engine2 rank-2 insertion; candidate-specific PubChem promotion fix included.\'),indent=2))\n    print(\'EXPORTED FROZEN FINAL25\',len(rows),\'candidate rows\',flush=True)\n'
MODULES['neural_scores.py'] = '"""Runs after the public pipeline process exits, freeing its GPU memory."""\nimport json,sys,importlib.util,types,hashlib\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch\n\ndef load_model(bundle,checkpoint,device):\n    # Private module namespace prevents collision with public engines\' model/features modules.\n    package=\'_casmi_own\';p=types.ModuleType(package);p.__path__=[str(bundle)];sys.modules[package]=p\n    for name in [\'features\',\'model\']:\n        spec=importlib.util.spec_from_file_location(package+\'.\'+name,Path(bundle)/(name+\'.py\'))\n        mod=importlib.util.module_from_spec(spec);sys.modules[spec.name]=mod;spec.loader.exec_module(mod)\n    features=sys.modules[package+\'.features\'];cls=sys.modules[package+\'.model\'].SpectrumCandidateModel\n    cfg=json.loads((Path(bundle)/\'config.json\').read_text());model=cls(cfg).to(device)\n    ck=torch.load(Path(bundle)/checkpoint,map_location=device,weights_only=True)\n    model.load_state_dict(ck[\'state_dict\']);model.eval()\n    return model,features,cfg\n\n@torch.inference_mode()\ndef score(bundle,checkpoint,evidence,queries,output,device=\'cuda\'):\n    torch.set_num_threads(4)\n    if device==\'cuda\':assert torch.cuda.is_available(),\'GPU required for neural scoring\'\n    model,feat,cfg=load_model(bundle,checkpoint,device)\n    frame=pd.read_parquet(evidence);q=pd.read_parquet(queries);q[\'molecule_id\']=q.molecule_id.astype(str)\n    query={mid:g for mid,g in q.groupby(\'molecule_id\',sort=False)};cache={};parts=[];errors=[]\n    for index,(mid,g) in enumerate(frame.groupby(\'molecule_id\',sort=False)):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\').copy()\n        g[\'neural_ok\']=False\n        if not g.processed.iloc[0]:\n            for c in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[c]=0.\n            parts.append(g);continue\n        try:\n            assert mid in query,mid\n            records=[]\n            # Use EVERY observed query spectrum; no adduct merging and no new spectrum selection rule.\n            for row in query[mid].itertuples(index=False):\n                try:\n                    pk,n=feat.peaks(row.ms2_mzs,row.ms2_normalized_intensities,row.precursor_mz,cfg[\'max_peaks\'])\n                    if n<cfg[\'min_peaks\']:continue\n                    m,a=feat.metadata(row.precursor_mz,row.adduct,row.ionization_mode,row.collision_energy_ev)\n                    records.append((pk,n,m,a))\n                except (ValueError,TypeError):continue\n            if not records:raise ValueError(\'No usable spectra after the training preprocessing filters\')\n            fp=[]\n            for s in g.smiles:\n                if s not in cache:cache[s]=feat.structure(s,cfg[\'fp_bits\'])[-1]\n                fp.append(cache[s])\n            F=torch.tensor(np.stack(fp),device=device,dtype=torch.float32);M=model.encode_molecules(F)\n            cos_all=[];fp_all=[]\n            for st in range(0,len(records),64):\n                b=records[st:st+64]\n                x=torch.tensor(np.stack([v[0] for v in b]),device=device)\n                lengths=torch.tensor([v[1] for v in b],device=device,dtype=torch.long)\n                meta=torch.tensor(np.stack([v[2] for v in b]),device=device)\n                adduct=torch.tensor([v[3] for v in b],device=device,dtype=torch.long)\n                Z,P=model.encode_spectra(x,lengths,meta,adduct)\n                cos_all.append((Z@M.T).cpu().numpy());fp_all.append((P@F.T/cfg[\'fp_bits\']).cpu().numpy())\n            cos=np.concatenate(cos_all);fps=np.concatenate(fp_all)\n            combined=cos/cfg[\'temperature\']+cfg[\'fp_score_weight\']*fps\n            assert np.isfinite(combined).all(),mid\n            g[\'neural_cos\']=cos.mean(0);g[\'neural_fp\']=fps.mean(0);g[\'neural_score\']=combined.mean(0)\n            g[\'neural_std\']=combined.std(0);g[\'n_spectra\']=len(records);g[\'n_adducts\']=query[mid].adduct.nunique()\n            g[\'neural_ok\']=True\n            parts.append(g)\n        except (ValueError,TypeError) as exc:\n            errors.append(dict(molecule_id=mid,error=str(exc)))\n            for col in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[col]=0.\n            parts.append(g)\n        if (index+1)%25==0:print(\'NEURAL SCORED\',index+1,\'molecules\',flush=True)\n    result=pd.concat(parts,ignore_index=True)\n    assert len(result)==len(frame)\n    result.to_parquet(output,index=False)\n    Path(str(output)+\'.errors.json\').write_text(json.dumps(errors,indent=2))\n    print(\'NEURAL input failures kept at baseline:\',len(errors),flush=True)\n    print(\'NEURAL FEATURES\',result.shape,checkpoint,flush=True)\n\nif __name__==\'__main__\':score(*sys.argv[1:])\n'
MODULES['submission.py'] = 'import json,hashlib\nfrom pathlib import Path\nimport pandas as pd,numpy as np\nfrom ranking import rrf_order\n\ndef apply(evidence,baseline,output,weight=.25,offset=3.):\n    f=pd.read_parquet(evidence);base=pd.read_csv(baseline,dtype=str,keep_default_na=False)\n    groups={mid:g for mid,g in f.groupby(\'molecule_id\',sort=False)};rows=[];audit=[]\n    assert base.molecule_id.is_unique and set(groups)==set(base.molecule_id)\n    for mid,text in base.itertuples(index=False):\n        g=groups[mid].sort_values(\'baseline_rank\',kind=\'stable\');original=text.split(\';\')\n        assert g.smiles.tolist()==original,mid\n        active=bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\n        order=rrf_order(g.neural_score.values,weight,offset) if active else np.arange(len(g))\n        new=g.smiles.iloc[order].tolist()\n        assert len(new)==len(original) and set(new)==set(original),mid\n        assert len(new)<=25 and len(set(new))==len(new)\n        rows.append((mid,\';\'.join(new)))\n        audit.append(dict(molecule_id=mid,processed=active,changed=new!=original,\n            changed_top1=new[0]!=original[0],candidate_count=len(new),weight=weight))\n    result=pd.DataFrame(rows,columns=[\'molecule_id\',\'smiles\'])\n    assert result.molecule_id.tolist()==base.molecule_id.tolist()\n    result.to_csv(output,index=False)\n    out=Path(output).parent;pd.DataFrame(audit).to_csv(out/\'neural_rerank_audit.csv\',index=False)\n    report=dict(method=\'baseline rank + neural rank reciprocal-rank fusion AFTER every baseline rule\',\n        neural_weight=weight,rrf_offset=offset,weight_is_validated=False, validation_scope="Fixed-weight submission experiment; no validation performed",\n        candidate_set_preserved=True,changed_rows=sum(x[\'changed\'] for x in audit),\n        changed_top1=sum(x[\'changed_top1\'] for x in audit),rows=len(rows),\n        frozen_sha256=hashlib.sha256(Path(baseline).read_bytes()).hexdigest(),\n        experimental_sha256=hashlib.sha256(Path(output).read_bytes()).hexdigest())\n    (out/\'neural_submission_report.json\').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n    return report\n'
MODULES['ranking.py'] = '"""Pure deterministic ranking utilities shared by both notebooks."""\nimport numpy as np\nimport pandas as pd\n\ndef stable_rank(values):\n    values=np.asarray(values,dtype=float)\n    assert np.isfinite(values).all(), \'Nonfinite candidate scores\'\n    order=np.argsort(-values,kind=\'stable\');ranks=np.empty(len(values),int);ranks[order]=np.arange(1,len(values)+1)\n    return ranks\n\ndef rrf_order(neural_scores,weight=0.15,offset=3.0):\n    assert weight>=0 and offset>=0\n    n=len(neural_scores);baseline=np.arange(1,n+1)\n    # Zero weight guarantees exact baseline order even when neural scores are unavailable.\n    if weight==0:return np.arange(n)\n    ranks=stable_rank(neural_scores)\n    return np.argsort(-(1/(offset+baseline)+weight/(offset+ranks)),kind=\'stable\')\n\ndef metrics(frame,score=\'prediction\'):\n    out=[]\n    for mid,g in frame.groupby(\'molecule_id\',sort=False):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\')\n        order=np.argsort(-g[score].to_numpy(),kind=\'stable\')\n        labels=g.label.to_numpy()[order];where=np.flatnonzero(labels)\n        r=int(where[0]+1) if len(where) else None\n        out.append(dict(molecule_id=mid,mrr25=1/r if r and r<=25 else 0.,top1=int(r==1),\n                        candidate_recall=int(bool(len(where))),rank=r))\n    return pd.DataFrame(out)\n\nBASE_FEATURES=[\'baseline_rr\',\'main_rr\',\'prefusion_rr\',\'engine2_rr\',\'pubchem_rr\',\'engine_agreement\',\n    \'library_max\',\'proxy_gl\',\'ice_missing\',\'gl_missing\',\'ice_z\',\'gl_z\',\n    \'neural_cos_z\',\'neural_fp_z\',\'neural_score_z\',\'neural_rr\',\'neural_std\',\n    \'n_spectra_log\',\'n_adducts\',\'neural_top_gap\',\'neural_agrees_top1\',\'engine2_agrees_top1\',\n    \'neural_delta_rank\',\'neural_x_library\',\'neural_x_agreement\',\'neural_x_sim_missing\',\n    \'main_missing\',\'engine2_missing\',\'pubchem_missing\']\n\ndef make_features(frame):\n    f=frame.copy()\n    for src in [\'baseline\',\'main\',\'prefusion\',\'engine2\',\'pubchem\']:\n        r=pd.to_numeric(f[src+\'_rank\'],errors=\'coerce\');f[src+\'_rr\']=np.where(r>0,1/(3+r),0)\n        if src in [\'main\',\'engine2\',\'pubchem\']:f[src+\'_missing\']=(~(r>0)).astype(float)\n    for src in [\'ice\',\'gl\',\'neural_cos\',\'neural_fp\',\'neural_score\']:\n        v=pd.to_numeric(f[src],errors=\'coerce\');v=v.where(np.isfinite(v))\n        if src in [\'ice\',\'gl\']:f[src+\'_missing\']=v.isna().astype(float)\n        grp=v.groupby(f.molecule_id,sort=False)\n        f[src+\'_z\']=((v-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    f[\'neural_rank\']=0.;f[\'neural_top_gap\']=0.;f[\'neural_agrees_top1\']=0.;f[\'engine2_agrees_top1\']=0.\n    for _,g in f.groupby(\'molecule_id\',sort=False):\n        idx=g.index;rank=stable_rank(g.neural_score.values);f.loc[idx,\'neural_rank\']=rank\n        vals=np.sort(g.neural_score_z.values)[::-1]\n        f.loc[idx,\'neural_top_gap\']=float(vals[0]-vals[1]) if len(vals)>1 else 0.\n        f.loc[idx,\'neural_agrees_top1\']=float(g.iloc[np.argmin(rank)].baseline_rank==1)\n        f.loc[idx,\'engine2_agrees_top1\']=float(((g.baseline_rank==1)&(g.engine2_rank==1)).any())\n    f[\'neural_rr\']=1/(3+f.neural_rank)\n    f[\'engine_agreement\']=((f.main_rank>0).astype(int)+(f.engine2_rank>0).astype(int)+(f.pubchem_rank>0).astype(int))\n    f[\'n_spectra_log\']=np.log1p(f.n_spectra)\n    f[\'neural_delta_rank\']=(f.baseline_rank-f.neural_rank)/25\n    f[\'neural_x_library\']=f.neural_score_z*f.library_max\n    f[\'neural_x_agreement\']=f.neural_score_z*f.engine_agreement\n    f[\'neural_x_sim_missing\']=f.neural_score_z*f.ice_missing*f.gl_missing\n    X=f[BASE_FEATURES].astype(float).replace([np.inf,-np.inf],np.nan).fillna(0.)\n    assert np.isfinite(X.values).all()\n    return X\n\ndef residual_scores(frame,probability,beta):\n    p=np.clip(np.asarray(probability,float),1e-5,1-1e-5)\n    raw=pd.Series(np.log(p/(1-p)),index=frame.index)\n    grp=raw.groupby(frame.molecule_id,sort=False)\n    z=((raw-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    correction=float(beta)*z.values\n    if \'neural_ok\' in frame:correction=np.where(frame.neural_ok.values,correction,0.)\n    return -np.log(frame.baseline_rank.values.astype(float))+correction\n'
MODULES['bundle.py'] = "import hashlib,json,zipfile\nfrom pathlib import Path\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,'rb') as f:\n        for b in iter(lambda:f.read(8<<20),b''):h.update(b)\n    return h.hexdigest()\n\ndef find_bundle(input_root,output_root,pins):\n    roots=[]\n    for path in Path(input_root).rglob('final_model.pt'):\n        if (path.parent/'config.json').is_file():roots.append(path.parent)\n    if not roots:\n        archives=list(Path(input_root).rglob('casmi_spectrum_model_v1.zip'))\n        assert len(archives)==1, 'Attach the supplied casmi_spectrum_model_v1.zip as a Kaggle dataset (zipped or extracted)'\n        dest=Path(output_root)/'uploaded_model';dest.mkdir(parents=True,exist_ok=True)\n        with zipfile.ZipFile(archives[0]) as z:\n            for item in z.infolist():\n                target=(dest/item.filename).resolve()\n                assert target.is_relative_to(dest.resolve()),'Unsafe ZIP member'\n            z.extractall(dest)\n        roots=[p.parent for p in dest.rglob('final_model.pt')]\n    matching=[r for r in roots if all((r/n).is_file() and sha(r/n)==h for n,h in pins.items())]\n    assert len(matching)==1, f'Expected exactly one matching model bundle, found {len(matching)}. Use the attached v1 artifacts.'\n    return matching[0]\n"
MODULES['consensus.py'] = '"""Leader-safe consensus re-ranking for CASMI26 answer lists.\n\nRuns after the frozen 0.465 pipeline and after the neural scorer. It never reruns a\nmodel: it reorders the candidates of each molecule and, only when asked, lets a few\ncandidates that the pipeline left outside its list compete for the last places.\n\nOrder rule\n    rank 1   = leader of   1/(k + b) + w_top / (k + n)\n    ranks 2+ = the others, ordered by   1/(k + b) + w_mid / (k + n)\nwith b the baseline rank, n the neural rank among the same candidates and k the\noffset. With w_top == w_mid this is exactly the fixed-weight reciprocal-rank fusion\nof the public 0.466 notebook (``ranking.rrf_order``), which build_variants re-checks.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom ranking import rrf_order, stable_rank\n\nPRESETS = {\n    \'V0\': dict(w_top=0.00, w_mid=0.00, rescue=0, label=\'Baseline 0.465 - no neural term\'),\n    \'V1\': dict(w_top=0.25, w_mid=0.25, rescue=0, label=\'Neural RRF 0.25 - public 0.466 recipe\'),\n    \'V2\': dict(w_top=0.25, w_mid=0.50, rescue=0, label=\'Leader-safe consensus 0.25 / 0.50\'),\n    \'V3\': dict(w_top=0.25, w_mid=0.50, rescue=3, label=\'Leader-safe consensus + 3-slot tail rescue\'),\n    \'V4\': dict(w_top=0.25, w_mid=1.00, rescue=0, label=\'Bold consensus 0.25 / 1.00\'),\n}\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef leader_safe_order(neural_scores, w_top, w_mid, offset=3.0):\n    """Indices of the new order (baseline order = 0..n-1)."""\n    s = np.asarray(neural_scores, dtype=float)\n    if len(s) == 0:\n        return np.zeros(0, dtype=int)\n    first = np.asarray(rrf_order(s, w_top, offset), dtype=int)\n    if w_mid == w_top:\n        return first\n    lead = int(first[0])\n    rest = [int(i) for i in rrf_order(s, w_mid, offset) if int(i) != lead]\n    return np.asarray([lead] + rest, dtype=int)\n\n\ndef rescue_tail(smiles, scores, outsiders, slots, limit=25):\n    """Let outside candidates compete for the last `slots` places by neural score.\n\n    smiles, scores: the reordered list and its neural scores.\n    outsiders: [(smiles, neural_score)] whose metric identities differ from every\n    listed candidate and from each other (guaranteed by outsiders.py).\n    Free places of a list shorter than `limit` are offered as well.\n    Returns (new_list, number_of_outsiders_admitted).\n    """\n    smiles = list(smiles)\n    scores = [float(v) for v in scores]\n    if slots <= 0 or not outsiders or not smiles:\n        return smiles, 0\n    k = min(int(slots), len(smiles))\n    free = max(0, int(limit) - len(smiles))\n    head = smiles[:len(smiles) - k]\n    pool = [(s, v, 0) for s, v in zip(smiles[len(smiles) - k:], scores[len(scores) - k:])]\n    pool += [(str(s), float(v), 1) for s, v in outsiders if np.isfinite(float(v)) and str(s) not in smiles]\n    pool.sort(key=lambda t: (-t[1], t[2]))  # stable; on ties a listed candidate keeps its place\n    chosen = pool[:k + free]\n    out = (head + [s for s, _, _ in chosen])[:limit]\n    return out, int(sum(t[2] for t in chosen))\n\n\ndef read_outsiders(path):\n    out = {}\n    if not path or not Path(path).is_file():\n        return out\n    frame = pd.read_parquet(path)\n    if frame.empty or \'neural_ok\' not in frame.columns:\n        return out\n    score = pd.to_numeric(frame.neural_score, errors=\'coerce\')\n    frame = frame[frame.neural_ok.astype(bool) & np.isfinite(score)]\n    for mid, g in frame.groupby(\'molecule_id\', sort=False):\n        g = g.sort_values(\'baseline_rank\', kind=\'stable\')\n        out[str(mid)] = list(zip(g.smiles.astype(str), g.neural_score.astype(float)))\n    return out\n\n\ndef build_variants(evidence, baseline, outdir, presets=None, offset=3.0, outsiders=None):\n    """Write submission_<name>.csv for every preset; return (report, moves)."""\n    presets = dict(PRESETS if presets is None else presets)\n    frame = pd.read_parquet(evidence)\n    base = pd.read_csv(baseline, dtype=str, keep_default_na=False)\n    assert list(base.columns) == [\'molecule_id\', \'smiles\'] and base.molecule_id.is_unique\n    groups = {str(m): g.sort_values(\'baseline_rank\', kind=\'stable\') for m, g in frame.groupby(\'molecule_id\', sort=False)}\n    assert set(groups) == set(base.molecule_id), \'evidence and baseline cover different molecules\'\n    outs = read_outsiders(outsiders)\n    outdir = Path(outdir)\n    outdir.mkdir(parents=True, exist_ok=True)\n    report, moves = {}, []\n    for name, cfg in presets.items():\n        rows = []\n        stats = dict(active_rows=0, changed_rows=0, changed_top1=0, rescued=0)\n        for mid, text in base.itertuples(index=False):\n            original = text.split(\';\')\n            g = groups[mid]\n            assert g.smiles.tolist() == original, f\'evidence order differs from the baseline: {mid}\'\n            active = bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\n            if active:\n                order = leader_safe_order(g.neural_score.values, cfg[\'w_top\'], cfg[\'w_mid\'], offset)\n            else:\n                order = np.arange(len(g))\n            new = g.smiles.iloc[order].tolist()\n            new_scores = g.neural_score.astype(float).values[order].tolist()\n            rescued = 0\n            if active and int(cfg.get(\'rescue\', 0)) > 0 and outs.get(mid):\n                new, rescued = rescue_tail(new, new_scores, outs[mid], int(cfg[\'rescue\']))\n            assert 1 <= len(new) <= 25 and len(set(new)) == len(new), mid\n            if not rescued:\n                assert sorted(new) == sorted(original), f\'candidate set changed: {mid}\'\n            stats[\'active_rows\'] += int(active)\n            stats[\'changed_rows\'] += int(new != original)\n            stats[\'changed_top1\'] += int(new[0] != original[0])\n            stats[\'rescued\'] += int(rescued)\n            rows.append((mid, \';\'.join(new)))\n            if active:\n                b_rank = {s: i + 1 for i, s in enumerate(original)}\n                n_rank = dict(zip(g.smiles, stable_rank(g.neural_score.values)))\n                for pos, s in enumerate(new, 1):\n                    moves.append((name, mid, s, pos, b_rank.get(s, 0), int(n_rank.get(s, 0))))\n        out = pd.DataFrame(rows, columns=[\'molecule_id\', \'smiles\'])\n        assert out.molecule_id.tolist() == base.molecule_id.tolist()\n        path = outdir / f\'submission_{name}.csv\'\n        out.to_csv(path, index=False)\n        report[name] = dict(cfg, offset=offset, file=path.name, sha256=sha256(path), **stats)\n    moves = pd.DataFrame(moves, columns=[\'variant\', \'molecule_id\', \'smiles\', \'final_rank\', \'baseline_rank\', \'neural_rank\'])\n    (outdir / \'consensus_report.json\').write_text(json.dumps(report, indent=2))\n    return report, moves\n\n\ndef validate(path, sample_ids, baseline_path=None, allow_new=False, identity_of=None):\n    """Raise unless `path` is a well-formed answer file for the expected molecules."""\n    frame = pd.read_csv(path, dtype=str, keep_default_na=False)\n    assert list(frame.columns) == [\'molecule_id\', \'smiles\'], \'columns\'\n    assert frame.molecule_id.tolist() == list(sample_ids), \'molecule order\'\n    lists = frame.smiles.str.split(\';\')\n    assert lists.map(len).between(1, 25).all(), \'list length\'\n    assert lists.map(lambda v: all(x.strip() for x in v) and len(set(v)) == len(v)).all(), \'empty or repeated SMILES\'\n    if baseline_path is not None:\n        base = pd.read_csv(baseline_path, dtype=str, keep_default_na=False)\n        assert base.molecule_id.tolist() == frame.molecule_id.tolist(), \'baseline order\'\n        for before, after in zip(base.smiles, frame.smiles):\n            b, a = before.split(\';\'), after.split(\';\')\n            if allow_new:\n                assert len(a) >= min(len(b), 25), \'list got shorter\'\n            else:\n                assert sorted(b) == sorted(a), \'candidate set changed\'\n    if identity_of is not None:\n        for answer in frame.smiles:\n            keys = [identity_of.get(s) for s in answer.split(\';\')]\n            keys = [k for k in keys if k]\n            assert len(keys) == len(set(keys)), \'duplicate metric identity\'\n    return frame\n'
MODULES['outsiders.py'] = '"""Collect candidates that the frozen pipeline considered but left outside its final list.\n\nReads the pipeline\'s own record of its candidate lists (e004_candidate_inputs.json),\nwalks the fusion overflow, Engine 2, the main-engine tail and the PubChem channel in\nround-robin rank order, and keeps structures whose official metric identity (the frozen\nnotebook\'s own metric_identity, executed from its source) is new for the molecule.\nOutput rows follow the evidence schema the neural scorer reads; baseline_rank 26, 27, ...\nonly fixes their order.\n\nusage: python outsiders.py <frozen.ipynb> <root> <evidence.parquet> <out.parquet> [per_molecule] [max_checks]\n"""\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\n\nCOLUMNS = [\'molecule_id\', \'smiles\', \'metric_key\', \'source\', \'source_rank\', \'baseline_rank\', \'processed\']\n\n\ndef load_identity(frozen):\n    cells = json.loads(Path(frozen).read_text())[\'cells\']\n    hits = [\'\'.join(c[\'source\']) for c in cells\n            if c[\'cell_type\'] == \'code\' and \'def metric_identity\' in \'\'.join(c[\'source\'])]\n    assert len(hits) == 1, \'metric identity cell not found in the frozen source\'\n    scope = {\'__name__\': \'frozen_metric_identity\'}\n    exec(compile(hits[0], \'frozen_metric_identity\', \'exec\'), scope)\n    return scope[\'metric_identity\']\n\n\ndef source_lists(payload, mid):\n    eng = payload.get(\'engine2\', {}).get(mid) or {}\n    pc = payload.get(\'pubchem\', {}).get(mid) or {}\n    return [(\'fusion_overflow\', list(payload.get(\'fusion_overflow\', {}).get(mid) or [])),\n            (\'engine2\', list(eng.get(\'smiles\') or [])),\n            (\'main_tail\', list(payload.get(\'base_tail\', {}).get(mid) or [])),\n            (\'pubchem\', list(pc.get(\'pc\') or []))]\n\n\ndef collect(payload, evidence, identity, per_molecule=12, max_checks=30):\n    rows = []\n    active = evidence[evidence.processed.astype(bool)]\n    for mid, g in active.groupby(\'molecule_id\', sort=False):\n        mid = str(mid)\n        if mid not in payload.get(\'prefusion\', {}):\n            continue\n        listed = g.sort_values(\'baseline_rank\', kind=\'stable\').smiles.astype(str).tolist()\n        taken = {k for k in g.metric_key.tolist() if isinstance(k, str) and k}\n        seen = set(listed)\n        lists = source_lists(payload, mid)\n        depth = max((len(v) for _, v in lists), default=0)\n        picked = checks = 0\n        for r in range(depth):\n            if picked >= per_molecule or checks >= max_checks:\n                break\n            for name, values in lists:\n                if r >= len(values) or picked >= per_molecule or checks >= max_checks:\n                    continue\n                s = str(values[r]).strip()\n                if not s or s == \'CCO\' or s in seen:\n                    continue\n                seen.add(s)\n                checks += 1\n                key = identity(s)\n                if key is None or key[0] in taken:\n                    continue\n                taken.add(key[0])\n                picked += 1\n                rows.append((mid, s, key[0], name, r + 1, 25 + picked, True))\n    return pd.DataFrame(rows, columns=COLUMNS)\n\n\nif __name__ == \'__main__\':\n    frozen, root, evidence_path, out = sys.argv[1:5]\n    per = int(sys.argv[5]) if len(sys.argv) > 5 else 12\n    checks = int(sys.argv[6]) if len(sys.argv) > 6 else 30\n    t0 = time.time()\n    identity = load_identity(frozen)\n    payload = json.loads((Path(root) / \'e004_candidate_inputs.json\').read_text())\n    evidence = pd.read_parquet(evidence_path)\n    frame = collect(payload, evidence, identity, per, checks)\n    frame.to_parquet(out, index=False)\n    by_source = frame.source.value_counts().to_dict() if len(frame) else {}\n    print(\'OUTSIDERS\', len(frame), \'candidates for\', frame.molecule_id.nunique(), \'molecules\',\n          json.dumps(by_source), f\'{time.time() - t0:.0f}s\', flush=True)\n'
MODULE_SHA256 = {
 "runner.py": "86b74f3b4cd5767a2cbe1d291735c292b66b4fbf286a705c6cb777a1865883db",
 "export_evidence.py": "d4ec36a9f6216d2607fff4f481bc465dbc8fbc616ec4b24f12932e4c5e546e19",
 "neural_scores.py": "cb9e4d6732992e3310e951533fd0c644d3b515134ad03768c35e5a86551267d2",
 "submission.py": "d0027a893b1bfefe8066ede4f625c584fe0d4fa16bfe4bee32265ef34a7f007c",
 "ranking.py": "6f909cd7e41e88387c23109b93dd590885d2283509804279a58c624a758987d7",
 "bundle.py": "8595ef16ead9c9e6ac34a3398bff4b748f5668cdc517c396be18100a5b745cea",
 "consensus.py": "9e67ab49a852404587e62ca84e0fc551d70ce68ec8f7a47739aada6ca08d3989",
 "outsiders.py": "4dcf6c2b9c9b7be3ac8f7a616683495430fcb4e0c25b449684d132daaa8e7b5c"
}
MODEL_PINS = {
 "features.py": "a97ad9b3ab3711517864d56e497e1672cb29bb05683c72b52db9f9b0a17b1455",
 "model.py": "2547418857386d1c2513fd1ab6c70961e6a96e73d9d495bbfe3052197e69dc39",
 "config.json": "563fb2a2a81ca48ea094f9ab78c157a48a6e9a5503e85e6ce8f0fb3615c1cc03",
 "final_model.pt": "44436b2daec64373e66f9120dd2389c24fc83dd9430ebeeac18e3cb49dcb9642"
}
FROZEN_SHA256 = '47db922027a44fcc91c96095e05ef743263bef805b86e117cc8ecdb745515e2a'   # elig0: frozen 0.465 notebook with the elig0 patches (original d1e757c2b71b...)
FROZEN_B64 = '...'
print(f'{len(MODULES)} modules and the pinned pipeline source ready ({len(FROZEN_B64) * 3 // 4 / 1e6:.2f} MB)')


# ── Eligibility (ours, elig0): mounted datasets, input hash check, null Engine 2 module ─────────────────────
# Generated by scripts/make_pub_variant.py ELIG=0 (scripts/pub_variant/elig.py, elig_cell.py, null_engine.py).
ELIG_DATASETS = ['ahmedberatozer/casmi26-v4b-models', 'ahmedberatozer/casmi26-v3-models', 'ahmedberatozer/casmi26-fpnet-full1', 'ahmedberatozer/casmi26-v2-pool', 'ahmedberatozer/casmi26-iceberg', 'ahmedberatozer/casmi26-glacier', 'dmitriigluzdov/casmi26-pubchem-popularity-prior', 'dmitriigluzdov/casmi26-fold-safe-fpnet', 'sanpier/casmi-spectrum-model-v1', 'metric/rdkit-2026-3-3-wheel']
ELIG_UPSTREAM = ['ahmedberatozer/casmi26-v4b-models', 'ahmedberatozer/casmi26-v3-models', 'ahmedberatozer/casmi26-fpnet-full1', 'ahmedberatozer/casmi26-v2-pool', 'ahmedberatozer/casmi26-iceberg', 'ahmedberatozer/casmi26-glacier', 'dmitriigluzdov/casmi26-pubchem-popularity-prior', 'dmitriigluzdov/casmi26-fold-safe-fpnet', 'sanpier/casmi-spectrum-model-v1', 'metric/rdkit-2026-3-3-wheel']
ELIG_DETACHED = ['dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets', 'dmitriigluzdov/casmi26-natural-product-knowledge-table', 'dmitriigluzdov/casmi26-natural-product-panel-benchmark', 'sanpier/catalogue-experiment-saner']
ELIG_CHECKED_VERSIONS = {'ahmedberatozer/casmi26-v4b-models': '2026-09-26 12:21:47', 'ahmedberatozer/casmi26-v3-models': '2026-09-23 10:45:15', 'ahmedberatozer/casmi26-fpnet-full1': '2026-09-29 10:10:49', 'ahmedberatozer/casmi26-v2-pool': '2026-09-21 08:18:58', 'ahmedberatozer/casmi26-iceberg': '2026-09-26 09:48:06', 'ahmedberatozer/casmi26-glacier': '2026-09-28 09:32:41', 'dmitriigluzdov/casmi26-pubchem-popularity-prior': '2026-10-27 17:07:06', 'dmitriigluzdov/casmi26-fold-safe-fpnet': '2026-10-21 14:41:29', 'sanpier/casmi-spectrum-model-v1': '2026-10-23 17:40:23', 'metric/rdkit-2026-3-3-wheel': '2026-09-08 22:23:23'}
# full SHA-256 of larger files (v4g container MANIFEST.json 2593c1e95b98...; v3 fpnet_0 / 1 = v4b fe_A / fe_B)
ELIG_FULL_REFS = {'ahmedberatozer/casmi26-v2-pool/pool_fp.npy': (908986707, 'f02d91eeb4e2be5687ce678c199aa09a3acda1e821478372fc23fd1c7edf55e8'), 'ahmedberatozer/casmi26-v2-pool/pool_frag_mass.npy': (357116708, '9afbdc02856b287e06e6c9f61a9f0032b2dd53b20557942c1d00542d9c490ffc'), 'ahmedberatozer/casmi26-v2-pool/pool_meta.parquet': (36692358, '82b8c40315611289b9a4004b3af18d6d3402b3cd5baa4ee5723da07b7e2f6bb0'), 'ahmedberatozer/casmi26-v2-pool/train_fp_sel.npy': (352761118, '0e6ddfbfb188b6cffb9f1b0ad9c7ba268ef709ce60b2711d05f98096cd772a8d'), 'ahmedberatozer/casmi26-iceberg/wheels/rdkit-2025.3.6-cp312-cp312-manylinux_2_28_x86_64.whl.ice': (36119142, 'ed5565d4d6423def7c52eb20a6263afcd979383757086dec92f57e77d7c7f856'), 'ahmedberatozer/casmi26-v3-models/fpnet_0.pt': (198790165, 'edccf479a2a2e53461a06bc5e1aee4341b22abb32302c4f6d2c45281000d742a'), 'ahmedberatozer/casmi26-v3-models/fpnet_1.pt': (198790165, '6f60a22b3c3b21aa48d47328bb9523bdf07cc004bfb72966e0f9713507c24f2e')}
ELIG_FULL_DATASETS = ['ahmedberatozer/casmi26-v3-models', 'ahmedberatozer/casmi26-fpnet-full1', 'ahmedberatozer/casmi26-v2-pool', 'ahmedberatozer/casmi26-iceberg', 'ahmedberatozer/casmi26-glacier', 'metric/rdkit-2026-3-3-wheel']
ELIG_FROZEN = {'original_sha256': 'd1e757c2b71b46e6e8a5777f14302a4737d0fdbcd56d8365093c039b93c5fd88', 'patched_sha256': '47db922027a44fcc91c96095e05ef743263bef805b86e117cc8ecdb745515e2a', 'patched_cells': [5, 9, 12, 14, 16, 18, 26, 32, 36, 39, 45, 54], 'skipped_cells': [9, 12, 14, 16, 18]}
NULL_ENGINE_SHA256 = 'c96661897a3ed61be8f6a11615abfcaf98ddf3ae16feface817dd39d9a7ae1fb'
NULL_ENGINE_SRC = '"""Null Engine 2 of the eligible fork (elig0; fix F2a of docs/fork_eligibility.md). Ours, MIT.\n\nEngine 2 of the frozen 0.465 pipeline (prvsiyan FP models / ranker rows / COCONUT bank, megayak rows, sergiomalv\nChEBI/LMSD, franciscoangulo PubChemLite) is not run. Every reader of its output (ENG, eng_lists.json) gets Engine 1\'s\nown lists instead, in Engine 2\'s schema (eng_runner.py of frozen cell 32):\n\n    {molecule_id: {\'smiles\': [smiles, ...], \'keys\': [key, ...]}}      (top 40 per molecule)\n\n`null_engine_lists` builds that dict: each molecule\'s ranked list is copied verbatim (same SMILES, same order, the\nplaceholder \'CCO\' and empty entries dropped), then the tail is appended (only SMILES whose key is new for the\nmolecule) up to n entries. Keys come with the list (`keys`) or from `key_of`; they are the tautomer-canonical\nInChIKey14 (Ahmed\'s chem.score_key), the identity Engine 2 wrote (its canon_key) and that cell 45 fuses on.\nCopying the list verbatim (repeats included) makes a weighted reciprocal-rank fusion of the list with itself\norder-preserving, whatever identity the fusion uses: every entry scores (1 + alpha) times its own score.\nPure functions; only write_lists touches the disk.\n"""\nimport json\nimport os\n\nPLACEHOLDER = \'CCO\'      # the frozen pipeline\'s empty-list placeholder; Engine 2 never emitted it\n\n\ndef _get(mapping, mid):\n    """mapping[mid] for a str or raw molecule id; None when absent (or mapping is None)."""\n    if mapping is None:\n        return None\n    if mid in mapping:\n        return mapping[mid]\n    return mapping.get(str(mid))\n\n\ndef _usable(smiles):\n    """The frozen readers\' own filter (cells 45, 49, 52: `s and s != \'CCO\'`), so list positions line up exactly."""\n    return isinstance(smiles, str) and smiles != \'\' and smiles != PLACEHOLDER\n\n\ndef null_engine_lists(mids, lists, keys=None, tails=None, key_of=None, n=40):\n    """Engine 2\'s schema from Engine 1\'s lists.\n\n    mids:    molecule ids to cover; each one gets an entry (empty lists when it has no candidate), so the frozen\n             asserts that Engine 2 covers every active molecule hold (cells 48 and 50).\n    lists:   {mid: ranked SMILES}, copied verbatim (placeholder / empty entries dropped), at most n.\n    keys:    {mid: keys parallel to lists[mid]} (Engine 1\'s own keys), or None to take key_of(smiles).\n    tails:   {mid: ranked SMILES} appended after the list; an entry is kept only when its key (key_of) is truthy and\n             new for the molecule and its SMILES is not listed yet.\n    key_of:  smiles -> key (required when keys is None or tails is given).\n    n:       maximum entries per molecule (Engine 2 wrote its top 40).\n    Returns {str(mid): {\'smiles\': [...], \'keys\': [...]}} in the order of mids (first occurrence of each id)."""\n    assert isinstance(n, int) and n > 0, n\n    out = {}\n    for mid in mids:\n        m = str(mid)\n        if m in out:\n            continue\n        head = list(_get(lists, mid) or [])\n        if keys is not None:\n            head_keys = list(_get(keys, mid) or [])\n            assert len(head_keys) == len(head), (m, len(head_keys), len(head))\n        else:\n            assert key_of is not None or not head, \'key_of is required without keys\'\n            head_keys = [key_of(s) if _usable(s) else None for s in head]\n        smiles, ids = [], []\n        for s, k in zip(head, head_keys):\n            if len(smiles) >= n:\n                break\n            if _usable(s):\n                smiles.append(s)\n                ids.append(k)\n        tail = list(_get(tails, mid) or [])\n        if tail and len(smiles) < n:\n            assert key_of is not None, \'key_of is required for tails\'\n            seen_keys, seen_smiles = {k for k in ids if k}, set(smiles)\n            for s in tail:\n                if len(smiles) >= n:\n                    break\n                if not _usable(s) or s in seen_smiles:\n                    continue\n                k = key_of(s)\n                if not k or k in seen_keys:\n                    continue\n                smiles.append(s)\n                ids.append(k)\n                seen_keys.add(k)\n                seen_smiles.add(s)\n        out[m] = {\'smiles\': smiles, \'keys\': ids}\n    return out\n\n\ndef summary(eng, mids=None):\n    """One log line; \'engine lists <N> \' matches the notebook\'s compact-trace filter."""\n    rows = list(eng.values()) if mids is None else [eng.get(str(m), {\'smiles\': []}) for m in mids]\n    sizes = [len(r[\'smiles\']) for r in rows]\n    return (f\'engine lists {len(eng)} | nonempty {sum(1 for s in sizes if s)} | candidates {sum(sizes)} | \'\n            f\'min {min(sizes) if sizes else 0} max {max(sizes) if sizes else 0}\')\n\n\ndef write_lists(eng, path):\n    """Write eng_lists.json (Engine 2\'s file) atomically; keys that are numpy strings are written as str."""\n    tmp = str(path) + \'.tmp\'\n    with open(tmp, \'w\') as handle:\n        json.dump(eng, handle, default=str)\n    os.replace(tmp, path)\n    return path\n'
V43_HASHES_SHA256 = 'a62f4f7e632639963116113c4d657907cebcef54462f89260960cf70e280cc89'   # compact copy of casmi26-pubm's file (original sha256 b07ab9e3b6ee...)
V43_HASHES_JSON = '[{"path":"/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv","bytes":43619,"sha256":"34c45718efeb03b9d130c7f1ed5a1a72f92c8818479a49d5feef08b5a9f18e04","full":true},{"path":"/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet","bytes":4848729,"sha256":"2fa29c4aa06e93b48e0cd6619fd2de3f4e956f0019d242141678b7287f2b43e0","full":true},{"path":"/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet","bytes":3033286496,"sha256_head_tail":"df3579eb4bd7f3b55f6f39f769fd3137abb7cb1ff2e6d9243297f38273ed2fa1","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/ATTRIBUTION.md","bytes":5124,"sha256":"6918a04bda1d78fee311ac2acf29b1a3736321e74b68c44d05bcb98f7062b728","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/MANIFEST.json","bytes":45195,"sha256":"2593c1e95b989309e9c5057d1c00216c22c75f50d87a960b363d24f99f1bad8f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/README.md","bytes":9323,"sha256":"db3790cee6e5ba1afe228f6fc30b4ccfeb48c993b09fac673877ab498f6c6d12","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/SOURCE-LICENSES.md","bytes":1962,"sha256":"0343e31e5a9e064b524491bfe02a5f0f127ad9643a049882cd7e74bdd7014b94","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi-chebi-lmsd-structures/ATTRIBUTION.md","bytes":2428,"sha256":"fb49f6a959f562b7edb50a9103eccbbdf55a2c38327f09b989a369311fd280d7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi-chebi-lmsd-structures/bio_fp.npy","bytes":85341539,"sha256_head_tail":"037d393c90b5d7ca9b20a289c1ed94553cf955616db6062d1cd77e0816e1528b","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi-chebi-lmsd-structures/bio_mass.npy","bytes":787592,"sha256":"2407a2ee04bd6bc468db876e7c0569842e7652432386cf1101dc85ef5a5611df","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi-chebi-lmsd-structures/bio_meta.pkl","bytes":14266637,"sha256":"bfc92d81dafe8244a86f3a7b74b0da66b61605a9581b78c60acea619908404d4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi-chebi-lmsd-structures/bio_structures.parquet","bytes":5407545,"sha256":"6bef7106d640ed370046e79d67b86932ba26da3a3632e6901e3b040fad56aa88","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi-chebi-lmsd-structures/stats.json","bytes":608,"sha256":"dd0d5686dd54824a40857b75338ed03dc922a2580031d246bf1995b43af57a5e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-fp-models-v2/fp_merged_m1.pt","bytes":144110255,"sha256_head_tail":"d21da200e05f5de323e7fd2b69e9106e182842b28066e87dd939d3b3bdb6a3b1","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-fp-models-v2/fp_single_s2.pt","bytes":144110255,"sha256_head_tail":"d8399f203383cfd1325b47e04bbba4a76ec80c3ddb658a36c7828c1f8a483c48","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-fpnet-full1/fpnet_full1.pt","bytes":198791233,"sha256_head_tail":"9f3d11798b4e9051bdc1bff3478fe0a7a81b865a6ba84f3b5194ee0511f44256","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/LICENSE-NOTICE.txt","bytes":1489,"sha256":"563c9951ffb47543beeff63df8006b0ba5ab9785a23acfb5311a6b4ceaf44f55","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/MANIFEST.json","bytes":6286,"sha256":"53bf3f8724cb0d8ddbb104254deda88b0a90502e3ab031aa42cbbf010b38634d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/README.md","bytes":2423,"sha256":"430b17c774e84f76204aa83b2672d39d31de9f1479eff8a2a3663bcc8ce000ac","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/ckpt/glacier.pt","bytes":60531887,"sha256_head_tail":"ec42d203c09a167fc2d0dc9020679f05de07ec5920ff26eca2da21b737b88788","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/gl_fuse.py","bytes":7533,"sha256":"ea390f5f29705c015f39c509cc2b86e6a360d0bb90db61c87c507068f38ae58e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/gl_runner.py","bytes":39880,"sha256":"5c4f74313ac54263d05d987c39d012d81ec57c5f35e1095ecd49a5bcb6440fca","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/shim/LinSATNet/__init__.py","bytes":3656,"sha256":"a0ffdf64f610ad6a86ef560d3ff62f06a9c2b53ed13473d85eae87953f39e6af","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/shim/dgl/__init__.py","bytes":10029,"sha256":"35dd2028bbb1fb14442d7a4cbbd9cb260c1063ab05dd556ddaf1f77aa2f3d2e2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/shim/dgl/backend/__init__.py","bytes":36,"sha256":"f85e782b33669b9567ba24c46931db87506637870d723c8696e66515ff0f9ee7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/shim/dgl/backend/pytorch.py","bytes":964,"sha256":"e7bf3d120f37ebc40fb8dec000f637ec516b75610b21717a4d7abe3cb40f37c7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/shim/dgl/function.py","bytes":1803,"sha256":"4dd78301a49129085e3234ff902999c7921979a5e09e5415a3aaf3a0aa9d27ff","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/shim/dgl/nn/__init__.py","bytes":1958,"sha256":"cd78a6aaee83a58cadb14a9a41f8c8e92a07c3c2bba52f6a83b4eb8ac4fd62a2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/shim/torch_scatter/__init__.py","bytes":3120,"sha256":"d8f0b42adc06bb499514f68378c826900cbcb2e9764a4ea403bc160497c842a6","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/LICENSE","bytes":1092,"sha256":"cb323bf6812b7ebc608d6ded23cf0245ab4633d68b1998010ece3ea3d2810fe3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/__init__.py","bytes":266,"sha256":"e4a927730a084dd169e994b1fa712c5e17709e0a869f00aadb4709df9855c487","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/common/__init__.py","bytes":381,"sha256":"521b8cd399abda6a52cd1cc3304d46557035c70f3332be50122c9e68032d1351","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/common/chem_utils.py","bytes":26034,"sha256":"c9842ef76eb831394c53fdd5050d87842edaebd46427a722d43983286495f96e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/common/denoising_utils.py","bytes":3478,"sha256":"34f9658456896a4adcd4302c783d7253727fe527454bec2b7c91f224a4cc0051","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/common/fingerprint.py","bytes":1649,"sha256":"d3032ecff1a43472af33ce1ea307223941b014290b7507c44e9ee028f5d764d3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/common/misc_utils.py","bytes":96988,"sha256":"bbee369888121c45384a4c93c5c0811bd3b4d8ba9bd038aa119f04b1ca4d1122","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/common/parallel_utils.py","bytes":13635,"sha256":"60dbeac1f6d2b52009b40329790d6ca05fdd28efb70a1c23586616d6c984c691","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/common/splitter.py","bytes":2676,"sha256":"47ea4c05697ddaa8bafe777c58ebb7ff0d9de762a5ee8345b6ec38cf9496827c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/glacier/dataset.py","bytes":46377,"sha256":"547a09d8629a45c8fad43dcb60277a36906b1d382520c959950562d109014360","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/glacier/joint_model.py","bytes":76110,"sha256":"9481602423011e830c94b8fcd857ac2be3cb31b2c24d714a3ae0c1edd235c293","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/graphormer/__init__.py","bytes":351,"sha256":"b31b131343597bb736ee984291cbe0688383b507cd6b22d4d9ca01159ac440f2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/graphormer/graphormer_graph_encoder.py","bytes":6925,"sha256":"074a59296c0a2903fd5018c27705ea1da2393f09c84ba30fb32acbf3827729ea","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/graphormer/graphormer_graph_encoder_layer.py","bytes":3593,"sha256":"d98ba908616e231187afe2409409c06269e5d2017bd12c0d85c6ef093e8272f7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/graphormer/graphormer_layers.py","bytes":5937,"sha256":"26c8d693bebc8904a975aeb11cf343b9c7a0fd9a120f828ed597bd8180bc606d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/graphormer/multihead_attention.py","bytes":2404,"sha256":"f11b79e3e4f8147cbbe97638e27b7046dfb8c2ed714b35409925f9bb4c5f0815","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/iceberg/dag_data.py","bytes":36155,"sha256":"80a4f2abd57a59077bf988c9a3f975de36cf856c189aadf9f871382054494814","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/iceberg/graph_mutate.py","bytes":9359,"sha256":"06c343312d44eedc1498c5513b33bcf5b1bc950f05dcb87e18827a0a7a81c55d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/magma/fragmentation.py","bytes":28327,"sha256":"5fc6ac347a09ea8486154ce827eea0ea810a0c7f6a277279ffd4c5abd1e93744","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/nn_utils/__init__.py","bytes":378,"sha256":"2f7f5b3d2537b6bde6a28be84908826901cb2306756b680a57934c39831ade57","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/nn_utils/dgl_graph_ops.py","bytes":18955,"sha256":"d65b239032eae4ec8cb2d0b9850fd773542722ff953c96827ab59000d13eba9f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/nn_utils/dgl_modules.py","bytes":23408,"sha256":"b98dfc02dad06a67e2c9ee61670ebc0c96a8b2a62859335888f8189d5677e2e4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/nn_utils/form_embedder.py","bytes":10681,"sha256":"30bb3670c8b21e01ebb3fb2548dac7f8e0cac589fd4cfccc51589ddf55c1ea4b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/nn_utils/mol_graph.py","bytes":10573,"sha256":"62d11349b0c9d06335384b6fabfcddffaf78ece4c1d6f5c18ac426746d1e7d89","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/nn_utils/nn_utils.py","bytes":41337,"sha256":"f07a9018515ed0204b305741c5b13db9285d400ee7ff3e5bfa12db756a87146f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-glacier/src/ms_pred/nn_utils/transformer_layer.py","bytes":30192,"sha256":"8884d91afd479bef9b05d4a4527953ade8fd9353332ca3c6eddb8e1b048d2a92","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/LICENSE-NOTICE.txt","bytes":1003,"sha256":"2bdc24cdcfda1c536a17d8af6a85b6028b19369527d84db8cc81800228897bce","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/MANIFEST.json","bytes":6163,"sha256":"78ab2dff112b1e9a043a8025bf8b57839be3f4b94f2e87fc61becbef4c804f8b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/README.md","bytes":3879,"sha256":"1a36864cb19161b84742f9cb8f323f795b5b3e8e1d739474b7b205acd2d8601f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/ckpt/gen.pt","bytes":14561244,"sha256":"deef0451c1870d2125653bb212c9d95c5b5d9bf80156dfa9b22fedda7db49bea","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/ckpt/inten.pt","bytes":13793037,"sha256":"c49e330e049d3a8b0095026ed694fcf70045c24305354853732aaa5f89d55068","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/fuse.py","bytes":8349,"sha256":"6d1bdf0c63ff8844d6994bb83cdadf07951ee40ed87ea5a2d870577c6fc76bb1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/ice_runner.py","bytes":31551,"sha256":"dc0d5d1e14053e2e7d9088817c620d6615372abc3afdd02be2fc7ff9ee0b2802","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/shim/dgl/__init__.py","bytes":10029,"sha256":"35dd2028bbb1fb14442d7a4cbbd9cb260c1063ab05dd556ddaf1f77aa2f3d2e2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/shim/dgl/backend/__init__.py","bytes":36,"sha256":"f85e782b33669b9567ba24c46931db87506637870d723c8696e66515ff0f9ee7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/shim/dgl/backend/pytorch.py","bytes":964,"sha256":"e7bf3d120f37ebc40fb8dec000f637ec516b75610b21717a4d7abe3cb40f37c7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/shim/dgl/function.py","bytes":1803,"sha256":"4dd78301a49129085e3234ff902999c7921979a5e09e5415a3aaf3a0aa9d27ff","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/shim/dgl/nn/__init__.py","bytes":1958,"sha256":"cd78a6aaee83a58cadb14a9a41f8c8e92a07c3c2bba52f6a83b4eb8ac4fd62a2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/shim/torch_scatter/__init__.py","bytes":3120,"sha256":"d8f0b42adc06bb499514f68378c826900cbcb2e9764a4ea403bc160497c842a6","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/LICENSE","bytes":1092,"sha256":"cb323bf6812b7ebc608d6ded23cf0245ab4633d68b1998010ece3ea3d2810fe3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/__init__.py","bytes":266,"sha256":"e4a927730a084dd169e994b1fa712c5e17709e0a869f00aadb4709df9855c487","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/common/__init__.py","bytes":381,"sha256":"521b8cd399abda6a52cd1cc3304d46557035c70f3332be50122c9e68032d1351","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/common/chem_utils.py","bytes":26034,"sha256":"c9842ef76eb831394c53fdd5050d87842edaebd46427a722d43983286495f96e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/common/denoising_utils.py","bytes":3478,"sha256":"34f9658456896a4adcd4302c783d7253727fe527454bec2b7c91f224a4cc0051","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/common/fingerprint.py","bytes":1649,"sha256":"d3032ecff1a43472af33ce1ea307223941b014290b7507c44e9ee028f5d764d3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/common/misc_utils.py","bytes":96988,"sha256":"bbee369888121c45384a4c93c5c0811bd3b4d8ba9bd038aa119f04b1ca4d1122","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/common/parallel_utils.py","bytes":13635,"sha256":"60dbeac1f6d2b52009b40329790d6ca05fdd28efb70a1c23586616d6c984c691","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/common/splitter.py","bytes":2676,"sha256":"47ea4c05697ddaa8bafe777c58ebb7ff0d9de762a5ee8345b6ec38cf9496827c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/iceberg/dag_data.py","bytes":36155,"sha256":"80a4f2abd57a59077bf988c9a3f975de36cf856c189aadf9f871382054494814","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/iceberg/gen_model.py","bytes":34864,"sha256":"467a117823c4e1885636cbad4d4925f69c68722aea460cf8fc91e08cfcb6c755","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/iceberg/inten_model.py","bytes":34601,"sha256":"821942bdcb0282dd229deb6cafcfd6f07ab29eeb9539446acaa88229d6341ff6","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/iceberg/joint_model.py","bytes":7968,"sha256":"d1cd923d0729e6a0b141f97926e4ed8d4172b01ee4b0212781f3835f2f558f74","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/magma/fragmentation.py","bytes":28327,"sha256":"5fc6ac347a09ea8486154ce827eea0ea810a0c7f6a277279ffd4c5abd1e93744","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/magma/run_magma.py","bytes":23083,"sha256":"b55f7b3914835e53641ca3e0c337e7b1b5acc61d914268fd26d61263d0f56b6d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/nn_utils/__init__.py","bytes":378,"sha256":"2f7f5b3d2537b6bde6a28be84908826901cb2306756b680a57934c39831ade57","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/nn_utils/dgl_graph_ops.py","bytes":18955,"sha256":"d65b239032eae4ec8cb2d0b9850fd773542722ff953c96827ab59000d13eba9f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/nn_utils/dgl_modules.py","bytes":23408,"sha256":"b98dfc02dad06a67e2c9ee61670ebc0c96a8b2a62859335888f8189d5677e2e4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/nn_utils/form_embedder.py","bytes":10681,"sha256":"30bb3670c8b21e01ebb3fb2548dac7f8e0cac589fd4cfccc51589ddf55c1ea4b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/nn_utils/mol_graph.py","bytes":10573,"sha256":"62d11349b0c9d06335384b6fabfcddffaf78ece4c1d6f5c18ac426746d1e7d89","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/nn_utils/nn_utils.py","bytes":41337,"sha256":"f07a9018515ed0204b305741c5b13db9285d400ee7ff3e5bfa12db756a87146f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/src/ms_pred/nn_utils/transformer_layer.py","bytes":30192,"sha256":"8884d91afd479bef9b05d4a4527953ade8fd9353332ca3c6eddb8e1b048d2a92","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/wheels/einops-0.8.2-py3-none-any.whl.ice","bytes":65638,"sha256":"54058201ac7087911181bfec4af6091bb59380360f069276601256a76af08193","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/wheels/lightning_utilities-0.15.3-py3-none-any.whl.ice","bytes":31906,"sha256":"6c55f1bee70084a1cbeaa41ada96e4b3a0fea5909e844dd335bd80f5a73c5f91","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/wheels/pytorch_lightning-2.6.6-py3-none-any.whl.ice","bytes":853045,"sha256":"71f95c7b22f25c4f91cc659e1734bcdc6c69c8b66543f2ed2d78dbe29ce2f167","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/wheels/rdkit-2025.3.6-cp312-cp312-manylinux_2_28_x86_64.whl.ice","bytes":36119142,"sha256_head_tail":"8dd92140c41e70f8199165f5dac0fc0965f7161457cce193a8419fc9fabb5ac3","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-iceberg/wheels/torchmetrics-1.9.0-py3-none-any.whl.ice","bytes":983384,"sha256":"bfdcbff3dd1d96b3374bb2496eb39f23c4b28b8a845b6a18c313688e0d2d9ca1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-pubchemlite-cc0-candidate-fingerprints/MANIFEST.json","bytes":920,"sha256":"f333f6fa150f807f2981faa7dcb930b90c6f5022015f7314f428cb8bd84e42cb","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-pubchemlite-cc0-candidate-fingerprints/pubchemlite_fp.npy","bytes":334454048,"sha256_head_tail":"c8348e30bbf8d4b3294cab083e1e55841a49dc8334be9211df2ab59ef5008fe2","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-pubchemlite-cc0-candidate-fingerprints/pubchemlite_mass.npy","bytes":3086208,"sha256":"f107bdf361b75fb2f6115abb2a28eda053b476b6c3694f4b0a2bfdadb248730f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-pubchemlite-cc0-candidate-fingerprints/pubchemlite_meta.pkl","bytes":23642096,"sha256":"3c9a18c7662022bce51759b1660292b712fe7544459020e38299d2f0747517ce","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-ranker-features/rank_train.npz","bytes":9553020,"sha256":"41ccec87fba5f8ab255aaf5a723567f25bd766064265c5cc2b83ba2cf15f57f8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-simulated-ranker-rows/README.md","bytes":1879,"sha256":"89757d07af932c35e4d73e505bfc4a0e81655d04a5722903e879b026563336b4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-simulated-ranker-rows/ours_fpnet_merged_26k.pt","bytes":144111564,"sha256_head_tail":"07fe04872abfe41e5bdff101d9c21a70953f56a78aa623d0a1e2c8ab7b9aefbd","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-simulated-ranker-rows/ours_fpnet_single_16k.pt","bytes":144111657,"sha256_head_tail":"e11ea32bdf88866fa32fe1ed86c5f52556aa5815dd14bce8defa20beacde68a1","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-simulated-ranker-rows/sim_rank_rows_fp16k.npz","bytes":25439306,"sha256":"d3cc072767e667b43d5d4d88a1eded08aba4dab5fb56f502353d9e8ed6c42456","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-simulated-ranker-rows/sim_rank_rows_fppair.npz","bytes":25437252,"sha256":"20e5d7d0c5a4ed63b8e79884d5cc681fd60e8bfa94ecac103994354385432801","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-simulated-ranker-rows/sim_rank_rows_nofp.npz","bytes":26872207,"sha256":"c19ae76f69c7a3f23e67d2e27441830213eee941aa5658b47b1798872a61a7e9","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol/TRAINING.md","bytes":3536,"sha256":"a3a6d86205670948e8ea2518bafa5b8af6e4b160a51dc3b68b5768ddffaae097","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol/held_out_molecules.csv","bytes":18738,"sha256":"c846c7e58ac65201dc57ef1ea4bd2945f1240bcf1a761ae9d807df255b8f5d61","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol/held_out_sids.npy","bytes":2248,"sha256":"dc8dd384ebdfcc2c301d7ac02b3a8f2306d21a3eea96f02f75bfcd239d85dc30","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol/history.json","bytes":3269,"sha256":"4dfb1931601ea8a81f87574395abcc8da102c73bb7e79b6a3a774d2758e38398","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol/prep.py","bytes":5583,"sha256":"e0734797a108817a9c3880f81750430fb8e891cbbaf0d6acb8214797659e216f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol/spec2mol.pt","bytes":66731290,"sha256_head_tail":"82422cc76f831e52c6dcc84ffcda11e9c658169c61ef3d78fe1ed2c9f3d1cf77","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol/spec2mol_features.py","bytes":1512,"sha256":"0fd9286aff5564dc327511f17a4af23b3d52eb14515e92ea828024cab9ae80be","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol/spec2mol_model.py","bytes":2528,"sha256":"c2bfc970392ec862f684d2fc2e0d2f77a80863cc87b40ef3f442d0218ff5e1e4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol/train.py","bytes":6724,"sha256":"83f0be770170d3f7ceaf5b32a9221461c34ba05ec5636d6640d420802bac5283","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol-r/TRAINING.md","bytes":2255,"sha256":"c90b1b069c0b4d8eeec0c58a4a426e7f42bf375443cff365e2717c37fc59a79a","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol-r/history_val.json","bytes":430,"sha256":"1f693695917f82597d84608aa6ae2dc9035e1dbab8f3f2833e91636b98347402","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol-r/prep_morgan.py","bytes":1360,"sha256":"ebb201f064e10d13056d743b09025b4bbca65f7f2317bd48063d9e7eb0a19ecc","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol-r/spec2mol_r.pt","bytes":11893127,"sha256":"b78ec5949fec91678dc5b8c8c4b4dca00b38adf9454c36b2cfe7d81500a53cdf","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol-r/spec2mol_r_features.py","bytes":870,"sha256":"c31cfc0730dd1208b1644c96070ee20731dcc4ef6e62fa9077f05dd304867dcd","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol-r/spec2mol_r_model.py","bytes":2707,"sha256":"a688a23ce0533508cb08e103a29d932ff6597ae4fc20c46d9309e2a3f63ed372","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol-r/spec2mol_r_spectrum.py","bytes":1512,"sha256":"0fd9286aff5564dc327511f17a4af23b3d52eb14515e92ea828024cab9ae80be","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-spec2mol-r/train_r.py","bytes":6961,"sha256":"938a4352ec63a3a106a4fc98b30d2e865ef20e7731e46d0e1753714d1fe15db0","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v2-pool/fp_bits.npy","bytes":41032,"sha256":"c3869e86449d6fed6e79e9655f8a7d35bd55c3138ecddfc1dae5072627e78677","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v2-pool/pool_fp.npy","bytes":908986707,"sha256_head_tail":"21008c78d9625b4c920c47597648521bbb6936cc58707dd5d24f5deb2a4deca0","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v2-pool/pool_frag_mass.npy","bytes":357116708,"sha256_head_tail":"8231024539fe9d63a594ebc04e872dcc10b6254238f7b1833f05dc2edcc8699f","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v2-pool/pool_frag_off.npy","bytes":5685744,"sha256":"354015a6eab8b6762f9124384e959d606e961c35b999d1b088153d69538765e7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v2-pool/pool_meta.parquet","bytes":36692358,"sha256_head_tail":"2e40fd0b297d40014ce18f462ed2c16e0ed60328fe7bd970a40c882fb0528074","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v2-pool/train_fp_sel.npy","bytes":352761118,"sha256_head_tail":"69e68823cce84e6ee20c9f2113468d37249aaa7818825e84e39ded06746dc1a7","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v2-pool/train_structs.parquet","bytes":23306834,"sha256":"f20aa1cac23af42e430a6633707fda34e48dea97114bdc3e443897f692df3656","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/__init__.py","bytes":0,"sha256":"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/build.py","bytes":4979,"sha256":"86776c14e237dc64dbf965d9bc200cc670dc7226bf946938380729d7e80c56f8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/chem.py","bytes":12328,"sha256":"99474f23d8a67040b0c0f8c70e73592a6f7f656f50103e350ca473451e2169f1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/derive.py","bytes":6929,"sha256":"7c1e8c3afb4c17a3ae58fbc6c36fd4cfc552da97b9129c9b46cca3cdc6ee4892","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/engine.py","bytes":26449,"sha256":"d914bf9d5eb56cfff434b5d53a4d55881b97697aa834a627d9932a4f8c650d27","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/fpnet.py","bytes":9551,"sha256":"0ea6e372d1f045a877d350234c2ccf5d93fdd98cb23bd4c22376bf7e99547608","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/frag.py","bytes":11172,"sha256":"809f5582494cee4cf6dcace4b04699b8427e81022b78d42162cc0239a57773de","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/graphs.py","bytes":2850,"sha256":"8dcbe16bbf3b34b7a08866234eb9abf2de713d21c93c653e7156e94e9debcd55","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/library.py","bytes":7577,"sha256":"21ee24dee9f39123da518fb17802a394e6dc46b147d5dbdf276210b60185f831","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/pool.py","bytes":2370,"sha256":"279444f6c03834adbbfc0fad321e24187585d7cd5afdc363eff083dc263a0286","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/ranker.py","bytes":4189,"sha256":"a7ee491b5a3d3decb3887afe3931adb10c11964468fb204e8c27b42757424fe4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/spectra.py","bytes":10057,"sha256":"e2481761cdfb2515e8a945751f85f6b9e885680a8e974d7ac35c78bbf279ae74","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/structs.py","bytes":2791,"sha256":"2c70df89e8f96b3e716651fe8815d0a05af3ad26e2b448e552ea53c26ede334a","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/traindata.py","bytes":9425,"sha256":"cecb329d96362f11d16b40a26f1888f092610ff6202812408c13e2c293c6a2bc","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/code/casmi/xenc.py","bytes":6938,"sha256":"89fbb470e6fc085f7363bf79e4517c06d43eed3c1df197b7f374b65dacdb7310","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/fpnet_0.pt","bytes":198790165,"sha256_head_tail":"9b19b196dc6c90e79880d9e37546d6514b6ac04ad6573fbd4128412a55490ec4","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/fpnet_1.pt","bytes":198790165,"sha256_head_tail":"77bd2958d1ccd793b36ea325fb7c11f8f7203b0043775bfd9c242dddec505864","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v3-models/ranker_0.pkl","bytes":14162986,"sha256":"b7d5e5bf7a70b35526a78a999def236c795ae1f114e4e360728f1cea068e8e68","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/LICENSE-NOTICE.txt","bytes":1363,"sha256":"b9c26d77338c03cb4387ae61bf4f6524eb77ee59c8530bc9f8d33a742a913a3b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/MANIFEST.json","bytes":9209,"sha256":"e64b960037769971e146a35932e29ed3f042de1716e4dbe281ebd84a7cfc3b6a","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/__init__.py","bytes":0,"sha256":"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/build.py","bytes":4979,"sha256":"86776c14e237dc64dbf965d9bc200cc670dc7226bf946938380729d7e80c56f8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/chem.py","bytes":12328,"sha256":"99474f23d8a67040b0c0f8c70e73592a6f7f656f50103e350ca473451e2169f1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/derive.py","bytes":6929,"sha256":"7c1e8c3afb4c17a3ae58fbc6c36fd4cfc552da97b9129c9b46cca3cdc6ee4892","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/dreams_model.py","bytes":32798,"sha256":"5c09c3d22e8d2c09678b796cb1d57a449442437847e094f49f6ed2619a73a83c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/engine.py","bytes":26449,"sha256":"d914bf9d5eb56cfff434b5d53a4d55881b97697aa834a627d9932a4f8c650d27","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/fpnet.py","bytes":9551,"sha256":"0ea6e372d1f045a877d350234c2ccf5d93fdd98cb23bd4c22376bf7e99547608","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/fpnet6.py","bytes":14661,"sha256":"32542bb0039b2d24d9911030559f13221e630cef7df0d10c7d65f54424da2633","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/frag.py","bytes":11172,"sha256":"809f5582494cee4cf6dcace4b04699b8427e81022b78d42162cc0239a57773de","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/graphs.py","bytes":2850,"sha256":"8dcbe16bbf3b34b7a08866234eb9abf2de713d21c93c653e7156e94e9debcd55","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/library.py","bytes":7577,"sha256":"21ee24dee9f39123da518fb17802a394e6dc46b147d5dbdf276210b60185f831","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/pool.py","bytes":2370,"sha256":"279444f6c03834adbbfc0fad321e24187585d7cd5afdc363eff083dc263a0286","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/ranker.py","bytes":4189,"sha256":"a7ee491b5a3d3decb3887afe3931adb10c11964468fb204e8c27b42757424fe4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/spectra.py","bytes":10057,"sha256":"e2481761cdfb2515e8a945751f85f6b9e885680a8e974d7ac35c78bbf279ae74","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/structs.py","bytes":2791,"sha256":"2c70df89e8f96b3e716651fe8815d0a05af3ad26e2b448e552ea53c26ede334a","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/traindata.py","bytes":9425,"sha256":"cecb329d96362f11d16b40a26f1888f092610ff6202812408c13e2c293c6a2bc","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/casmi/xenc.py","bytes":6938,"sha256":"89fbb470e6fc085f7363bf79e4517c06d43eed3c1df197b7f374b65dacdb7310","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/chemfeat.py","bytes":6158,"sha256":"8fedf317042f989b1569ca65c0d6d87a0387f9fc9934504420a95d8d973f8746","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/ctxlib.py","bytes":20737,"sha256":"c050e4b0ac937a7d65fa14ac55c6904e2232cb077830a20b653ac6df22922943","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/derive6.py","bytes":8635,"sha256":"0e4b436d6b978b2e2e1af8767d6b3c32a6b1fc0e4b64960d5eeeaa8885ddf68a","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/fams/__init__.py","bytes":0,"sha256":"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/fams/analog_struct.py","bytes":30583,"sha256":"2311219220c65bc83c2cd447a8cdb07db27a984124f6c88d687494556e7337b6","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/fams/derivation.py","bytes":29276,"sha256":"eb9aa474bef8e1d067cadccec0101042878ebb5aedf50d1168dc9dfd884dceae","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/fams/frag2.py","bytes":29368,"sha256":"bf17577ef673f8ab0121caa1c9335dc1e827e2f1eb44b3bbde0b0c0cbe858e6c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/fams/fragnet.py","bytes":46806,"sha256":"d0952d34d227013b8486eff36c30a4233f491d2e32c3f816b9d000c493b7234e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/fams/model_views.py","bytes":3492,"sha256":"aa964b12265bb70144d2d653b600a7f1a4972829032274bab76ea9e39afd7e79","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/fragcaps.py","bytes":6766,"sha256":"ba6280f9a46ca28cd72b8a7a0324724a73fd0c67a13acdd82a8193ac30d17a83","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/hooks_v1.py","bytes":4338,"sha256":"d0cb1e37566a54bbb3082ccd29579ac47fc2526c5d64ef073dc0e628ee5657e8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/pool_coresmi_buf.npy","bytes":19703633,"sha256":"c0b2ce3ff292cae239fbb1f1670f345462dbab31e920b17d2e52bef9f9e5fc53","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/pool_coresmi_off.npy","bytes":5685744,"sha256":"570664567fb313291f46e30aecfd9e82e8526eadde651cb2dbb98ead8d2220d5","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/pool_elsmi_buf.npy","bytes":35033614,"sha256_head_tail":"c3bafd961df9dac067d4adbbe1a83b21eb911e3561dfe194183a7a1464f5b0ee","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/pool_elsmi_off.npy","bytes":5685744,"sha256":"4624def96688547f5edf7fa347cf9498ff0ac80a9fb669dd53d4bfc20212602e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/pool_fvec.npy","bytes":14214148,"sha256":"8eebe4be5c87aa87a498fb998ce013cdc0d0f9aa251d6184ec22f7aad755e07b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/pool_hash.npy","bytes":45484992,"sha256_head_tail":"235265fa351ccc8eae7d841e0133b004430f0a59abcd2d305c3185ece77d5c49","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/pool_num.npy","bytes":11371344,"sha256":"08f8831f7abada09392570fc27af1fd625a23dd2391ddca65bcb88b3cb2c7430","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/train_coresmi_buf.npy","bytes":6940196,"sha256":"5352fdcbea2bc09be258cb4047644491919dd289d28422eb51d8f3fdea3a864e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/train_coresmi_off.npy","bytes":2206616,"sha256":"7f72b42397cb1198521cf09caa72ac4c2788544ba2673e8447cc12fbd6d8aece","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/train_elsmi_buf.npy","bytes":10880676,"sha256":"9f2b3d602cb5ada97d86a10790f30bb7c070b6c990dfe7c099551cd60ee79d87","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/train_elsmi_off.npy","bytes":2206616,"sha256":"4d84e388a066c85d3496b16f670ed2b39e3a297c8490d2625dd80e0a0f547854","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/train_fvec.npy","bytes":5516328,"sha256":"f30560e0d23e04a3d9e23828a49ef0b9f08977789e41fb00005835d3d970cca2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/train_hash.npy","bytes":17651968,"sha256":"cf5c28edcf09e0a7a76df1b44db6a4638a42ca826ddacd9b2f2f8429a6293dc7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/analog_struct/train_num.npy","bytes":4413088,"sha256":"7267bb0bdc9e4d698328c4430f1cac2afd6e37a549ef597ee2eafb922f16997b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/derivation/parity_fixture.pkl","bytes":98158,"sha256":"bc12af267cbcd05942da8903d9058d20cc23d96c91233967d4a20102fe785176","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/derivation/prior_tables.pkl","bytes":20738118,"sha256":"217905818dd187da47cb458a60aeae2a5f2b0139904b4e320f70e41a9bd74c90","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/frag2/caps.txt","bytes":149,"sha256":"0ebe04b72563f828e4a6391bea393fdeb37777f87fc4c24aa408e41ea3d2ced2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/frag2/pool_cap.npy","bytes":15635550,"sha256":"6be43a73f773e50decde505a10e23a3114f09bdc2005fe9c51cc87bf0d851dbe","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/frag2/pool_f1_mass.npy","bytes":62531984,"sha256_head_tail":"757faa3a42dfac0253d5dec333274d7133669bee12add2f635b26836aa527e1a","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/frag2/pool_f1_off.npy","bytes":5685744,"sha256":"de25d731fd3fb09319538949e07c34135a3671ac7bd48dd2dbf9eece08a3121d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/fragnet/countsA.npz","bytes":1568346,"sha256":"42f37e29b933552175b91e732d9bbbcb08c54bea849976dd55858bc115701f10","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/fragnet/featB.npz","bytes":131473600,"sha256_head_tail":"f6654776d4db28dc4b5705409953903ad3f6b3cee4c3bc65bafa2f1eca52eab8","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/fragnet/rec_aoff.npy","bytes":5685744,"sha256":"9aadeb10c9629498e20679aa24fc403b1e1e2b3b76dd9799ffc9c4c7d0c1187b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/fragnet/rec_atom.npy","bytes":66704036,"sha256_head_tail":"fdcfb68be1b680084d1c58e2d08b4daed5c144f13e42c2471af688fca0874184","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/fragnet/rec_boff.npy","bytes":5685744,"sha256":"0d15c8c4072a3bdd23637edeb24843f6adbf9d9dd270adc8fdf82c00436d91da","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/fragnet/rec_bond.npy","bytes":71602625,"sha256_head_tail":"1dcc4cea749085c36b5d3d0016d7384bda7a9132f287e3bedff63463dfc06f57","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/fragnet/rec_ok.npy","bytes":710829,"sha256":"efa48d27dbc1ee7b512b8b376076bd6a6c544c5b86013166e6a52c352b451c68","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/fragnet/sets.pkl","bytes":9343635,"sha256":"922fafcbb454509f1397de6da7bafb19d46edb43520f54405abdbc994c760551","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/fragnet/stageA.npz","bytes":268385,"sha256":"3a501ecbfae647d86d7fda87605eda2d573475201b2482ca02a130aa2a3d7a68","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/res/fragnet/stageB.txt","bytes":544829,"sha256":"6916c25c6f32fdea4a20f5c0b4b5cf45a3f6bbbfa625552195a7909e8ae8685b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/code/fe_v4/v1engine.py","bytes":8391,"sha256":"9074546910ba850a98bb95afe531669c62a26c15ab8b6bf9d6b6020c77ea0e8a","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/fe_A.pt","bytes":198790165,"sha256_head_tail":"9b19b196dc6c90e79880d9e37546d6514b6ac04ad6573fbd4128412a55490ec4","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/fe_B.pt","bytes":198790165,"sha256_head_tail":"77bd2958d1ccd793b36ea325fb7c11f8f7203b0043775bfd9c242dddec505864","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/fe_D.pt","bytes":486278727,"sha256_head_tail":"3682f9abdbd10199f5a8fd56d5b1a20ac8a277ce63c0a9fad08d7d938b0735f6","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/fpnet_0.pt","bytes":198790165,"sha256_head_tail":"9b19b196dc6c90e79880d9e37546d6514b6ac04ad6573fbd4128412a55490ec4","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/casmi26-v4b-models/ranker_v4b.pkl","bytes":14301548,"sha256":"6172e926fb2de244f0b9189ed54e1ed8a15e76d166592b724129a4d6a34931da","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/chebi-lipidmaps-casmi26/bio_fp.npy","bytes":54399176,"sha256_head_tail":"190d310ba180c5665258dcbdc392bf0908da9cbdd177b60c4539d97254235e97","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/chebi-lipidmaps-casmi26/bio_mass.npy","bytes":502080,"sha256":"7b67b125f7bed307c2ac08bf41ad92044bfc2177519c5d4bb36c2c51789a2921","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/chebi-lipidmaps-casmi26/bio_meta.pkl","bytes":5545993,"sha256":"b94aaa0c68d6a0d7bb36db267d271f679e4286b0cd2de31f982f14f0f1bcc051","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/coconut-casmi26-candidates/coco_fp.npy","bytes":378349391,"sha256_head_tail":"c04290155d84e249f625e4f7f03f1580b922e6ec4493e52da203575ef0892fc3","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/coconut-casmi26-candidates/coco_mass.npy","bytes":3491240,"sha256":"2b2af92c3185fa8cd52a4a3e3e7fd7eda4d94e8b9402662a3a4c55ffb13c6e5c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/coconut-casmi26-candidates/coco_meta.pkl","bytes":41450156,"sha256_head_tail":"474ea048de8e7bfdf8ad7a53d04d3236043edf7e37c0e4cde578e9b3aaf945c0","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/coconut-casmi26-candidates/fp_bits.npy","bytes":55568,"sha256":"a61e372fc41091bf5c7467e08134081266262665077cb9e2c56d73e4887f3013","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/hengck23-dreams-enveda-casmi26/Enveda180_DreaMS_embeddings.npy","bytes":4725903488,"sha256_head_tail":"b15b7cf54ba94ffc4f757ed93821548f8ca8290316c1aecb5167b7409cb06afb","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/hengck23-dreams-enveda-casmi26/Enveda180_DreaMS_index.parquet","bytes":26015327,"sha256":"b44a5fcded9de1d40c163b6ef541a2e9be4de07eff89b82741c6450e0c3b48cc","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/hengck23-dreams-enveda-casmi26/Enveda180_DreaMS_meta.pkl","bytes":256717058,"sha256_head_tail":"4b49427809108ca1a71143fd4556777c87fe19b9c161a9b90ea36951c097f25f","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/hengck23-dreams-enveda-casmi26/MassSpecGym_DreaMS_embeddings.npy","bytes":946602112,"sha256_head_tail":"404c5e222f90fddf71b4571906069bb3a04c6b27d6d06ed8c9927ce28d3ca552","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/hengck23-dreams-enveda-casmi26/MassSpecGym_DreaMS_meta.pkl","bytes":519242516,"sha256_head_tail":"dc9d79a3be06116766a76aeed11a2b90b8dd4f57827e712eb4b090345350570e","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/hengck23-dreams-enveda-casmi26/dreams_html_results_v2.py","bytes":13275,"sha256":"bd02a1a8aa411689f1869a4145674d07d7281de3638624cc2d397374998016d7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/hengck23-dreams-enveda-casmi26/dreams_minimal_model.py","bytes":21916,"sha256":"e7021a21e7b229cac99b4ec37dbfd85cf4044195d8245ff9c57f923eb2d3ff66","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/hengck23-dreams-enveda-casmi26/embedding_model_state_dict.pt","bytes":386418933,"sha256_head_tail":"9e5ee6265af9e57b19f534f7a621a76b4c01136470fa1f029e3defcd5a5349ee","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/hengck23-dreams-enveda-casmi26/run_demo.py","bytes":6159,"sha256":"27361d04f0b89ea2f98884862054fa79ca886d6bddcb1a921da8c87ce85b8d44","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/pubchem_tier__pc_mass.npy","bytes":847004040,"sha256_head_tail":"48a7e299fa2882048fc15ccf1382b0c9497822a2b111cbd34ab5001998b4a91d","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/pubchem_tier__pc_off.npy","bytes":847004048,"sha256_head_tail":"dd32178fa19c3b8002badff4fa6a38f6de35b008409f83de1da38157d1f6b7fc","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/pubchem_tier__pc_smiles.npy","bytes":5520120190,"sha256_head_tail":"52de5fd30a8206e93102cefd3b0bbcf76a312a2ac68d35b4bff18a5f494338dd","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/rdkit-2026-3-3-wheel/rdkit-2026.3.3-cp310-cp310-manylinux_2_28_x86_64.whl","bytes":37258660,"sha256_head_tail":"7f481dc143a2176358b4dff44d47b1fa8e94d5d7b99e747f33f7078d61bb405e","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/rdkit-2026-3-3-wheel/rdkit-2026.3.3-cp311-cp311-manylinux_2_28_x86_64.whl","bytes":37257559,"sha256_head_tail":"913375ee03dca177d0c63130c9cffa2eab62d01f23c9d8231beb0ae31cff5216","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/rdkit-2026-3-3-wheel/rdkit-2026.3.3-cp312-cp312-manylinux_2_28_x86_64.whl","bytes":37188323,"sha256_head_tail":"541080dfb208851bda9f9574a7b4c7260b63693e77b2cce6dddd9f96f9a406bd","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/rdkit-2026-3-3-wheel/rdkit-2026.3.3-cp313-cp313-manylinux_2_28_x86_64.whl","bytes":37187145,"sha256_head_tail":"8ff8e91a8e50c8b3c6e3219473b93fa5bd4c6dbe66e592846b60098cc6b4753c","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/rdkit-wheels/numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl","bytes":16708577,"sha256":"a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/rdkit-wheels/pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl","bytes":6937129,"sha256":"0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/rdkit-wheels/rdkit-2025.3.6-cp313-cp313-manylinux_2_28_x86_64.whl","bytes":36117945,"sha256_head_tail":"2b9eb3ff51d6bbe2f6afda2e104664010ff38cda0b3594b2dee03b7976e1df0a","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/FILE_DESCRIPTIONS.json","bytes":3117,"sha256":"b161e9754390f016f703015b3c136d6d55703ad59e666ded908c6a1e762999db","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/FILE_DESCRIPTIONS_FIVEFOLD.json","bytes":17997,"sha256":"2683b009883e15d0a97dd6bd84e3d635304df787111378604693b125ec652dff","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/FIVEFOLD_VALIDATION.json","bytes":11755,"sha256":"37b5794a1edf1508e2251a51f50e83f55763f98a2938f52f6a7292d4e3284220","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/MANIFEST_20261019.json","bytes":5578,"sha256":"24a522855d346635e70ba8e033ea5780ba46527c82c7f4f16f934431dcf3fd82","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/MANIFEST_FIVEFOLD.json","bytes":9013,"sha256":"1ee36e9b73172efd5f1cc90655949695718ffab5b258efe22f9d06ec8a13ff84","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/MODEL_INPUTS.json","bytes":910,"sha256":"672a9a9c9a00c191c84faba8a1c8e2555969644e09d70844851de04f7889b6ef","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/PANEL_REPLAY_PUBLIC.json","bytes":12718,"sha256":"90e8e64f0f5f4ef106aaf17ecd3c174bbc2650d1c557829a43701cbd4f0e65fd","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/README.md","bytes":6597,"sha256":"8767f5c201c499cbac9b20630c12ccfbb877fb9a1665c3338cfb0b3358eb320c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/SHA256SUMS_FIVEFOLD.txt","bytes":4890,"sha256":"73d1e104222c11d47e603ee18856bdf0ac5dd817b492cefdb73d35d74cd199bd","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/SOURCE_CHECK_20261019.md","bytes":1351,"sha256":"aace993777bd139fb0faa5ba5f536b761ed59379c0e918e6602d462f2a44401e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/SPLIT_AUDIT.json","bytes":1337,"sha256":"d4b994b663a7ddd944603b5a2f025c0a76a4d1648fa6deccf402b5c026137661","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/VALIDATION_20261019.json","bytes":2245,"sha256":"ba95c885ec552b84c3122f5f427abe3d4856c1cee0a4a5d23d56c0d21b4868e6","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/audit_public_split.py","bytes":3346,"sha256":"74b871df50a832d37b5ba5945d6a73197432a12a82ffb822c1aa1b0fd01a8ab8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/check_inputs_fold0.json","bytes":116402,"sha256":"fd86ff6d71318467d3b2e6b1acfba8c637832f9d439e00f119b3451be46d8fc9","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/check_inputs_fold1.json","bytes":113100,"sha256":"b8fbabb0bcd9ff5a49c55b4f6d29b1bae9d1df0fde2742bb933e574e3817dfd1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/check_inputs_fold2.json","bytes":114814,"sha256":"0ef10737eb1a7b36c1b11ad8e0e5608e93ec3f081c4d579c3467bf4b4f6e69a3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/check_inputs_fold3.json","bytes":114520,"sha256":"1e07b116cba8b03ebed56742237fc61776649fd5db4d02600a9bd9207e61bd7f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/check_inputs_fold4.json","bytes":113878,"sha256":"cc71eb650504be29a6601d8c906fa5d4123b71732671c7f6bb66ac4d4f52bbff","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/eval_ckpt.py","bytes":3791,"sha256":"3f4b99080516bd19688b6517ece4db4acc8052367eb4a628c81ef2569dc07005","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/eval_prior.py","bytes":6806,"sha256":"b904eb2c880a560e73c57f546da30250c39dfb8d39c4a5559d1393a005d04123","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fivefold_retrieval_fold0.json","bytes":552,"sha256":"f9248a03a92ebcecf8fe4ad215174020acad2b03b7ac9f110c160ae49b5272be","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fivefold_retrieval_fold1.json","bytes":523,"sha256":"047ff34c804ea074608bfe4b6527c31718498e25d26aec40eb5e44af27b3b824","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fivefold_retrieval_fold2.json","bytes":548,"sha256":"4b05a03cdf0d892a0f325baab2b17895ce74b05966fc3eb60d9e1baa3d195084","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fivefold_retrieval_fold3.json","bytes":534,"sha256":"0ddf7cc53e03e4e5218a9de3a4231eb8655eca30f0f324ff96abaf6f8a30fa62","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fivefold_retrieval_fold4.json","bytes":535,"sha256":"b7e1a907958d13cb9d9c76ca26c6606fbaccfa367b86295a35e2a8e40148ae84","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fold0_history.json","bytes":10041,"sha256":"45daac623930d53d66c80c294225fbd040b4cc4618e871e72384c1b180e79494","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fold1_history.json","bytes":9991,"sha256":"19aafead125fae7c28863d0eac0ac9689fd4768584c92f601197977fcc81d2fb","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fold2_history.json","bytes":10140,"sha256":"9af69bac4bdf45892af3406e74ebf20a79e4b665bb706c496c8009f64113e2f4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fold3_history.json","bytes":10013,"sha256":"7548f540580f1332ab929fce5547f050bd19154530212084d2b510a1c2643115","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fold4_history.json","bytes":9970,"sha256":"07fc55f12c752cd6fe3e144bf2c7383286f5cb58a0b9fc3fc82847582c044322","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fp_bits.npy","bytes":41032,"sha256":"c3869e86449d6fed6e79e9655f8a7d35bd55c3138ecddfc1dae5072627e78677","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fpnet_fold0.pt","bytes":198789484,"sha256_head_tail":"7311ff942756bc97ad8a5dce8856dc24b45c573fec21c4091777c164a87a8a4f","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fpnet_fold1.pt","bytes":198791276,"sha256_head_tail":"0cd7b8ea0cf8b9fb0299d321e273532e9f5d0ef14ed71a5a5f7f1c74bc963bf1","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fpnet_fold2.pt","bytes":198791276,"sha256_head_tail":"fb43de7ad4453281cd4668c407740c7df1034774e169c9ceba9168942e542edf","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fpnet_fold3.pt","bytes":198791276,"sha256_head_tail":"394dcd24b1c46f5495f5213239e1ecc990d8e6e0669e0bb71e13672bb5bb8578","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/fpnet_fold4.pt","bytes":198791276,"sha256_head_tail":"35b0dbe3e12e070e5640b7aebfe5e4ee7ffb7ad1fa4c5b9ab69ad10bca8cbf75","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/gate_diagnostics.csv","bytes":365797,"sha256":"07bf47a47b5119e9a6dc0e7e61654c9e4db366a71e5249002f5533ca47311836","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/identity_audit.json","bytes":41499,"sha256":"99a0350ec9b0e967e4d60154f3598c46d07f6b4ff3aa2257eeda1a78210d5dda","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/input_subgroup_diagnostics_fold0.csv","bytes":144245,"sha256":"5578b3800027071bdf52fca4c704cae34ff8a99efd78efb67bf4d2b19e957362","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/input_subgroup_diagnostics_fold1.csv","bytes":140202,"sha256":"461fbe5ed8c3e76a68d9ad87cb3eff29ebb04c331e97ac2cc6048191df09ffa2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/input_subgroup_diagnostics_fold2.csv","bytes":143247,"sha256":"b93bafec27259a6324649928472e00e33c25c23a31b24f66929977d86977fdf5","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/input_subgroup_diagnostics_fold3.csv","bytes":142695,"sha256":"1084ded0334f512b0a71dd29797443bb0e46fc7b18ae98e9f4780a0ac09dc10d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/input_subgroup_diagnostics_fold4.csv","bytes":141887,"sha256":"5a68a20d99ceaf548a9c0ad2f50ce038e1419fe18674ec6b99511ad62b175c25","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/make_split.py","bytes":1438,"sha256":"136bb35028db11b614e6ad544b044f35199d614d52d2349e2191cda6df77d4fd","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/panel.parquet","bytes":23269,"sha256":"93469d9bd7e51f9e0c2e891f59844a1ffc56c0921035110495b598c9b4e52eb3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/panel_fpnet_per_molecule.csv","bytes":270090,"sha256":"3f0fb337dd8014e07639973a56eaf2e8d4a86e83ae1c7c15663480eabe621ee6","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/panel_fpnet_scores.parquet","bytes":51937874,"sha256_head_tail":"18eb0d89c7309ebe93f4efc381b876b73b1c05c4759e88530701ee62ebdf715d","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/panel_logits.npz","bytes":20466740,"sha256":"383e4d9aaaecd8bda1277851064cce840887b733c48477ce5099b71b0152403d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/panel_metric_identities.parquet","bytes":926118,"sha256":"4b8f5d342dea2845bed6dbaa8c790fd84b59f446d1f5af6b0e94ef79e4c6aa68","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/results_eval_fold0.json","bytes":581,"sha256":"fb52500534bd316ea738b9be7f5f98a1028d677917159d9f283c10e4e834b3e7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/results_eval_public_A.json","bytes":577,"sha256":"fa9d77771af7a7c67cc3fb949478a581c0c459153774d9f56e94b0c7948374ca","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/results_prior_fold0.json","bytes":9194,"sha256":"d4abf95580a0eed8982cc5c0fcb3d222e212efc9c1cc5bb30d18347b8b73bdbf","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/split.parquet","bytes":2204455,"sha256":"dd62549f81b9540890e0da17a06d0dcd216a2781b2c31687bd333445fe57d45e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/summarize_panel_fpnet.py","bytes":2015,"sha256":"61e7a8ec23313ec0bc460ebbeb9d25adcf6261172609a5aeeff3ea42018007b1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/train_fivefold.py","bytes":12969,"sha256":"ad92253e5c4901237fb6c9952fbec8671d4da7f7d8ce8b83ca3ecd25f1f6178b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/train_fold0.log","bytes":53968,"sha256":"e124e29db6bd84061a1904b4f43b524a504575596cb62c89bee3509c0aec7819","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet/train_fpnet.py","bytes":9744,"sha256":"46b4dceebb2badeac78e0d5e55849169f5bef8da70a3adf1a7431fe34e603312","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/CHANGED_FILES.json","bytes":3803,"sha256":"f5ac6afddeb1b45c4a6783bb840de093a4b251b0182876d1bf7fbeb7690eaa34","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/DESCRIPTOR_METHOD.json","bytes":3694,"sha256":"46456e341e87785ae96d3d13aa835a864bcb67a72611a4454cfbda63706b59b1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/FILE_DESCRIPTIONS.json","bytes":1499,"sha256":"ed182aaf2f4c13a976e60adef555618285b4c7a8e035413e5e398454b7c8481e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/JOIN_NOTES.json","bytes":1397,"sha256":"ed2d4901e4ab1b0b3925181cd0aa0a298b7639bf1171194624c053d1bb8b85b8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/LOTUS_REFRESH.json","bytes":14743,"sha256":"61cfde26dc65fcc96cf54a493c06840b05f5d3765c76bc87bf9731c18e321bce","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/MANIFEST_20261019.json","bytes":3204,"sha256":"0af26ae26df15b2963b5b682c3cede7b3aeb030c3e20290deb0bd9f8f9c69f02","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/MANIFEST_20261019_2.json","bytes":1103,"sha256":"dedd5ba32d5eb80f7d9b06367fd965ea6054d522b510a6186cf2252e104905af","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/MANIFEST_20261020.json","bytes":1953,"sha256":"97b89178b630d7ce6449a1836c342c3963034140a154655bf1d14716857fb00f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/README.md","bytes":10938,"sha256":"d229d3c54fa029c92f9980f2a5f97da6b40b9e55603845d31c4c5d8256a6313a","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/SOURCE_CHECK_20261019.md","bytes":1351,"sha256":"aace993777bd139fb0faa5ba5f536b761ed59379c0e918e6602d462f2a44401e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/SUMMARY.json","bytes":486,"sha256":"3d7f23504a56bc802e9249bb4c317e92167adf1cef4aa583af30d3adad2eeb87","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/TECHNICAL_NOTES.md","bytes":36249,"sha256":"f34a3a37c8754118cd94f7c70f451786dfb58aaca5d53e2a2028bb14ce527e79","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/VALIDATION_20261019.json","bytes":2174,"sha256":"92f7ffe9f855ee601b13c82f93d568f9fa56f1fe188b1ac3f9edc87bc5e61b77","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/build_np.py","bytes":8965,"sha256":"56d9d7da8566d596c2a503bad4ab70878f9ef02f1b46e7f30a3cf2b733271ef7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/candidate_joins.parquet","bytes":4468200,"sha256":"ba079638bfa47f0af076a1ae61510129402fce696209c4a1fd05fd7cb5854985","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/descriptors.parquet","bytes":5679231,"sha256":"89ab934a63c0a5303ab1cfc1ec80459daac4b69a648835c92770aadb927375bd","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/external_reference_spectra/DOWNLOADS.csv","bytes":12279,"sha256":"972d0f677c7e33658951ae812348528a6672506e75d60f56a0dbbbdc6f483ac8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/external_reference_spectra/INTEGRATION.json","bytes":4032,"sha256":"8035b250d2bd1a3af576fb9e5abcbcd45a5a79f016f7d1cc9af1dfb026d05494","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/external_reference_spectra/LICENCES.md","bytes":4467,"sha256":"10fa8e1c4c11bbe823c05c9b2aefff9892fe99a1737e2bccf3d46cc3fb44fbfb","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/external_reference_spectra/SOURCE_MANIFESTS.json","bytes":27651,"sha256":"64005bddbe99a5219ffdb8b6c39079ef1352f719e02ddf754a10a928b78396b2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/external_reference_spectra/external_candidate_additions.parquet","bytes":13047,"sha256":"8e8a88866e11f94ecc5fe2df5f8a9cbb1bc144e6c45387438124b2946947e376","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/external_reference_spectra/external_delta.parquet","bytes":4607296,"sha256":"e75558832935f28e85a5419022c4cfba4d0d4b49f1063544c317c8ae61c2ac41","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/external_reference_spectra/external_record_quality.parquet","bytes":94211,"sha256":"4f5b227d1c874624182c76e8f8406ba3ee86521744831cb461d6198c898b6c9e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/external_reference_spectra/external_scorer_sid_links.parquet","bytes":36920,"sha256":"8cd61b8f110bfd6686684d330609a1d0f0d7cc96eab32b9cdd3352939ef443ef","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/external_reference_spectra/external_structure_coverage.parquet","bytes":43715,"sha256":"0a284c243ea4b42151040e6f9e1c9612a6a8f42422cf8835582c5891db08c2e2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/lotus_structure_genus.csv","bytes":15679755,"sha256":"22a5b0f0d4380781c1a6d67d605ce00b2a8c5d7dfa8214343191535de820da44","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/lotus_structure_genus.parquet","bytes":2513093,"sha256":"bbcb313b2001963e8be4485c8a3f2cb318f041b96124704bcb26d4a6004d1436","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/natural_products.csv","bytes":41057714,"sha256_head_tail":"86ee1e71465f2970af52c04a65d1ee1e5f9fd84f4570e023d7599454f6a123cf","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/natural_products.parquet","bytes":9575768,"sha256":"6ccb833e47d88715fca1d614b05cae92fa333fa388ae1d64226f2695a62081aa","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/ANNOTATION_METHOD.json","bytes":522,"sha256":"e54d9287855639c261bc66487aeefae118b67fdc58de45067e9305be93191836","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/FILE_DESCRIPTIONS.json","bytes":31479,"sha256":"cf86424461c536fcd2fbe8e0654f6c0fd84b90aaec91a26ce2af6a4e514f854d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/FRAGMENT_METHOD.json","bytes":1574,"sha256":"36496fa99211603505235a7e611f375bd2a216de2fca40c0e9debdadfb8b5411","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/GRAPH_METHOD.json","bytes":947,"sha256":"57bf9eb7e0e3ff72f8adbef3d17329ef89eafa5e0977a47b816a95b466606693","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/MEASUREMENT_SUMMARY.json","bytes":1616,"sha256":"0e2f7a66710abc690cd8067c3517c7906b76c092a2c36521688c6511b1f0283a","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/METHOD_CREDITS.json","bytes":1562,"sha256":"ec4bbbe40038d02667de9a82d6d8aa687234152a7a1c0339810403b815146824","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/PARITY_SENSITIVITY_METHOD.json","bytes":820,"sha256":"95294775e979bb751a68135f3c3ce971bfad3a3801b54f9d9fea69b92b5ff58b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/PROVENANCE.json","bytes":2137,"sha256":"8cace7fe456b5a4d4dff55613ff0301744b1e74ebdcfe507e0df842f6b1eff92","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/STRUCTURE_ENUMERATION.json","bytes":965,"sha256":"0c30c129d911aa80755ba040b4899f3a6cadcf4919afb5cc38884549a20a917b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/additional_measurements_v25.json","bytes":1523,"sha256":"89446ba37cb63d86ea9f0207a19b91a53889831b3127ea23795ad897326a9154","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/adduct_census.csv","bytes":5256,"sha256":"b9ac29b3554ea7bb68fe7cd412fe832a1876ab32d85716ec10eb505cae5c8850","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/annotation_by_energy.csv","bytes":2370,"sha256":"ee25d3328ef2de328bd23370d95742b98e4221d89998c83d1e074d38749a88be","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/annotation_summary.csv","bytes":1132,"sha256":"b8d6a077e0bbe94dcb9bd5221d39afe3bfbe349e37958ed3618bdd2e2f2a7406","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/bonds.csv","bytes":17201,"sha256":"7f6184f83778e11f6c69d01d5301ac9b122f477050dbadedf8449fb7cb1a0e39","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/collections.csv","bytes":299,"sha256":"92c8acbdcc6ab47f17785cb32654a12b6575c95b8594798ff615029b3551a018","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/electron_parity_by_energy.csv","bytes":2866,"sha256":"63ab859a29cd4e59cc8e5416837c89fcc9b5672cb7133370c1834a27e90654ff","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/electron_parity_sensitivity.csv","bytes":13901,"sha256":"471d523efdfdd73716e7a4e01b4485d48ece3ff55ea0dbf047d12401db0c468d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/element_comparison_v25.csv","bytes":1593,"sha256":"7d2c2d7e6196bff5c970d2b7b13b484c36699862bed6c922d6c2e7641c3d3089","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/element_count_distribution.csv","bytes":28774,"sha256":"ec33ee868c160ea18456d63c6383287dca49dcc739e69810982839fe43ad5561","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/element_groups.csv","bytes":2433,"sha256":"ad14011008ca3081e8f5dd7282458b5ded70509e4ef5fba706a1dcfff6f35516","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/element_table_v25.md","bytes":1047,"sha256":"70a831df8acbccaa0580c647b81e18f3d190a86287a73eb7a5a048f170fc1101","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/elements.csv","bytes":8120,"sha256":"65021292fea55c7cee77161e70812a5564947f3d3011be16fe2da2ac1fd2fcb4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/example_spectra_v25.json","bytes":12608,"sha256":"3148366acdc317420cb9d54d6a88572d2099d2a28081c81dcbf2e2892cae79c5","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/figures/elements.png","bytes":64734,"sha256":"c4499f1be55941583fbeb9b2222dacf326e5f20fb93ca1203d9f8174e0a8ee12","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/figures/example_stick_spectra_v25.png","bytes":283745,"sha256":"c98b0c6883573b6dcaa305924a6fdfb3ed8a9a17c1346f4bc749ce5bf521db9d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/figures/fragment_catalogue_coverage.png","bytes":120851,"sha256":"35b946c2a8fcdb6785f530214b42d082416e97df87167de060fb648fd5da2baa","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/figures/fragmentation_graph.png","bytes":168115,"sha256":"ed414d719bc166758d754f73e622d61d9aba46aeaf946cea312429870098d4f4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/figures/fragmentation_tree_v25.png","bytes":333597,"sha256":"9acaa87d6707f95637f68eb928429327672c07b6a5bb18ce1f22ca10e8560fc6","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/figures/neutral_losses.png","bytes":99357,"sha256":"96f208e0ebe89c8219d04ebea1314f0b01850119b2911a3b7efa9d3590c421a1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/formula_coverage.csv","bytes":2993,"sha256":"0f8d09ded49cd49f59fbc8a15eb4ded78f8a216d5201e397acdddf38c2a13f66","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/fragment_elements.csv","bytes":1546,"sha256":"61298f67ae8245a22edcd26c86c226fbaebb07adef6c7c0137adfc63e774edee","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/fragment_formula_catalogue.csv","bytes":726009,"sha256":"24c31fd8385965b42c6e888bd612547a968dc81860003caf36152fe2796c8cf0","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/fragment_ions_by_energy.csv","bytes":1741333,"sha256":"f800d13131c55a8db81226be9c7a8323e992130dc93d39659f3f80d20ac25bd4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/fragment_isotope_checks.csv","bytes":94488,"sha256":"959a909f5944a497c0bd6886e835bfe4d8da597685714d5b0c959147e1bebfd2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/fragment_isotope_summary.csv","bytes":287,"sha256":"e1886fa297820676613ee1ce17b66fd38c9131682faa4695ee179d73cfca5d66","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/fragmentation_graph.json","bytes":163448,"sha256":"52bb5641110ee9e35d539117491c823a1b806a76f9497c6e4ffcb9929f8bdfce","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/frequent_fragment_ions.csv","bytes":2067769,"sha256":"721cfdee3beda9dc873d8def8ab6ecd0d1743aef67a96183689befa8be3f0668","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/frequent_precursor_losses.csv","bytes":2609,"sha256":"21dcea8fc99930a10aa1f65029b9a8f3bd182940433400e56a07a0e7a5154c27","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/graph_candidates.json","bytes":1927,"sha256":"367bbbf06c6f7413d35608564fc759a16f5aa7c3828b65648d1c8c52abc591fa","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/graph_energy_behavior.csv","bytes":704,"sha256":"f43799e6fcb9932f6e847cba4a7294b470e7f20e4833c2bf6473f94c2692c6af","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/graph_energy_sampling.json","bytes":267,"sha256":"1882e65c3ae0bb4da8d645ab3ce0edad50be5b88a866051d80a172f9c2e7ed7b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/loss_events.csv","bytes":276332,"sha256":"31d213eb4ceb5a3ba232f1f898392961131dd02a403f3a6cd706f207b49232ea","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/mass_defect.csv","bytes":734,"sha256":"269f2cd0690a36599a12c54a91d77855fe96cafb0dd94b52909ccfe2276dd162","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/neutral_loss_table_v25.csv","bytes":1552,"sha256":"4584169d14b734c746c8feffe4204ce2a42d28911509a19c810a6172ef4dc3fe","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/neutral_loss_table_v25.md","bytes":1664,"sha256":"a0625939ff0b0c9f7aa9dde799c18ed373f476a41585672a085b800cd5197bc7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/neutral_losses.csv","bytes":7979,"sha256":"c77ef06f3e54deb0e1d9797610c1a3b14fbfe4676ea729aebe03713d65078d21","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/neutral_losses_by_energy.csv","bytes":37894,"sha256":"b7b1f57595d0de6c13de97ec7e4caec96cb7d97fece11a0d33d7094b8197ae49","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/parent_tier_coverage.json","bytes":222,"sha256":"b15d6b4b8653615490dc937e867dd3e7f2b4a6e9e5eecea72bee915f35159ea4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/physical_rules.csv","bytes":8482,"sha256":"d37ad50bb60add9d0d7375b0605c12a23e991ea609a04364945d5e7434a57091","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/physical_rules.json","bytes":11565,"sha256":"6ccc62e3cb47ea7435f4dbccbd5c90f39aed4f587822480886f2a86baa692ea2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/rule_method.json","bytes":1017,"sha256":"20e9534a0ca7f13cc3510e1b2a324a00cf785312bd7f327c220691b5919ecf99","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/rule_retention.csv","bytes":1106,"sha256":"9295300e9b59f0f9fcebbbae893ff42edd8e8e0c6c75a579f4e67c52b1b2559e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/rule_true_decoy_retention.csv","bytes":1635,"sha256":"c1f62808af2620094aa40ae47d069ea5fdd8f0e42ceb58ea91d6cdc334b4a087","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/sampling.json","bytes":3605,"sha256":"0fc132d4435f0418396834da5aadaf7a50923af5db1b18a47684991acd3f3689","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/spectrum_summary.csv","bytes":695856,"sha256":"ae61b574a39ef6b5fec824400a4865f7ab272f7b321f68ee1130f62459187c1b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/structural_fragment_matches.csv","bytes":724588,"sha256":"de0c0947358897bd3135323722c0d1932babdde17167ac0595985f49093c13e1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/structural_parent_summary.csv","bytes":36152,"sha256":"fdac0453c9f7e477357506ab8bbe1e1adae4d5fa7bae5ccd659eebebe00715d7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/structure_coverage.csv","bytes":1513,"sha256":"4605097212a45fff933ea3ae6a12c59adc0d3eff585affd5c1c3745e059aeb01","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/structure_properties.csv","bytes":3339,"sha256":"9273676235b07884d783bf66439831a461fd27d602d683e7cd0d895d620da661","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/tier_census.json","bytes":428,"sha256":"78edb44b1931e4980add31024ff5278287b01aa5a8c98221f853af1519a088dc","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/tier_element_count_distribution.csv","bytes":15789,"sha256":"de05378ec050279aa9e2147ec23c4a2675f386e961125fabdcb95509edfd89d9","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/tier_element_groups.csv","bytes":499,"sha256":"d397e44a816abc4da25e0c4cc0e491a10eba08b007e85969c15a4d13724c7f82","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/tier_elements.csv","bytes":1624,"sha256":"ca42830e481fd04bf1a693eca6473489de21148e9603fadb64d308b800bee717","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/tier_structure_properties.csv","bytes":213,"sha256":"c44bee463ae6b8ac0d25f62c920f2c0d5dd11e935eaeb21ff3ab9cc49c0ba7e5","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/true_decoy_rows.csv","bytes":277969,"sha256":"0ff37c8799b40aa30a22e26ae9673432fc4e5b55af83bc3fdca02c1c4e18b7b9","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/true_decoy_summary.csv","bytes":842,"sha256":"df59815e81e3d811e2024116ec45049cb051a8af77d386281fc85457fff3743e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/spectra_chemistry/unusual_training_structures.csv","bytes":14953,"sha256":"8c3d0320604cf512dcc2a9edc00224ff1d02436bd917c338b210b44f221bc2f3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/sugar_and_acyl_units/MANIFEST.json","bytes":1281,"sha256":"e084f80c63e6cf220ff13ef426dccc4908b66548cedfe352c6e9d39c1ce38d3d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/sugar_and_acyl_units/README.md","bytes":3906,"sha256":"dc2884434699db41f91d04321e2208a080729208cf32ae3a4b3dca5deeac930d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/sugar_and_acyl_units/knowledge_units.parquet","bytes":2309306,"sha256":"60966bec41f36213f126cc8fbe85e3653fbf88d1abb24310c320603132f6ddec","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/sugar_and_acyl_units/pool_units.parquet","bytes":9219758,"sha256":"20726668d69ad5ca93d3159701210df4f097a1f84015e8044a67843f1c6c3567","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/sugar_and_acyl_units/selected_rules.json","bytes":11846,"sha256":"145046d119d53fd17ac9b0267ed3e024283f4ee18160a09fce0d62cb664091f0","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/sugar_and_acyl_units/spectrum_validation.json","bytes":1347,"sha256":"6d6d1992948daf6e4fdccd694a50ec3d6ef131a85ed20da8fde35c0bda922b9f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/sugar_and_acyl_units/structure_validation.json","bytes":18217,"sha256":"e00c8bdb7658c62f96af8817264a3997739a26f45d880648736ecdbdd5061fc8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/sugar_and_acyl_units/sugarloss_units.py","bytes":5270,"sha256":"c0d3b30949e7d3b9641043dcc654a0d3a3b9780aec28252d1dc69f574b235102","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/sugar_and_acyl_units/training_observed_losses.parquet","bytes":32381810,"sha256":"0da9ed8478bfd4d44defbeea2c1708505f332b9a260ad7c65d4fcfcfe63d4205","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table/sugar_and_acyl_units/unit_definitions.json","bytes":1871,"sha256":"53b2940e73019e9d866ea15bc2282cc8453333ee826bded03034eae11166cc66","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/ATTRIBUTION.md","bytes":1150,"sha256":"3da83ddc2bd1ca652e9ad613ab9c900193e944e30d63aeab4db1af46ccbee91f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/CHANGED_FILES.json","bytes":4528,"sha256":"daa1d1952acee83d76195bcf9d89aea98952576a69ef9e87d53fb06988a8dcc5","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/DIFFICULTY_METHOD.json","bytes":1848,"sha256":"25e531e821b9721366640146add4d020a5c5d6c8afab5081d21b5b491b3e4737","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/FILE_DESCRIPTIONS.json","bytes":5532,"sha256":"a3e854d9af217f8dbe1e27be940adfd4765bd42240df5268ac055c9ff29f2141","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/LOTUS_REFRESH.json","bytes":16926,"sha256":"e601a2a46d15a5a3be3059a34d740a31a7c9872cf21b7c2d8bc757da9faa4fbf","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/MANIFEST_20261019.json","bytes":11268,"sha256":"ea45f9708875330def650404be1da5b89cf85e41b19d4b82609e84635e365503","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/MANIFEST_20261019_2.json","bytes":2355,"sha256":"6405d2ba96651feafdfae76ea03447e8f5f46d15f404c368e8360bbb01c2cf2e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/PANEL_REPLAY_PUBLIC.json","bytes":13121,"sha256":"0f85f66547c22fa66d521e9b5d98ef330b818300c82248096b804ce6f0298525","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/README.md","bytes":8808,"sha256":"079418d44abee11eb9b71391cc9541934f84cbd0cadd624baf12d0eb711259ce","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/SOURCE_CHECK_20261019.md","bytes":1779,"sha256":"3ed13ceef9616c1cd9b0e6b57667de0345431cfc9717dcbd5db26277eb4fcc7f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/SUMMARY.json","bytes":188,"sha256":"d6db82749b9e71a00066313e756d388ffbc1a61fafeeb766449cabc37715f63b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/TECHNICAL_NOTES.md","bytes":51233,"sha256":"3ac43aa88b6f11a7dbd8a2209c84fbce158d8ee8c63776b043c9f2de7dc50fcb","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/VALIDATION_20261019.json","bytes":2791,"sha256":"305dc2fb04d445af65e7f0ea6ba8c4f52de7bf6b9a17de28c117c22af05350bb","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/VALIDATION_20261020.json","bytes":11387,"sha256":"62310f65b7ce0cdd56dd90a3affd8a3aabc300fa626d35ce43b2e914538f9ff2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/aligned_public_priors.py","bytes":1178,"sha256":"89873c7312c16be2180d44bb9eb3c4e9ce19c821e74ab92bb41666ea0c0f9248","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/MANIFEST_20261019_2.json","bytes":1262,"sha256":"12de2fa1e693326088edec3f3fad91fee2a7072167dc05707339bd7de0e4dc8a","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/README.md","bytes":5396,"sha256":"d551ed2066812b1c00188f06a44148dd5fdcbe46cd0420b3264a49044ad6dee3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/SPLIT_METHOD.json","bytes":1415,"sha256":"058952c7968cde06ae3906bbd46778a2bbce53e2533456eca6c41a6455591f08","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/SUMMARY.json","bytes":1299,"sha256":"d01e89f166e9daef483262f9f05d74d3b94138f6842437ac72c3e08933e2b057","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/TECHNICAL_NOTES.md","bytes":16681,"sha256":"2c7483a9cc2089113897d3ff25f41bd90783f5198be6cc890b6dea5065da93cc","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/analogue_domain_summary.parquet","bytes":7732,"sha256":"c02d3b84b030217ad14854465eaac60f887a28cb21998af4f4f62fc204915952","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/build_folds.py","bytes":8031,"sha256":"b837474eccf3922c8517c00bce0166edd34e53115e43298696381b52e05e38e4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/spectra_folds.parquet","bytes":23678772,"sha256":"68446abc365cd0a175c8fcf8081828989e92a65d3493312848792d3465d91403","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/spectra_splits.parquet","bytes":9272714,"sha256":"518be062cfa33077cc2b145fe03a61c62be3ccbd0a2675edbd07d1ac78775235","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/split_categories.csv","bytes":4223,"sha256":"16d801c4aa3ff7d3a217df37bfebfcb29334aee57310dfa14f471351f1bbbaf0","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/split_summary.parquet","bytes":10206,"sha256":"f03e9aa6c8cd34e83273b568ebaa84b9b48dcbeab2a5c7a35ca20c927e164090","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/structure_splits.parquet","bytes":6302907,"sha256":"05498719c7564cedd673bfd954191452dbc211dd0b31a9825357fa191325e92f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/casmi26-identity-disjoint-validation-folds/structures.parquet","bytes":23795229,"sha256":"7c16c4968a15956e94eba58e3ff94d4a6c2f6899b10a2f2824047601a46bd6c8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/fpnet_window_eval.py","bytes":3791,"sha256":"3f4b99080516bd19688b6517ece4db4acc8052367eb4a628c81ef2569dc07005","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/gate_diagnostics.csv","bytes":364864,"sha256":"77fba33d3dcd91c657784d50d516ce959d6d2a848bf3ad45b6facdcf4f285f31","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/gate_eval.py","bytes":4731,"sha256":"6981fcc607f3a37f312c7444f318cde3d9c24fd0dc1d44ede4f2646d388bb44f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/identity_audit.json","bytes":41499,"sha256":"99a0350ec9b0e967e4d60154f3598c46d07f6b4ff3aa2257eeda1a78210d5dda","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/lotus_replay_identity_additions.parquet","bytes":4204,"sha256":"0e07c9cf9d96ff865571b7083a2507eb785321c90c3672f86a530934a755d82f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/main_candidates.parquet","bytes":1639248,"sha256":"63854375ebd0f3242e31e31069363b497f1fd80a2bfcb3222ee69a848e74ae67","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/main_eval.py","bytes":8183,"sha256":"c964f41bd119714c42457e55c1413e0d047bb74f6d1886a6ac128d05eae5f833","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/main_tables.py","bytes":6758,"sha256":"5fe5a58df457499b4a858e5a4b8e286af98b61e0ef7b3c4efe139bfeef6b0423","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/merge_eval.py","bytes":6323,"sha256":"84987a423f675b8e5bfa6fc81dfc01cb17a3b76145f5f8c7581bf4f4851979c6","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/np_eval.py","bytes":8643,"sha256":"0e6ef7e99425371ad2d0dbed7fac2678070332d6082efd9eba413a3ccb6232a2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/np_poolfz.py","bytes":3554,"sha256":"8507129d86e2836921d25bf899991340d2441b331c75d39fb519ce7b0daad3fd","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/np_tables.py","bytes":8155,"sha256":"ef73cac738ebd5dce839b16144e663edd5e27967e810d9d84c63585e26a9e736","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/panel.parquet","bytes":23269,"sha256":"93469d9bd7e51f9e0c2e891f59844a1ffc56c0921035110495b598c9b4e52eb3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/panel_difficulty.parquet","bytes":75661,"sha256":"2e29797046cddc22245fc6e46778c98fb1032137eb4009bf86df6a7371bec3ec","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/panel_fpnet_per_molecule.csv","bytes":269574,"sha256":"9686d66b4f26612f1eefe186d2959309ee1f3a6e43d865fe32fd2cfc8c9242fe","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/panel_fpnet_scores.parquet","bytes":51937874,"sha256_head_tail":"18eb0d89c7309ebe93f4efc381b876b73b1c05c4759e88530701ee62ebdf715d","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/panel_logits.npz","bytes":20466740,"sha256":"383e4d9aaaecd8bda1277851064cce840887b733c48477ce5099b71b0152403d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/panel_metric_identities.parquet","bytes":926118,"sha256":"4b8f5d342dea2845bed6dbaa8c790fd84b59f446d1f5af6b0e94ef79e4c6aa68","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/panel_same_formula.parquet","bytes":8342392,"sha256":"b7442441146f227f2db7e4a5dee458cd173d8304641080b69c87dbc0a5c3d7c7","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/panel_tier_formulas.parquet","bytes":8209234,"sha256":"29069629e529a9aa44d07ccfcfcd745eb27d26526d8191dff2136cee9a2be703","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/panel_truth_formula.parquet","bytes":7962,"sha256":"814f7acd3dd0b13ca52d0f9c88959c618091779ccd2296c075385903cd33ac7c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/prior_replay.json","bytes":1685,"sha256":"445324f6111fd50dffd3b4f17e632946a31b5936d4a55bb4cfbf27a22b0dda20","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/prior_rule_per_query.parquet","bytes":9192,"sha256":"fbbb18496a2e7874a03faa28ecec33b8ef8ab3bf5767f787dbcba6cd784ad7e4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/prior_rule_summary.csv","bytes":385,"sha256":"0cda766ef21a825bc85eacece06f6d01a0fc1d9e9f3fa4abe053d2090e31ac1f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/pubchem_candidates.parquet","bytes":51122745,"sha256_head_tail":"772223a4ee8f97dc925ee93ce59e563793216cbfe1354035528c710c6c11ac15","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/pubchem_signals.parquet","bytes":12726869,"sha256":"0cdc19e8daa55d22d89f97760f7ea3d5b1814d5d63be9f66844050d4f6d1627e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/pubchemlite_annotations.parquet","bytes":13570437,"sha256":"7a054d9b5933f7bda90965f9671dac234c2bf2420435cd39c2e69177b6a0f15d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/replay_public_priors.py","bytes":4283,"sha256":"a6d1667f55b3778a5b89d2e1cc5b381b34637d783be01a79f00e6fdaa527f30c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/results_fingerprint_seen_unseen.json","bytes":1989,"sha256":"47a8c1f664417606df7ed791a45e78dc99c715535201ff6fc540c02bdd24cfd3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/results_main_channel.json","bytes":10781,"sha256":"22900fd5fa4b9307f3de66dd7ccf921da76d5882a2ea332778e131013678a53c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/results_merge_rules.json","bytes":3494,"sha256":"49f7e2db24de942d43e5f19141539667e1ae00396c7420dab715db70ffea0af9","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/results_pubchem_channel.json","bytes":6358,"sha256":"f621c777c5f7547fddb923b7e8ea266283454241d46d165e06a4cd432dc89dc1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark/summarize_panel_fpnet.py","bytes":2015,"sha256":"61e7a8ec23313ec0bc460ebbeb9d25adcf6261172609a5aeeff3ea42018007b1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/ATTRIBUTION.md","bytes":1077,"sha256":"c9393e830e6fd5e586b3e93522d12c25f1f02d44f110be9004a47536f6883cdb","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/BENCHMARK_EXTRAS.csv","bytes":3291,"sha256":"79298c9d85a8e5c7f08e46b604da88804543fae80cf09d2dc4d00e1a503de142","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/BENCHMARK_EXTRAS_METHOD.json","bytes":3140,"sha256":"a402896eb9ca16e6734c976bbc8141a649cd8a627eb55fa0ffe301c98664391d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/BENCHMARK_METHOD.json","bytes":2667,"sha256":"efa8b5a341327e07639b14a2ecc049769d0f910a37b746f0a32601a32b86aa10","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/BENCHMARK_SOURCES.csv","bytes":14461,"sha256":"1c500b0e6f92786db8516b6dccc4d6f521051cb1c4f7e247b48ab17e5c540bc3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/CHANGELOG.md","bytes":1059,"sha256":"75eb1c80ac319dcae5491468fb93f9539de762692a1f0ad1307494a75d0d097b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/COVERAGE_SOURCES.json","bytes":1069,"sha256":"60d0792a09465695aefa6ec2d2c7bbffc7dba8a15a49c6fc50bdcfdf488737a2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/FILE_DESCRIPTIONS.json","bytes":25770,"sha256":"6a5f1a99556752ce9107ef6c3e4209a56555683c4cb5bed2bd9b4b66226dde29","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/FORMULA_COUNTS.json","bytes":1216,"sha256":"b8ed23c3e71a066f0f570759b26b3d18c26ee06fe2f3341de1b930e892397e5d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/FORMULA_JOINS.json","bytes":1401,"sha256":"c8817c52179d876fd0a3a70c352370ad8516593a513aef0f5a511d1c0a97c15b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/GROWTH_INPUTS.json","bytes":8258,"sha256":"64c88e0c6923b2b6f4ef742aae8cfc7c4d4fd1b3f2d6504687877f71ef207fbe","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/GROWTH_MANIFEST.json","bytes":6352,"sha256":"cae173a5ab7e21c4d86a57579ce1bd106f96502a11fbbac8cdf1205a07e5e73c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/GROWTH_README.md","bytes":1750,"sha256":"e22267d0c8c0368ef75d6b0e806f4fbb0c9786ac369b4212f83f0fbdb3bed1d1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/INDEX.json","bytes":276375,"sha256":"6b1723229c5a5d6247a56bb29d68748814966a3adb8d113c1fc1f67a85cde78d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/JOIN_NOTES.json","bytes":1397,"sha256":"ed2d4901e4ab1b0b3925181cd0aa0a298b7639bf1171194624c053d1bb8b85b8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/LOTUS_REFRESH.json","bytes":14743,"sha256":"61cfde26dc65fcc96cf54a493c06840b05f5d3765c76bc87bf9731c18e321bce","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/MANIFEST.json","bytes":894,"sha256":"b4d3aa114e7ba04c618bb022d7943f0d43f0333ecd578dacbe15b8a666b3d67f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/MANIFEST_ANNOTATIONS.json","bytes":1789,"sha256":"f3d175e500794dad35c7dc0bf81130765528a2814cd5704d0364ff563e849743","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/MANIFEST_DELTA.json","bytes":1581,"sha256":"952f5499372d2c50f463e2dd04f5622c5611ec61006cd23079a62cb016740376","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/MANIFEST_FP_V1.json","bytes":18713,"sha256":"4fc9371aad09032f707508b6fe4d340b4c1c73dec6d85ff447ec31f2677160de","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/MANIFEST_LOTUS2026.json","bytes":3596,"sha256":"e650ee74778b7ed8b226fd635ca4f89997df43d1cccce8033b158253808fa492","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/MANIFEST_SOURCES.json","bytes":9614,"sha256":"b9fea0bd2dbfb9ace49416e41fc5f0afb4ed3aec86fe0666076a5a48cf7b2913","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/PUBCHEM_EXTRAS.json","bytes":3370,"sha256":"dc62556bfabdafb31ada943aec1b24d00be4e1a60c4c948880bd71d1539c8b25","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/README.md","bytes":14312,"sha256":"cca1041f3264296339dadf4b897094aa70f2e4ee2f77003e295c460b715dd6f9","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/REPORT.json","bytes":1268,"sha256":"7f8e576021c16a6ce8e9325b54c2afcdf244a31f8c165872f22df8ae6c5099ab","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/SOURCE_CATALOGUE.csv","bytes":108529,"sha256":"5b38eef46b4cb02318f01e8a4d4674d0133149a847e29698b13a7df10cc77c86","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/SOURCE_GROUPS.csv","bytes":87779,"sha256":"db2550bdda3d4307bc527402614ad8202bc3650bd9fa980b4eb50c8bc2fe5adf","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/SOURCE_GROUPS_SUMMARY.json","bytes":1398,"sha256":"9d7bba45d1f8f67dcecd9bc1c37ea1e972a451703d7100dedf7f9abbd626f4fd","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/SOURCE_GROUP_STATUS.csv","bytes":41842,"sha256":"075a1493270f8bdaa56cebdddd411f1b6b084403149b580e8f68af9ecbd8c583","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/SOURCE_INVENTORY.csv","bytes":48280,"sha256":"5e647a8d20bc93c59f92928dedba266449db7f026650f10f78776dee52be4bb1","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/TECHNICAL_NOTES.md","bytes":39742,"sha256":"3fe1e96067cb55b95fba4e3bdcd1f8544fc23e6142253e40ca9711a84bf21585","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/VALIDATION.json","bytes":45208,"sha256":"1b84ab06666e87752cbc860ff3b616c93492fc7ee3c0f0579626f0f0220002d5","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/VALIDATION_SOURCES.json","bytes":830,"sha256":"5d48334934ddb711ac83e5f892e7549fefb4a3d8eb4a439f7fcb622e42550471","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/VISIBLE_WINDOW_COVERAGE.json","bytes":2189,"sha256":"434116196a8fbb073902ccb73c82709874a73dc33da517d5143fa97a4b2c106c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/aligned_public_priors.py","bytes":1178,"sha256":"89873c7312c16be2180d44bb9eb3c4e9ce19c821e74ab92bb41666ea0c0f9248","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/casmi26-pubchem-tier/pc_mass.npy","bytes":847004040,"sha256_head_tail":"48a7e299fa2882048fc15ccf1382b0c9497822a2b111cbd34ab5001998b4a91d","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/casmi26-pubchem-tier/pc_off.npy","bytes":847004048,"sha256_head_tail":"dd32178fa19c3b8002badff4fa6a38f6de35b008409f83de1da38157d1f6b7fc","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/casmi26-pubchem-tier/pc_smiles.npy","bytes":5520120190,"sha256_head_tail":"52de5fd30a8206e93102cefd3b0bbcf76a312a2ac68d35b4bff18a5f494338dd","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_meta.json","bytes":355,"sha256":"f22e43772e65656f67e1575e4269f0ad309e22ecb359bc81147445cfe50c4758","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_cid.npy","bytes":145340,"sha256":"6c60a856461bdb6ad2a28c66baf5b67c34b804146cf8ca8bed0eb55ffddeb398","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_ik14.npy","bytes":508370,"sha256":"4f74a9d10d3690c0f5dd18d5f8937fb74de2b6f1fede780ca17f2db6bb9ba87a","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_lotus_refs.npy","bytes":145340,"sha256":"ab0ea32e7ece083440d0186af54175c539f7a20687ea593a6771084e2a0b7be5","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_lpmid.npy","bytes":72734,"sha256":"7089da71194227bd405004728db2462b26961be156e4838c580508cef2370c6b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_lsid.npy","bytes":72734,"sha256":"673a198394971fc1b78fecf77b6f9a89000719ac0b76c1aa7ffdeb81486f2a29","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_lsynonyms.npy","bytes":72734,"sha256":"12d1ac3c4d320f57e6dd0e83aae8aeb598bc79c42186e8b07f55185b84824838","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_mass.npy","bytes":290552,"sha256":"ef82c84ce4ca3d9a00a2a5bbe265bda379b5e8525f8ec92bfd45cea408d7c011","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_np.npy","bytes":36431,"sha256":"30b3e56727e9e8f08ab5928066ae9974edefa724ccb6e4b6ceb2f63c4c23c31c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_off.npy","bytes":290560,"sha256":"edb3c5565dab80f7c7cddb7d6070f8f79f8fbfb758f9becb0c98c27cea85f5fb","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_smiles.npy","bytes":2320603,"sha256":"604c68274fb313bd032fd16315b3c928cd619f29137bd4cb67bfc6c38fae4d75","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_src_mask.npy","bytes":72734,"sha256":"52ceaac2cfca47983d3b8328a30220e264ed5cfff3140fb04a7ecb55bdea551e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pc_src_observed.npy","bytes":36431,"sha256":"e734af8c8cf8e20fb2d0c5460220109c69a23f45b3a2a7ff2bb8922623569455","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/delta_pubchem_tier.csv","bytes":4058259,"sha256":"58a8be91430b423f609ccf1dfd53956ccbfbb621561b4ea0b76a9b242d384d35","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprint_index.py","bytes":11360,"sha256":"ab055cc7ea46891e5961684c987306e752d814d29e8737da00fadd3669bd35e5","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-000.bin","bytes":6042670344,"sha256_head_tail":"02c47ee4dc3a659a143e18902ec4570947dce18549898294154281b71684e5ce","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-001.bin","bytes":5795714624,"sha256_head_tail":"0c1b3effefe0becc35be0736fa8032988bf94c9ef1d77865ea0ce2d4ff030bc9","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-002.bin","bytes":5831598080,"sha256_head_tail":"c2179dab3bbb184ce9484e9ebf77bf27267cc3e49ac0872bffdc78708a8c1a6e","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-003.bin","bytes":5739473664,"sha256_head_tail":"86231bee31fe37ea6e5e9b12dfd8cf81ea6dbbe1c429ddd5779f9235cad00780","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-004.bin","bytes":5995196160,"sha256_head_tail":"bcc7e2293c23967805c7f04122f4d57941e63dc6f487d91afd1924fbc5d68099","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-005.bin","bytes":5885642688,"sha256_head_tail":"bd891a8763d4d373bdef5616bf6099c617f8b1c6aaaeefb9ae6f8ab03565d4ca","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-006.bin","bytes":5737201216,"sha256_head_tail":"59da8ef2872211c23bfbe95cf96ccfaacd5f01da9f55793d1552c9194c4a1817","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-007.bin","bytes":5928809600,"sha256_head_tail":"71166235b548a5aeca0f291ae95a3152bd4336a4a8e768d401857f2bc39a1a39","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-008.bin","bytes":5744282944,"sha256_head_tail":"7b7f433b7eba970c3bd187b7ddf2b3630b8a449ca2cc20d7584c91a4aa7a77f8","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-009.bin","bytes":5934977152,"sha256_head_tail":"4a7ecce07372f44a03a45a75830afeb72996fc5aeb91202085388a5149c93643","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-010.bin","bytes":5714521408,"sha256_head_tail":"2707438eea8d2573b9e8ac908dd31243f1a35fa8f12b2f004fd0b74581615453","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/fingerprints-011.bin","bytes":4491199265,"sha256_head_tail":"a8800569c4f7f22fe29f48fa6c21b7f472db86abb9fd8145367674ea2a8733eb","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/formula_failures.parquet","bytes":149570,"sha256":"bf66fcfe964e97e090079cb36938f7f4886ae6871b111975eddcaecda2c1cedc","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_cid.npy","bytes":44836,"sha256":"dd182e3b29c435b4a8b6f8db9af271a26bd895d04dbe2f63c60e9c7a403eb4cf","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_ik14.npy","bytes":156606,"sha256":"4b5cc955bd472bf4b6f79111382f08da2b2e13d0955fd9f7ae7861df636f6867","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_lotus_refs.npy","bytes":44836,"sha256":"c36054e30c5762a76287242dd6905fe673f8998be356c2c10b4dfd2525d65d07","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_lpmid.npy","bytes":22482,"sha256":"6087f448d23c830d71b507f0d37d2fa03e8a7edb439fc4f11cac2c6dc296272f","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_lsid.npy","bytes":22482,"sha256":"b61869798298ec8bc909263a58f5025830f599ec8488d16ef4f94b5625a40e6b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_lsynonyms.npy","bytes":22482,"sha256":"82104705a06d5296d8fb14a6daa786e47fe193359b0fc881d7f32d439f0105da","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_mass.npy","bytes":89544,"sha256":"ca46c1c72932c1cc6b58ab9bcb833a2125281f20c0b623af9fa30b182561ef1b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_np.npy","bytes":11305,"sha256":"0247d671abeee3d605859f4f95b6d9876fca22539136039ece9c7f73c0ba6464","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_off.npy","bytes":89552,"sha256":"07ba3dc94264691e8a74912aa85c9e64415a4511d41e58055de36fdda34cce5e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_smiles.npy","bytes":559296,"sha256":"578590187454a7b01d3af9f994ab7cd148041a8fcab913bffa44a99909167ab4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_src_mask.npy","bytes":22482,"sha256":"9723906a530f498a678018ae5ea4c616690aaf6f435f2bbd9f588594b3676ea8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pc_src_observed.npy","bytes":11305,"sha256":"befe16f61edd8b2d9bb0947026c01b52f01763b6641b0ba402d13dd44a0ca3f5","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/growth_pubchem_tier.csv","bytes":2040254,"sha256":"c1f25947913aa809e63888ba66b635e02a42082a5beba8cf46ec99cd1de162f3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_cid.npy","bytes":423502084,"sha256_head_tail":"2acb98e2f2c9222c7cee18c24a2b7ae5e8601a2bc24d274f5bc34af7188178a0","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_different_parent_rows.npy","bytes":1037228,"sha256":"0189e73949696cd807135e52ebc9007577dcdac3afe338679ba7813b5ccd657d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_formula_isomers.npy","bytes":211751106,"sha256_head_tail":"028155f89990dfbe1a4c0f3a6aff7c48cab78d2dc267344c6208006aff20f606","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_ik14.npy","bytes":1482256974,"sha256_head_tail":"a3da60185163c878f6c1bb2ff1ebea1029cd6908cde61a458af02760d04f3989","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_lmesh.npy","bytes":211751106,"sha256_head_tail":"e97a28078e4cbf35e3bf12f3e00662b999cf26cb909e9fa60d5ff1ac4b8598b8","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_lotus_refs.npy","bytes":211751106,"sha256_head_tail":"318825086434f071fe619f217278bbc5317a17c30beecbd441bf0c7df0ca3720","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_lpatent.npy","bytes":211751106,"sha256_head_tail":"6fcc1f40f443ab16d144c6ce44fc818ec98b2bfe1c9e2ac2f8caad6a0548ae68","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_lpmid.npy","bytes":211751106,"sha256_head_tail":"b7c2a8addeba3d50b8c06aa005e1df401340ba60162ef602087bbdcdf93b4a80","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_lsid.npy","bytes":211751106,"sha256_head_tail":"a1056175d85d8be62ddcaf960c445e0714201ef58cde43cca4cecfb149a87445","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_lsynonyms.npy","bytes":211751106,"sha256_head_tail":"d9cb7a325967936b4ec0be9e946565620da4645f68e75610a41bd04597bc6db0","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_n_depositors.npy","bytes":211751106,"sha256_head_tail":"d7523288c660560a3f48d7ce3c60eec191bdff1f10fd730102fa5e6f7bfebf05","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_n_vendors.npy","bytes":211751106,"sha256_head_tail":"2f15536aec60fde65a5ddb85c4a304287d956d63566bcbf873eacbd419ee3945","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_np.npy","bytes":105875617,"sha256_head_tail":"5b50df0bba26e9428906efe11806930b4ec1deab066a4ca5a65b79930c0bc894","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pc_src_mask.npy","bytes":211751106,"sha256_head_tail":"aa0df3a648b07e34fd0c06ee4daaeebb7b84e2a105c308898f44590fdffcb324","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_cid.npy","bytes":2842932,"sha256":"5bd33739596de0d396c04fc50ec1268e95a427917016f06c00cc04e853155b25","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_formula_isomers.npy","bytes":1421530,"sha256":"cce7974779e71c4cd8ff41a3291bf1d57caa48f9a0a1fe7b1ff58a2699eff01e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_has_different_parent.npy","bytes":710829,"sha256":"7b23111c70942bf6c7ee770d103b50a9d9b1c4b530051fa851e3b39d05ea68a8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_lmesh.npy","bytes":1421530,"sha256":"c6309dfcfffaf601dad54de724d0b55956873314b626eac241decf68b6a862bc","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_lotus_refs.npy","bytes":1421530,"sha256":"b3a8e9a44f25c83cc8794a05e0800a1800caa36ebe7f857a10c6b823b51d4dc8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_lpatent.npy","bytes":1421530,"sha256":"ba80c57bc2e83538658a3c7e2ed27e11936d67adcaede98320ce8bd43bc48ce2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_lpmid.npy","bytes":1421530,"sha256":"4cbece84f0a8ae04895df2c7184ac1e121803b9cad1a6de16333848d4834bb5b","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_lsid.npy","bytes":1421530,"sha256":"461ebd8bb1035faf019075bd122f28445a65c52b4b09a209e5d49fad1f7d71e5","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_lsynonyms.npy","bytes":1421530,"sha256":"29bb614126f060af9d9d9257d9a28aeff9288c60227505fae378f0049e36c1b3","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_match.json","bytes":40,"sha256":"2d5c4329ae25830dd658e65a590200c6f570d50f26cc2d705c673e87ae6da240","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_n_depositors.npy","bytes":1421530,"sha256":"97a16d2ac5dc88198db85ea9c1e9ff1bb56763217d8ba3da8aa8f90d22f22ea2","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_n_vendors.npy","bytes":1421530,"sha256":"86a7c47053f0e7a3e1ffad8b934ffa070cf5b752b8c37a06c571fe7b097c40e4","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_np.npy","bytes":710829,"sha256":"87ad6f0afb6cf930b47ed035dd4acc20e1305d1b96ceeff1f8fdbddd6892e390","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_parent_cid.npy","bytes":2842932,"sha256":"1e5e783689c4859e103a719ca8cc51f366ca8c904a09e2d4e21ef5b20acf74e8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_popularity.csv","bytes":180875134,"sha256_head_tail":"fd1ce3432872f2de7111803a739c46c45a3af29a3e81619c8a3946bedd123ec3","full":false},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pool_src_mask.npy","bytes":1421530,"sha256":"6e4b8831abebadf7aa418fc192d58486a86084954e6cdc3ee53543533507d695","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pubchem_tier_sample.csv","bytes":2456397,"sha256":"9a7e83cefbfc4a1e56fa07901dbfb46f49c51e60de587c85eb47357aa8dbaed0","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/pubchemlite_annotations.parquet","bytes":13570437,"sha256":"7a054d9b5933f7bda90965f9671dac234c2bf2420435cd39c2e69177b6a0f15d","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/MANIFEST_20261019.json","bytes":13125,"sha256":"c75d11f02b4c2fdeba1ab4ef9e202fa6e598a2a8f2121606fddd9756bf6b6a3e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/MANIFEST_20261019_2.json","bytes":2274,"sha256":"059b36136ab2a3d01d672436f12dc4a1dd4045993469a327760ccc884b512074","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/MANIFEST_20261020.json","bytes":3384,"sha256":"5e3b7718ad48b19309af67a6cbae4f6892f2abcb66d5fcaa7793570d66f8cdb0","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/SOURCE_CHECK_20261019.md","bytes":1351,"sha256":"aace993777bd139fb0faa5ba5f536b761ed59379c0e918e6602d462f2a44401e","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/VALIDATION_20261019.json","bytes":2248,"sha256":"cfe5b19a2a93d7ec2abd41caad6f1ba7e1ca8f143e137129290883bad810bd1c","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/build_annot.py","bytes":10588,"sha256":"8fe742c60a4bd162cb289116307106bd8bc21ff194fe4fe6ec89928fc73855eb","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/build_delta.py","bytes":4318,"sha256":"df5f993f011ca3a5a9d1814e3c4fc9279895dfa6501febb9bc3b670d1556e7e5","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/build_pop.py","bytes":12053,"sha256":"d9f89fa705b027cba4eec6c75f66896e7b573ae3647b8e863cfc9001f3e064ff","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/build_sources.py","bytes":14959,"sha256":"178d9ff452ca34e43844fa700592ceee6554cc7fd1eedc997c49a0893886ab18","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/download_sources.py","bytes":3195,"sha256":"ac8481f5e763cf566cc841b2915b51df1a99ebcdfbcddaef1d99288a50db9bf8","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/evaluate_sources.py","bytes":9490,"sha256":"23fb840ba2a90034062c48a77f5d74dd297129780b48aa1f3769e8847e8e0529","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/make_preview.py","bytes":3564,"sha256":"f4564e5899de7a8d235164620c28e0b61ca882849438755781303400b6bb0add","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/make_source_groups.py","bytes":5262,"sha256":"31e71a2ca2da6eb1479f4862a63163d47c50365320cb5476029af7037b6a03dd","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/prepare_source_benchmark.py","bytes":3571,"sha256":"1afbc7aacd0d3c65ae547b016d794f46bfaabf95d7ddd0fa108e32b317dce011","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/pubchem_gap.py","bytes":5840,"sha256":"cbdc93b27fe4515d83542cec29b8bb6ae75f238bfae92b58f97e3feffa9e0d72","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/subtitle.txt","bytes":74,"sha256":"4dee933b288e63fbefb1796eac12851b792c9d2de003169a1c2273f8a19130aa","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/scripts/visible_window_coverage.py","bytes":4013,"sha256":"1649f2b5a8d264f9052c155dde6a024155b50587c2812a1acab0b6a5577ccfac","full":true},{"path":"/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/tier_match.json","bytes":101,"sha256":"4ecf39284ce3391fc0cbcd92d5706331f06843c903043639d9fd5e55559be13b","full":true}]'

# Runs before the pipeline (the stage functions are defined in the next cell and called in cell 12):
#  1. writes null_engine.py next to the pipeline modules (SHA-256 pinned). The patched frozen cells 39 and 45 import it
#     and give every reader of Engine 2's lists (ENG, eng_lists.json) Engine 1's own lists in Engine 2's schema.
#  2. records the mounts: every dataset of kernel-metadata.json present, none of the detached ones.
#  3. commit run only: hashes every mounted file of those datasets (and the competition files) that has a row in the
#     embedded copy of casmi26-pubm's v43_mounted_input_hashes.json, with frozen cell 54's digest (full SHA-256 up to
#     32 MiB, else size + SHA-256 of the first and last 4 MiB), matched by its path inside the dataset (the v4g
#     container held each upstream dataset in a folder of the same name), else by file name + size. Prints MATCH /
#     MISMATCH counts and the mismatching names; log only, it never stops the run. MATCH of a file over 32 MiB means
#     size + first and last 4 MiB only (the v43 rows hold no more), so:
#  4. commit run only: the larger files of the datasets that replace container folders (ELIG_FULL_DATASETS; v4b is
#     checked in full by frozen cell 26) are hashed in full and compared with the container MANIFEST's sha256
#     (ELIG_FULL_REFS); the others (fpnet_full1.pt, glacier.pt, the metric wheels) are recorded in elig0_report.json.
def _elig_write_null_engine():
    path = CODE / 'null_engine.py'
    path.write_text(NULL_ENGINE_SRC, encoding='utf-8')
    assert hashlib.sha256(path.read_bytes()).hexdigest() == NULL_ENGINE_SHA256, 'null_engine.py differs from its pin'
    return path


def _elig_root(ref):
    owner, slug = ref.split('/', 1)
    for p in (INPUT / 'datasets' / owner / slug, INPUT / slug):
        if p.is_dir():
            return p
    return None


def _elig_detached_present():
    found = []
    for ref in ELIG_DETACHED:
        slug = ref.split('/', 1)[1]
        found += [str(p) for p in [INPUT / slug] + sorted((INPUT / 'datasets').glob(f'*/{slug}')) if p.is_dir()]
    return found


def _elig_digest(path, limit=32 << 20, edge=4 << 20):
    """Frozen cell 54's quick_digest: (bytes, digest, full)."""
    size = path.stat().st_size
    h = hashlib.sha256()
    with path.open('rb') as source:
        if size <= limit:
            for block in iter(lambda: source.read(8 << 20), b''):
                h.update(block)
        else:
            h.update(source.read(edge))
            source.seek(size - edge)
            h.update(source.read(edge))
    return size, h.hexdigest(), size <= limit


def _elig_reference(rows):
    """v43 rows keyed by (dataset folder, path inside it) and by (file name, bytes)."""
    container = 'datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets/'
    by_rel, by_name = {}, {}
    for r in rows:
        p = r['path'].split('/kaggle/input/', 1)[-1]
        parts = p.split('/')
        if p.startswith(container) and len(parts) > 4:     # a folder of the container = one upstream dataset
            slug, rel = parts[3], '/'.join(parts[4:])
        elif parts[0] == 'datasets' and len(parts) > 3:
            slug, rel = parts[2], '/'.join(parts[3:])
        elif parts[0] == 'competitions' and len(parts) > 2:
            slug, rel = parts[1], '/'.join(parts[2:])
        else:
            slug, rel = '', p
        entry = dict(path=r['path'], bytes=int(r['bytes']), full=bool(r.get('full')),
                     digest=r['sha256'] if r.get('full') else r['sha256_head_tail'])
        by_rel[(slug, rel)] = entry
        by_name.setdefault((parts[-1], entry['bytes']), []).append(entry)
    return by_rel, by_name


def _elig_full_sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as source:
        for block in iter(lambda: source.read(8 << 20), b''):
            h.update(block)
    return h.hexdigest()


def elig_hash_check(datasets, comp, rows, full_refs=None, full_datasets=(), limit=32 << 20, edge=4 << 20, show=40):
    """Compare the mounted files with the v43 rows (and the larger files of full_datasets in full with full_refs);
    returns the report (never raises on a mismatch)."""
    t0 = time.time()
    full_refs = full_refs or {}
    by_rel, by_name = _elig_reference(rows)
    report = dict(match=0, match_full=0, match_head_tail=0, mismatch=0, by_path=0, by_name_size=0, no_row=0,
                  missing_rows=0, bytes_read=0, datasets={}, mismatches=[], missing=[], no_row_examples=[],
                  full=dict(files=0, bytes=0, match=0, mismatch=0, recorded=0, seconds=0.0, mismatches=[], recorded_files=[],
                            references_unchecked=[]))
    full_seen = set()
    targets = [(ref, ref.split('/', 1)[1], root) for ref, root in datasets] + [('competition', comp.name, comp)]
    for ref, slug, root in targets:
        d = dict(root=str(root) if root else None, files=0, match=0, mismatch=0, no_row=0)
        report['datasets'][ref] = d
        if root is None:
            continue
        seen = set()
        for path in sorted(p for p in root.rglob('*') if p.is_file()):
            rel = path.relative_to(root).as_posix()
            d['files'] += 1
            if ref in full_datasets and path.stat().st_size > limit:     # 4. full SHA-256 of the larger files
                t1, name, size = time.time(), f'{ref}/{rel}', path.stat().st_size
                sha, fr = _elig_full_sha256(path), report['full']
                fr['files'] += 1; fr['bytes'] += size; fr['seconds'] += time.time() - t1; report['bytes_read'] += size
                full_seen.add(name)
                if name in full_refs:
                    ok_full = (size, sha) == tuple(full_refs[name])
                    fr['match' if ok_full else 'mismatch'] += 1
                    if not ok_full:
                        fr['mismatches'].append(dict(file=name, bytes=size, sha256=sha, reference_bytes=full_refs[name][0],
                                                     reference_sha256=full_refs[name][1]))
                else:
                    fr['recorded'] += 1
                    fr['recorded_files'].append(dict(file=name, bytes=size, sha256=sha))
            row, how = by_rel.get((slug, rel)), 'by_path'
            if row is None:
                same = by_name.get((path.name, path.stat().st_size)) or []
                if same and len({r['digest'] for r in same}) == 1:
                    row, how = same[0], 'by_name_size'
            if row is None:
                d['no_row'] += 1; report['no_row'] += 1
                if len(report['no_row_examples']) < show:
                    report['no_row_examples'].append(f'{ref}/{rel}')
                continue
            seen.add(row['path'])
            size, digest, full = _elig_digest(path, limit, edge) if path.stat().st_size == row['bytes'] else \
                (path.stat().st_size, None, None)
            report['bytes_read'] += (size if full else 2 * edge) if digest else 0
            ok = digest is not None and digest == row['digest'] and full == row['full']
            report[how] += 1
            key = 'match' if ok else 'mismatch'
            report[key] += 1; d[key] += 1
            if ok:
                report['match_full' if full else 'match_head_tail'] += 1
            if not ok:
                report['mismatches'].append(dict(file=f'{ref}/{rel}', reference=row['path'], bytes=size,
                                                 reference_bytes=row['bytes'], digest=digest, reference_digest=row['digest']))
        for (s, rel), row in by_rel.items():            # reference rows of this dataset that are not on disk
            if s == slug and row['path'] not in seen and not (root / rel).is_file():
                report['missing_rows'] += 1
                report['missing'].append(f'{ref}/{rel}')
    report['full']['references_unchecked'] = sorted(set(full_refs) - full_seen)
    report['full']['seconds'] = round(report['full']['seconds'], 1)
    report['seconds'] = round(time.time() - t0, 1)
    return report


def elig0_setup():
    t0 = time.time()
    path = _elig_write_null_engine()
    roots = [(ref, _elig_root(ref)) for ref in ELIG_DATASETS]
    absent = [ref for ref, root in roots if root is None]
    detached = _elig_detached_present()
    spec_w = globals().get('SPEC_W')
    rep = dict(elig=ELIG, null_engine=str(path), null_engine_sha256=NULL_ENGINE_SHA256, frozen=ELIG_FROZEN,
               frozen_sha256_in_use=FROZEN_SHA256, datasets={ref: str(root) if root else None for ref, root in roots},
               absent=absent, detached_present=detached, spec_w=spec_w, checked_versions=ELIG_CHECKED_VERSIONS)
    print(f"ELIG {ELIG} | F1 datasets mounted {len(roots) - len(absent)} / {len(roots)}"
          f"{' (ABSENT: ' + ', '.join(absent) + ')' if absent else ''} | detached ones present: "
          f"{', '.join(detached) if detached else 'none'} | F2a null Engine 2 (null_engine.py {NULL_ENGINE_SHA256[:12]}) | "
          f"F3 SRC_BITS 0 (frozen cell 26), SPEC_W {spec_w} | F4 external reference spectra off (frozen cell 36) | "
          f"frozen source {FROZEN_SHA256[:12]} (patched from {ELIG_FROZEN['original_sha256'][:12]}; cells "
          f"{ELIG_FROZEN['patched_cells']})", flush=True)
    if spec_w != 0 or FROZEN_SHA256 != ELIG_FROZEN['patched_sha256'] or detached:
        print('ELIG WARNING: this run is not the elig0 configuration (SPEC_W, frozen pin or detached mounts)', flush=True)
    if IS_RERUN:
        print('ELIG HASH CHECK skipped in the scoring rerun (the commit run checks the same mounted versions)', flush=True)
    else:
        try:
            rows = json.loads(V43_HASHES_JSON)
            assert hashlib.sha256(V43_HASHES_JSON.encode()).hexdigest() == V43_HASHES_SHA256, 'embedded table differs'
            chk = elig_hash_check([(ref, root) for ref, root in roots if ref in ELIG_UPSTREAM], COMP, rows,
                                  full_refs=ELIG_FULL_REFS, full_datasets=ELIG_FULL_DATASETS)
            rep['hash_check'] = chk
            print(f"ELIG HASH CHECK | MATCH {chk['match']} (full SHA-256 {chk['match_full']}; over 32 MiB by size + first / "
                  f"last 4 MiB only {chk['match_head_tail']}) | MISMATCH {chk['mismatch']} | matched by path "
                  f"{chk['by_path']}, by name + size {chk['by_name_size']} | files without a v43 row {chk['no_row']} "
                  f"(not hashed) | v43 rows of these datasets missing on disk {chk['missing_rows']} | "
                  f"{chk['bytes_read'] / 1e9:.2f} GB read | {chk['seconds']} s", flush=True)
            fr = chk['full']
            print(f"ELIG FULL HASH | files over 32 MiB of {len(ELIG_FULL_DATASETS)} re-mounted datasets (v4b: frozen cell 26) "
                  f"| hashed {fr['files']} ({fr['bytes'] / 1e9:.2f} GB, {fr['seconds']} s) | vs container MANIFEST sha256: "
                  f"MATCH {fr['match']} | MISMATCH {fr['mismatch']} | references unchecked {len(fr['references_unchecked'])} "
                  f"| recorded without a reference {fr['recorded']} (elig0_report.json)", flush=True)
            for m in fr['mismatches']:
                print(f"  ELIG FULL MISMATCH {m['file']} | bytes {m['bytes']} vs {m['reference_bytes']} | "
                      f"{m['sha256'][:12]} vs {m['reference_sha256'][:12]}", flush=True)
            for name in fr['references_unchecked']:
                print(f'  ELIG FULL UNCHECKED {name} (not on disk, or not over 32 MiB)', flush=True)
            for m in fr['recorded_files']:
                print(f"  ELIG FULL SHA256 {m['file']} {m['bytes']} {m['sha256']}", flush=True)
            for ref, d in chk['datasets'].items():
                print(f"  ELIG HASH {ref}: files {d['files']} | match {d['match']} | mismatch {d['mismatch']} | "
                      f"no row {d['no_row']}", flush=True)
            for m in chk['mismatches'][:40]:
                print(f"  ELIG HASH MISMATCH {m['file']} | bytes {m['bytes']} vs {m['reference_bytes']} | "
                      f"{str(m['digest'])[:12]} vs {m['reference_digest'][:12]}", flush=True)
            for name in chk['missing'][:40]:
                print(f'  ELIG HASH MISSING {name}', flush=True)
        except Exception as exc:                          # log only: the check never stops the run
            rep['hash_check'] = dict(error=repr(exc)[:400])
            print('ELIG HASH CHECK FAILED (log only):', repr(exc)[:400], flush=True)
    rep['seconds'] = round(time.time() - t0, 1)
    (ROOT / 'elig0_report.json').write_text(json.dumps(rep, indent=2, default=str))
    return rep


ELIG_REPORT = elig0_setup()


In [ ]:
# ── Stage machinery: process runner, compact trace, and the four stage functions ───────────────
import contextlib, importlib

STAGES = []           # (stage, seconds, status) for the run summary
STAGE_MARKS = {}      # stage -> [(frozen cell, START/DONE, seconds since stage start)]
FROZEN_STAGE = {
    2: 'Official RDKit 2026.03.3', 6: 'Competition files', 9: 'Asset inventory', 12: 'Documentation prior table',
    14: 'Prior benchmark table', 16: 'Natural-product panel', 18: 'Natural-product knowledge table',
    26: 'Datasets, priors and wheels', 27: 'Determinism patch', 28: 'Preflight', 30: 'Commit or rerun switch',
    32: 'Engine 2 · two rankers', 34: 'PubChem channel', 36: 'Engine 1 · library, pool, spectrum cache',
    37: 'Engine 1 · models and rankers', 39: 'Engine 1 · candidate lists', 41: 'GLACIER + ICEBERG forward models',
    43: 'Gated PubChem merge', 45: 'Two-engine fusion', 48: 'Metric audit and safe tail', 49: 'Metric-identity fusion',
    50: 'Postflight record', 51: 'Fragment score gap', 52: 'Window-40 overlay', 53: 'Engine 2 rank-2 placement',
    54: 'Execution receipt', 56: 'Example answer table'}
CELL_MARK = re.compile(r'^PIPELINE CELL (\d+) (START|DONE)')
KEY_LINE = re.compile(r'FAILED|ERROR|Traceback|merge stats|RANK2_OVERLAY|EXPORTED FROZEN|NEURAL FEATURES|'
                      r'NEURAL input failures|OUTSIDERS|engine lists \d+ |pubchem lists \d+|errors \d+ \|')
BASELINE = ROOT / 'submission_baseline0465.csv'
VARIANTS_DIR = ROOT / 'variants'
RESCUE_SLOTS = {'V3': 3, 'CUSTOM': int(CUSTOM.get('rescue', 0))}.get(VARIANT, 0)
PIPELINE_DONE = NEURAL_OK = OUTSIDERS_OK = CONSENSUS_OK = False
REPORT, MOVES, SUBMITTED, consensus = {}, pd.DataFrame(), None, None


def run_stage(args, log, name, quiet=None):
    """Run one stage in its own process; keep the full log on disk and print a compact trace."""
    quiet = QUIET_LOGS if quiet is None else quiet
    env = {**os.environ, 'PYTHONHASHSEED': '0', 'PYTHONUNBUFFERED': '1'}
    env.pop('INC_PANEL', None)
    t0, marks, tail = time.time(), [], collections.deque(maxlen=150)
    print(f'▶ {name}', flush=True)
    with open(log, 'w') as handle:
        proc = subprocess.Popen([str(a) for a in args], cwd=ROOT, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            handle.write(line); handle.flush()
            t = time.time() - t0
            tail.append(line)
            m = CELL_MARK.match(line)
            if m:
                cell, kind = int(m.group(1)), m.group(2)
                marks.append((cell, kind, t))
                if quiet and kind == 'DONE' and cell in FROZEN_STAGE:
                    start = next((s for c, k, s in reversed(marks) if c == cell and k == 'START'), t)
                    if t - start >= 1:
                        print(f'  ✓ {t / 60:6.1f} min   {FROZEN_STAGE[cell]:<40s} {t - start:8,.0f} s', flush=True)
                continue
            if not quiet:
                if len(line) < 5000:
                    print(line, end='', flush=True)
            elif KEY_LINE.search(line) and len(line) < 800:
                print('      ' + line.strip()[:180], flush=True)
        status = proc.wait()
    seconds = time.time() - t0
    STAGE_MARKS[name] = marks
    STAGES.append((name, seconds, 'ok' if status == 0 else f'exit {status}'))
    if status:
        print(''.join(tail))
        raise RuntimeError(f'{name} failed with exit code {status}; full log: {log}')
    print(f'  done in {minutes(seconds)}', flush=True)
    return seconds


def materialise_sources():
    """Stage 0: write and verify the pinned sources, find the neural bundle, install the official RDKit."""
    global FROZEN
    t0 = time.time()
    for name, text in MODULES.items():
        (CODE / name).write_text(text)
        assert hashlib.sha256((CODE / name).read_bytes()).hexdigest() == MODULE_SHA256[name], name
    FROZEN = CODE / 'frozen_0465.ipynb'
    FROZEN.write_bytes(base64.b64decode(FROZEN_B64))
    assert hashlib.sha256(FROZEN.read_bytes()).hexdigest() == FROZEN_SHA256, 'Pinned pipeline source differs'
    assert sys.version_info[:2] in ((3, 12), (3, 13)), 'Python 3.12 or 3.13 is required'
    if str(CODE) not in sys.path:
        sys.path.insert(0, str(CODE))
    from bundle import find_bundle
    bundle = find_bundle(str(INPUT), ROOT, MODEL_PINS)
    (CODE / 'bootstrap.py').write_text(''.join(json.loads(FROZEN.read_text())['cells'][2]['source']))
    run_stage([sys.executable, CODE / 'bootstrap.py'], ROOT / 'bootstrap.log', 'Official RDKit wheel', quiet=False)
    STAGES[-1] = ('Sources and RDKit wheel', time.time() - t0, 'ok')
    tiles([('Pipeline source', FROZEN_SHA256[:12] + '…', 'SHA-256 verified'),
           ('Helper modules', f'{len(MODULES)}', '6 public + 2 new, SHA-256 verified'),
           ('Neural bundle', Path(bundle).name, '4 files, SHA-256 verified'),
           ('Stage 0', minutes(time.time() - t0), 'including the wheel install')])
    return bundle


def run_public_pipeline():
    """Stage 1: every cell of the pinned pipeline in one process, then the export of its final lists."""
    if not RUN_PIPELINE:
        note('Fast commit.', 'The pipeline is skipped in this commit run; the scoring rerun executes every stage.', BLUE)
        return False
    assert not (ROOT / 'work/spec_meta.parquet').exists() and not (ROOT / 'submission.csv').exists(), \
        'Start from a fresh session (Run All after a restart)'
    seconds = run_stage([sys.executable, CODE / 'runner.py', FROZEN, FROZEN_SHA256, ROOT],
                        ROOT / 'submission_pipeline.log', 'Public 0.465 pipeline')
    receipt = json.loads((ROOT / 'execution_receipt.json').read_text())
    full = receipt['is_full_run']
    tiles([('Molecules ranked', f"{receipt['processed_molecules']:,}", 'hidden test' if full else 'visible smoke subset'),
           ('Answer rows', f'{len(SAMPLE_IDS):,}', 'all ranked' if full else 'placeholders elsewhere'),
           ('Pipeline time', minutes(seconds), 'one process, every cell'),
           ('Baseline file', BASELINE.name, 'kept for reference')])
    return True


def score_final_lists():
    """Stage 2a: the public scorer on exactly the exported candidates."""
    try:
        run_stage([sys.executable, CODE / 'neural_scores.py', BUNDLE, 'final_model.pt',
                   ROOT / 'frozen_candidate_evidence.parquet', ROOT / 'effective_query_spectra.parquet',
                   ROOT / 'neural_candidate_features.parquet', 'cuda'],
                  ROOT / 'submission_neural.log', 'Neural scorer · final lists')
        return True
    except Exception as exc:
        note('Neural scorer failed.', f'The 0.465 answers will be submitted. {type(exc).__name__}: {str(exc)[:300]}', RED)
        return False


def score_outside_candidates(needed):
    """Stage 2b (optional): collect and score candidates ranked just outside the final lists."""
    elapsed_h = (time.time() - NB_T0) / 3600
    if not needed:
        return False
    if elapsed_h >= TIME_GUARD_HOURS:
        note('Tail rescue skipped.', f'The session has run {elapsed_h:.1f} h (guard {TIME_GUARD_HOURS} h).', YELLOW)
        return False
    try:
        run_stage([sys.executable, CODE / 'outsiders.py', FROZEN, ROOT, ROOT / 'frozen_candidate_evidence.parquet',
                   ROOT / 'outsider_evidence.parquet', OUTSIDERS_PER_MOLECULE],
                  ROOT / 'outsiders.log', 'Outside candidates · collect')
        if not len(pd.read_parquet(ROOT / 'outsider_evidence.parquet')):
            return False
        run_stage([sys.executable, CODE / 'neural_scores.py', BUNDLE, 'final_model.pt',
                   ROOT / 'outsider_evidence.parquet', ROOT / 'effective_query_spectra.parquet',
                   ROOT / 'outsider_neural_features.parquet', 'cuda'],
                  ROOT / 'outsiders_neural.log', 'Neural scorer · outside candidates')
        return True
    except Exception as exc:
        note('Tail rescue disabled.', f'{type(exc).__name__}: {str(exc)[:300]}', YELLOW)
        return False


def show_neural_summary():
    if not NEURAL_OK:
        return
    nf = pd.read_parquet(ROOT / 'neural_candidate_features.parquet')
    act = nf[nf.processed.astype(bool)]
    n_out = len(pd.read_parquet(ROOT / 'outsider_neural_features.parquet')) if OUTSIDERS_OK else 0
    tiles([('Candidates scored', f'{len(act):,}', f'{act.molecule_id.nunique():,} molecules'),
           ('Scored molecules', f'{int(act.groupby("molecule_id").neural_ok.all().sum()):,}', 'all spectra usable'),
           ('Spectra per molecule', f'{act.groupby("molecule_id").n_spectra.first().median():.0f}', 'median used by the scorer'),
           ('Outside candidates', f'{n_out:,}', 'scored for the tail rescue' if n_out else 'not needed for this variant')])


def build_consensus():
    """Stage 3a: the public 0.466 file, its exact rebuild, then every variant."""
    global consensus
    try:
        consensus = importlib.import_module('consensus')
        from submission import apply as public_apply
        evidence = ROOT / 'neural_candidate_features.parquet'
        with contextlib.redirect_stdout(io.StringIO()):      # the public helper prints its own report
            public_apply(evidence, BASELINE, ROOT / 'submission_public0466.csv', weight=0.25, offset=3.0)
        consensus.build_variants(evidence, BASELINE, ROOT / 'variants_check', {'V1': consensus.PRESETS['V1']}, 3.0)
        mine = pd.read_csv(ROOT / 'variants_check/submission_V1.csv', dtype=str, keep_default_na=False)
        public = pd.read_csv(ROOT / 'submission_public0466.csv', dtype=str, keep_default_na=False)
        assert mine.equals(public), 'V1 differs from the public 0.466 file'
        presets = dict(consensus.PRESETS)
        if VARIANT == 'CUSTOM':
            presets['CUSTOM'] = dict(w_top=float(CUSTOM['w_top']), w_mid=float(CUSTOM['w_mid']),
                                     rescue=int(CUSTOM.get('rescue', 0)), label='Custom weights')
        report, moves = consensus.build_variants(
            evidence, BASELINE, VARIANTS_DIR, presets, RRF_OFFSET,
            ROOT / 'outsider_neural_features.parquet' if OUTSIDERS_OK else None)
        print('V1 rebuilt exactly: the new code reproduces the public 0.466 file row for row.')
        return True, report, moves
    except Exception as exc:
        note('Consensus layer disabled.', f'{type(exc).__name__}: {str(exc)[:400]}', RED)
        return False, {}, pd.DataFrame()


def identity_map():
    keys = {}
    for path in (ROOT / 'neural_candidate_features.parquet', ROOT / 'outsider_neural_features.parquet'):
        if path.is_file():
            f = pd.read_parquet(path, columns=['smiles', 'metric_key'])
            keys.update({s: k for s, k in zip(f.smiles, f.metric_key) if isinstance(k, str)})
    return keys


def well_formed(path, same_candidates=True):
    """Minimal checks that do not depend on the new modules."""
    f = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert list(f.columns) == ['molecule_id', 'smiles'] and f.molecule_id.tolist() == SAMPLE_IDS, 'rows'
    lists = f.smiles.str.split(';')
    assert lists.map(len).between(1, 25).all(), 'list length'
    assert lists.map(lambda v: all(x.strip() for x in v) and len(set(v)) == len(v)).all(), 'SMILES'
    if same_candidates and BASELINE.is_file():
        b = pd.read_csv(BASELINE, dtype=str, keep_default_na=False)
        assert all(sorted(x.split(';')) == sorted(y.split(';')) for x, y in zip(b.smiles, f.smiles)), 'candidates'


def choose_and_submit():
    """Stage 3b: chosen variant -> V1 -> V0 -> public 0.466 file -> baseline file (-> sample in a fast commit)."""
    t0 = time.time()
    chain = []
    if CONSENSUS_OK:
        chain += [(n, VARIANTS_DIR / f'submission_{n}.csv') for n in dict.fromkeys([VARIANT, 'V1', 'V0'])]
    chain += [('public 0.466 file', ROOT / 'submission_public0466.csv'), ('baseline file', BASELINE)]
    chosen = None
    for name, path in chain:
        if not path.is_file():
            continue
        try:
            rescue = name in REPORT and REPORT[name].get('rescued', 0) > 0
            if rescue:
                consensus.validate(path, SAMPLE_IDS, BASELINE, allow_new=True, identity_of=identity_map())
            well_formed(path, same_candidates=not rescue)
            chosen = (name, path)
            break
        except Exception as exc:
            note(f'{name} rejected.', f'{type(exc).__name__}: {str(exc)[:300]}', YELLOW)
    if chosen is None:
        assert not IS_RERUN, 'No valid answer file in the scoring rerun'
        chosen = ('sample (fast commit)', COMP / 'sample_submission.csv')
    shutil.copyfile(chosen[1], ROOT / 'submission.csv')
    STAGES.append(('Consensus and checks', time.time() - t0, 'ok'))
    return chosen[0]


def show_variant_board():
    if CONSENSUS_OK:
        board = pd.DataFrame([{
            'Variant': k, 'Rule': v['label'], 'Leader weight': f"{v['w_top']:.2f}",
            'Ranks 2–25 weight': f"{v['w_mid']:.2f}", 'Tail rescue': v['rescue'] or '–',
            'Rows re-ordered': f"{v['changed_rows']:,}", 'Rank 1 changed': f"{v['changed_top1']:,}",
            'Rescued': f"{v['rescued']:,}"} for k, v in REPORT.items()])
        board.index = board.Variant
        table_html(board, highlight=SUBMITTED, numeric=board.columns[2:])
    sub = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
    tiles([('Submitted', SUBMITTED, 'copied to submission.csv'),
           ('Rows', f'{len(sub):,}', 'one per molecule, sample order'),
           ('Candidates per row', f'{sub.smiles.str.count(";").add(1).median():.0f}', 'median, at most 25'),
           ('SHA-256', hashlib.sha256((ROOT / 'submission.csv').read_bytes()).hexdigest()[:12] + '…', 'of submission.csv')])

In [ ]:
from pathlib import Path
_physical = '"""Untrained MetFrag-lite re-scorer: how much MS/MS intensity a candidate\'s <=2-bond-cleavage fragments explain.\n\nIndependent of pv.py (same idea as pv.fragment_masses / explain_score_adduct, new code): enumerate the neutral masses\nof the connected pieces left after removing 1 or 2 bonds of the heavy-atom graph (implicit H stay on their atom),\nremember the fewest cleavages that give each mass, then match every peak against\n`neutral fragment + dh*H + charge carrier` for dh in `h_shifts` and the carriers the precursor adduct allows.\nA peak\'s credit is the weight of its cheapest explanation (1 for 0/1 cleavages, `w2` for 2, `w3` for 3),\nand a spectrum\'s score is the credited share of its (weighted) intensity. Defaults follow the forum ablation on a\n241-structure same-formula isomer panel (+0.065 MRR over the common baseline): w2=0.6, linear intensity,\ntol 0.005 Da, H shifts -2..+3, at most 2 cleavages. `BASELINE_CFG` is the common/pv.py-style setting.\n\n    score_molecule(spectra, cand_smiles, cfg=None, cache=None) -> list[float | None]\n\n`spectra`: list of dicts {mz: ndarray, it: ndarray (base peak = 1), adduct: str, prec: float, mode: +1/-1}.\nReturns one score per candidate (mean over the spectra that have peaks; higher = better), None if the SMILES does\nnot parse. Fragment masses are cached per SMILES in `cache` (a dict; pass one in to reuse it across molecules).\nThe enumeration is numba-compiled when numba is importable (pure-Python fallback otherwise, ~100x slower).\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\n# monoisotopic masses (same values as pv.AMU); other elements come from RDKit\'s periodic table\nAMU = {\'C\': 12.0, \'H\': 1.00782503207, \'N\': 14.0030740048, \'O\': 15.9949146196, \'P\': 30.97376163,\n       \'S\': 31.97207100, \'F\': 18.99840322, \'Cl\': 34.96885268, \'Br\': 78.9183371, \'I\': 126.904473,\n       \'Na\': 22.9897692809, \'K\': 38.96370668, \'Si\': 27.9769265325, \'B\': 11.0093054, \'Se\': 79.9165213}\nH = AMU[\'H\']\nE = 0.00054857990\nPROTON = H - E\nH2O = 2 * H + AMU[\'O\']\nNH4 = AMU[\'N\'] + 4 * H\nFORMATE = AMU[\'C\'] + 2 * H + 2 * AMU[\'O\']\n\n# charge carriers a fragment ion may keep, per precursor adduct (added to the neutral fragment mass)\nCARRIERS = {\n    \'[M+H]+\': (PROTON,), \'[M-H2O+H]+\': (PROTON,), \'[M-2H2O+H]+\': (PROTON,),\n    \'[M+NH4]+\': (PROTON, NH4 - E), \'[M+Na]+\': (PROTON, AMU[\'Na\'] - E), \'[M+K]+\': (PROTON, AMU[\'K\'] - E),\n    \'[M-H]-\': (-PROTON,), \'[M-H2O-H]-\': (-PROTON,), \'[M+CH2O2-H]-\': (-PROTON, FORMATE - PROTON),\n    \'[M+Cl]-\': (-PROTON, AMU[\'Cl\'] + E),\n}\n\nDEFAULT_CFG = dict(\n    w1=1.0,                      # credit for a peak explained by the precursor or a 1-bond cleavage\n    w2=0.6,                      # credit for a peak only explained by a 2-bond cleavage\n    w3=0.3,                      # credit for 3-bond cleavages (only used when max_breaks=3)\n    intensity=\'linear\',          # \'linear\' | \'sqrt\' | a float power applied to `it`\n    mz_power=0.0,                # extra peak weight (mz / prec) ** mz_power (0 = off)\n    tol=0.005,                   # absolute m/z tolerance, Da\n    h_shifts=tuple(range(-2, 4)),  # hydrogen shifts dh applied to neutral fragments (-2..+3)\n    max_breaks=2,                # bond cleavages per fragment (1..3)\n    max_bonds_2=100,             # above this many bonds only 1-bond cleavages are enumerated (cost cap)\n    max_bonds_3=40,              # above this many bonds 3-bond cleavages are skipped\n)\n# the common baseline (pv.py-style): all cleavages equal, sqrt intensity, 0.01 Da, H shifts -2..+2\nBASELINE_CFG = dict(DEFAULT_CFG, w2=1.0, intensity=\'sqrt\', tol=0.01, h_shifts=tuple(range(-2, 3)))\n\n\ndef resolve_cfg(cfg=None):\n    """DEFAULT_CFG updated with `cfg` (unknown keys raise, so typos in variants do not pass silently)."""\n    out = dict(DEFAULT_CFG)\n    for k, v in (cfg or {}).items():\n        if k not in out:\n            raise KeyError(f\'frag_rescore: unknown cfg key {k!r}\')\n        out[k] = v\n    out[\'h_shifts\'] = tuple(int(d) for d in out[\'h_shifts\'])\n    if not 1 <= int(out[\'max_breaks\']) <= 3:\n        raise ValueError(\'frag_rescore: max_breaks must be 1, 2 or 3\')\n    return out\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# fragment enumeration (numba kernel with a pure-Python fallback: same source)\n# ---------------------------------------------------------------------------------------------------------------\ndef _find(parent, a):\n    while parent[a] != a:\n        parent[a] = parent[parent[a]]\n        a = parent[a]\n    return a\n\n\ndef _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, k):\n    """Union the kept bonds; if the graph splits, append each piece\'s mass with cleavage count k."""\n    for a in range(n):\n        parent[a] = a\n        acc[a] = 0.0\n    ncomp = n\n    for e in range(ea.shape[0]):\n        if removed[e]:\n            continue\n        ra = _find(parent, ea[e])\n        rb = _find(parent, eb[e])\n        if ra != rb:\n            parent[ra] = rb\n            ncomp -= 1\n    if ncomp < 2:\n        return cnt\n    for a in range(n):\n        r = _find(parent, a)\n        acc[r] += w[a]\n    for a in range(n):\n        if parent[a] == a:\n            out_m[cnt] = acc[a]\n            out_k[cnt] = k\n            cnt += 1\n    return cnt\n\n\ndef _enumerate(n, w, ea, eb, max_breaks, base_components):\n    """All piece masses after removing 1..max_breaks bonds (k = number removed); row 0.. are the intact pieces."""\n    nb = ea.shape[0]\n    n2 = nb * (nb - 1) // 2 if max_breaks >= 2 else 0\n    n3 = nb * (nb - 1) * (nb - 2) // 6 if max_breaks >= 3 else 0\n    cap = base_components + (base_components + 1) * nb + (base_components + 2) * n2 + (base_components + 3) * n3\n    out_m = np.empty(cap, np.float64)\n    out_k = np.empty(cap, np.int8)\n    parent = np.empty(n, np.int64)\n    acc = np.empty(n, np.float64)\n    removed = np.zeros(nb, np.bool_)\n    cnt = 0\n    for i in range(nb):\n        removed[i] = True\n        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 1)\n        if max_breaks >= 2:\n            for j in range(i + 1, nb):\n                removed[j] = True\n                cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 2)\n                if max_breaks >= 3:\n                    for l in range(j + 1, nb):\n                        removed[l] = True\n                        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 3)\n                        removed[l] = False\n                removed[j] = False\n        removed[i] = False\n    return out_m[:cnt], out_k[:cnt]\n\n\ntry:  # numba is on the Kaggle image and in requirements.txt; NUMBA_CACHE_DIR is set by nb/01_setup.py\n    from numba import njit\n    _find = njit(cache=True, nogil=True)(_find)\n    _emit_components = njit(cache=True, nogil=True)(_emit_components)\n    _enumerate = njit(cache=True, nogil=True)(_enumerate)\n    HAVE_NUMBA = True\nexcept Exception:  # pragma: no cover - fallback path\n    HAVE_NUMBA = False\n\n\ndef mol_graph(smi):\n    """(atom masses incl. implicit H, bond begin idx, bond end idx, n connected pieces) or None if unparsable."""\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    try:\n        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) and smi else None\n    except Exception:\n        m = None\n    if m is None or m.GetNumAtoms() == 0:\n        return None\n    pt = Chem.GetPeriodicTable()\n    w = np.empty(m.GetNumAtoms(), np.float64)\n    for a in m.GetAtoms():\n        s = a.GetSymbol()\n        am = AMU.get(s)\n        if am is None:\n            am = pt.GetMostCommonIsotopeMass(s)\n        w[a.GetIdx()] = am + a.GetTotalNumHs() * H\n    ea = np.array([b.GetBeginAtomIdx() for b in m.GetBonds()], np.int64)\n    eb = np.array([b.GetEndAtomIdx() for b in m.GetBonds()], np.int64)\n    return w, ea, eb, len(Chem.GetMolFrags(m))\n\n\ndef fragment_levels(smi, max_breaks=2, max_bonds_2=100, max_bonds_3=40):\n    """Sorted neutral fragment masses grouped by the fewest cleavages that produce them: [k=0, k=1, ..., k=max].\n\n    k=0 holds the intact molecule (sum of its pieces, and each piece if the SMILES has several). None if the\n    SMILES does not parse.\n    """\n    g = mol_graph(smi)\n    if g is None:\n        return None\n    w, ea, eb, npieces = g\n    nb = len(ea)\n    mb = int(max_breaks)\n    if mb >= 3 and nb > max_bonds_3:\n        mb = 2\n    if mb >= 2 and nb > max_bonds_2:\n        mb = 1\n    levels = [np.array([w.sum()])]\n    if npieces > 1:  # salts / mixtures: each intact piece also counts as uncleaved\n        from rdkit import Chem\n        m = Chem.MolFromSmiles(smi)\n        levels = [np.unique(np.concatenate([[w.sum()], [w[list(ix)].sum() for ix in Chem.GetMolFrags(m)]]))]\n    if nb == 0:\n        return levels + [np.zeros(0) for _ in range(int(max_breaks))]\n    fm, fk = _enumerate(len(w), w, ea, eb, mb, npieces)\n    fm = np.concatenate([levels[0], fm])\n    fk = np.concatenate([np.zeros(len(levels[0]), np.int8), fk])\n    key = np.round(fm, 5)\n    order = np.lexsort((fk, key))                  # by mass, then fewest cleavages first\n    key, fm, fk = key[order], fm[order], fk[order]\n    first = np.ones(len(key), bool)\n    first[1:] = key[1:] != key[:-1]\n    fm, fk = fm[first], fk[first]\n    return [np.sort(fm[fk == k]) for k in range(int(max_breaks) + 1)]\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# scoring\n# ---------------------------------------------------------------------------------------------------------------\ndef _carriers(adduct, mode):\n    c = CARRIERS.get(adduct)\n    if c is not None:\n        return c\n    if isinstance(adduct, str) and adduct.endswith(\'-\'):\n        return (-PROTON,)\n    if isinstance(adduct, str) and adduct.endswith(\'+\'):\n        return (PROTON,)\n    return (PROTON,) if (mode or 1) > 0 else (-PROTON,)\n\n\ndef _prep_spectrum(s, c):\n    """Peak weights and the neutral masses each peak would correspond to, for every (carrier, dh) pair."""\n    mz = np.asarray(s.get(\'mz\', ()), np.float64).ravel()\n    it = np.asarray(s.get(\'it\', ()), np.float64).ravel()\n    n = min(len(mz), len(it))\n    mz, it = mz[:n], it[:n]\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\n    mz, it = mz[ok], it[ok]\n    if len(mz) == 0:\n        return None\n    p = c[\'intensity\']\n    iw = it if p == \'linear\' else np.sqrt(it) if p == \'sqrt\' else it ** float(p)\n    if c[\'mz_power\']:\n        prec = float(s.get(\'prec\') or mz.max())\n        iw = iw * (mz / prec) ** float(c[\'mz_power\'])\n    tot = iw.sum()\n    if not tot > 0:\n        return None\n    shifts = np.array([car + dh * H for car in _carriers(s.get(\'adduct\'), s.get(\'mode\')) for dh in c[\'h_shifts\']])\n    q = (mz[None, :] - shifts[:, None]).ravel()  # neutral mass needed, per (shift, peak)\n    return iw / tot, q, len(shifts), len(mz)\n\n\ndef _score_spectrum(levels, weights, prep, tol):\n    iw, q, ns, npk = prep\n    best = np.zeros(npk)\n    for arr, wt in zip(levels, weights):\n        if len(arr) == 0 or wt <= 0:\n            continue\n        a = np.searchsorted(arr, q - tol, \'left\')\n        b = np.searchsorted(arr, q + tol, \'right\')\n        hit = (b > a).reshape(ns, npk).any(axis=0)\n        best = np.where(hit & (wt > best), wt, best)\n    return float(iw @ best)\n\n\ndef candidate_levels(smi, cfg=None, cache=None):\n    """Cached fragment_levels for one SMILES under `cfg` (only the enumeration keys matter)."""\n    return _levels_cached(smi, resolve_cfg(cfg), cache)\n\n\ndef _levels_cached(smi, c, cache):\n    key = (smi, int(c[\'max_breaks\']), int(c[\'max_bonds_2\']), int(c[\'max_bonds_3\']))\n    if cache is not None and key in cache:\n        return cache[key]\n    try:\n        lv = fragment_levels(smi, c[\'max_breaks\'], c[\'max_bonds_2\'], c[\'max_bonds_3\'])\n    except Exception:\n        lv = None\n    if cache is not None:\n        cache[key] = lv\n    return lv\n\n\ndef score_molecule(spectra, cand_smiles, cfg=None, cache=None):\n    """One score per candidate: mean explained-intensity share over the molecule\'s spectra (None: unparsable)."""\n    c = resolve_cfg(cfg)\n    cache = {} if cache is None else cache\n    weights = [1.0, float(c[\'w1\']), float(c[\'w2\']), float(c[\'w3\'])][: int(c[\'max_breaks\']) + 1]\n    preps = [p for p in (_prep_spectrum(s, c) for s in (spectra or [])) if p is not None]\n    tol = float(c[\'tol\'])\n    out = []\n    for smi in cand_smiles:\n        lv = _levels_cached(smi, c, cache)\n        if lv is None:\n            out.append(None)\n            continue\n        if not preps:\n            out.append(0.0)\n            continue\n        out.append(float(np.mean([_score_spectrum(lv, weights, p, tol) for p in preps])))\n    return out\n\n\ndef warmup():\n    """Compile the numba kernel once (a few seconds) so the first real molecule is not charged for it."""\n    fragment_levels(\'CCOC(=O)c1ccccc1O\')\n    return HAVE_NUMBA\n'
_post = "from __future__ import annotations\nimport hashlib\nimport json\nimport math\nimport os\nimport time\nfrom functools import lru_cache\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem\nfrom rdkit.Chem import rdMolDescriptors\n\n@lru_cache(maxsize=30000)\ndef mol_formula(s):\n    try:\n        m = Chem.MolFromSmiles(str(s))\n        return rdMolDescriptors.CalcMolFormula(m) if m is not None else None\n    except Exception:\n        return None\n\ndef finite(x,default=np.nan):\n    try:\n        t=float(x)\n        return t if np.isfinite(t) else default\n    except (ValueError,TypeError):\n        return default\n\ndef select_spectra(frame,max_n=12):\n    out=[]\n    by_bucket={}\n    for r in frame.itertuples(index=False):\n        mz=np.asarray(r.ms2_mzs,dtype=float).ravel()\n        it=np.asarray(r.ms2_normalized_intensities,dtype=float).ravel()\n        n=min(len(mz),len(it));mz,it=mz[:n],it[:n]\n        good=np.isfinite(mz)&np.isfinite(it)&(it>0)\n        if good.sum()<5:continue\n        order=np.argsort(it[good])[-120:]\n        mz,it=mz[good][order],it[good][order]\n        e=finite(getattr(r,'collision_energy_ev',30),30.)\n        adduct=str(getattr(r,'adduct','[M+H]+'))\n        mode=1 if str(getattr(r,'ionization_mode','positive')).lower()=='positive' else -1\n        item={'mz':mz,'it':it,'adduct':adduct,'prec':finite(getattr(r,'precursor_mz',None),float(mz.max())),'mode':mode}\n        bucket=(adduct,round(e/10))\n        by_bucket.setdefault(bucket,[]).append((int(good.sum()),item))\n    for key in sorted(by_bucket):\n        out += [v for _,v in sorted(by_bucket[key],key=lambda x:-x[0])[:2]]\n    return out[:max_n]\n\ndef support(payload,mid,s):\n    eng=(payload.get('engine2',{}).get(mid) or {}).get('smiles') or []\n    pc=(payload.get('pubchem',{}).get(mid) or {}).get('pc') or []\n    main=payload.get('base_tail',{}).get(mid) or []\n    fusion=payload.get('fusion_overflow',{}).get(mid) or []\n    def rank(v):return v.index(s)+1 if s in v else 0\n    return dict(engine2=rank(eng),pubchem=rank(pc),main=rank(main),fusion=rank(fusion))\n\ndef score_candidates(frag,spec,candidates,cache):\n    if not spec:return None\n    opt=np.asarray([np.nan if a is None else a for a in frag.score_molecule(spec,candidates,cache=cache)],float)\n    old=np.asarray([np.nan if a is None else a for a in frag.score_molecule(spec,candidates,cfg=frag.BASELINE_CFG,cache=cache)],float)\n    if not np.isfinite(opt).all() or not np.isfinite(old).all():return None\n    chunks=[spec[i:i+2] for i in range(0,len(spec),2)]\n    group=np.array([[np.nan if v is None else v for v in frag.score_molecule(ch,candidates,cache=cache)] for ch in chunks],float)\n    return opt,old,group\n\ndef decision(mid,control,listed,outs,query,payload,frag,cache,allow_tail=True,allow_top=True):\n    diag={'molecule_id':mid,'top1_changed':False,'tail_changed':False,'local_swaps':0,'reason':'unchanged','n_spectra':0,\n          'candidate_count':len(control),'top1_frag_margin':0.,'top1_neural_margin':0.,'new_candidate_source':''}\n    bysmiles=listed.set_index('smiles',drop=False)\n    if len(bysmiles)!=len(listed) or any(s not in bysmiles.index for s in control):\n        diag['reason']='evidence_mismatch';return control,diag\n    if not bool(listed.neural_ok.all()) or not bool(listed.processed.all()):\n        diag['reason']='unscored';return control,diag\n    n=len(control)\n    if n<3:return control,diag\n    anchor=mol_formula(control[0])\n    if anchor is None:\n        diag['reason']='formula_missing';return control,diag\n    cand=list(control[:min(12,n)])\n    outrows=[]\n    if allow_tail and outs is not None and len(outs) and n>=20:\n        for row in outs.sort_values('baseline_rank').itertuples(index=False):\n            if not bool(row.neural_ok) or not bool(row.processed):continue\n            s=str(row.smiles)\n            if s in control or mol_formula(s)!=anchor:continue\n            ev=support(payload,mid,s)\n            sources=int(bool(ev['engine2']))+int(bool(ev['pubchem']))+int(bool(ev['main']))\n            if sources<2 and not (0<ev['fusion']<=3):continue\n            outrows.append(row)\n            cand.append(s)\n            if len(outrows)>=8:break\n    spec=select_spectra(query)\n    diag['n_spectra']=len(spec)\n    if len(spec)<2:\n        diag['reason']='few_spectra';return control,diag\n    data=score_candidates(frag,spec,cand,cache)\n    if data is None:\n        diag['reason']='frag_unavailable';return control,diag\n    frag_new,frag_old,energy=data\n    pos={s:i for i,s in enumerate(cand)}\n    leader=control[0];lp=pos[leader]\n    neu_vals=np.asarray([finite(x) for x in listed.neural_score],float)\n    neural_sd=max(float(np.nanstd(neu_vals)),1e-6)\n    result=list(control)\n    lib=finite(bysmiles.loc[leader,'library_max'],0.)\n    if allow_top and lib<.87 and len(spec)>=3:\n        contenders=[]\n        for ix,s in enumerate(control[1:min(6,n)],1):\n            if mol_formula(s)!=anchor:continue\n            j=pos[s]; ev=bysmiles.loc[s]; lead_ev=bysmiles.loc[leader]\n            if not np.isfinite(frag_new[j]) or frag_new[j]<.27:continue\n            margin=float(frag_new[j]-frag_new[lp]); oldmargin=float(frag_old[j]-frag_old[lp])\n            if margin<.095 or oldmargin<.028:continue\n            votes=float((energy[:,j]>energy[:,lp]+.025).mean())\n            if votes<.70:continue\n            nmargin=(finite(ev.neural_score)-finite(lead_ev.neural_score))/neural_sd\n            if not np.isfinite(nmargin) or nmargin<.23:continue\n            if finite(ev.neural_cos)<finite(lead_ev.neural_cos)+.003 or finite(ev.neural_fp)<finite(lead_ev.neural_fp)-.012:continue\n            supports=0\n            for col in ('engine2_rank','pubchem_rank','main_rank'):\n                rank=finite(ev.get(col),0)\n                if 0<rank<=5:supports+=1\n            simulator_vote=0\n            for col in ('ice','gl'):\n                a,b=finite(ev.get(col)),finite(lead_ev.get(col))\n                if np.isfinite(a) and np.isfinite(b):\n                    if a>=b:simulator_vote+=1\n                    elif a<b-.35:simulator_vote-=1\n            if supports<1 and simulator_vote<1:continue\n            if simulator_vote<=-1:continue\n            contenders.append((margin+.05*nmargin+.025*simulator_vote+.01*supports,s,margin,nmargin))\n        if contenders:\n            _,winner,margin,nmargin=max(contenders,key=lambda v:v[0])\n            result.remove(winner);result.insert(0,winner)\n            diag.update(top1_changed=True,top1_frag_margin=round(margin,5),top1_neural_margin=round(nmargin,5),reason='fragment_neural_top1')\n    swaps=0\n    for i in range(2,min(11,n)):\n        prev,s=result[i-1],result[i]\n        if mol_formula(prev)!=mol_formula(s):continue\n        j,k=pos.get(s),pos.get(prev)\n        if j is None or k is None:continue\n        if frag_new[j]-frag_new[k]<.085 or frag_old[j]-frag_old[k]<.015:continue\n        if (energy[:,j]>energy[:,k]+.010).mean()<.65:continue\n        neui,neup=finite(bysmiles.loc[s,'neural_score']),finite(bysmiles.loc[prev,'neural_score'])\n        if not np.isfinite(neui) or neui<neup-.08*neural_sd:continue\n        result[i-1],result[i]=result[i],result[i-1]\n        swaps+=1\n        if swaps>=2:break\n    diag['local_swaps']=swaps\n    if not diag['top1_changed'] and swaps:diag['reason']='adjacent_fragment_consensus'\n    if allow_tail and outrows and len(result)==25:\n        last=result[-1]\n        if last in pos:\n            lastp=pos[last]\n            eligible=[]\n            listed_keys=set(listed.metric_key.dropna().astype(str))\n            for row in outrows:\n                s=str(row.smiles);j=pos[s]\n                if not isinstance(row.metric_key,str) or row.metric_key in listed_keys:continue\n                if not np.isfinite(frag_new[j]) or frag_new[j]<.28:continue\n                fm=float(frag_new[j]-frag_new[lastp])\n                if fm<.065 or frag_old[j]-frag_old[lastp]<.02:continue\n                if (energy[:,j]>energy[:,lastp]+.01).mean()<.65:continue\n                if finite(row.neural_score)<finite(bysmiles.loc[last,'neural_score'])+.22*neural_sd:continue\n                if finite(row.neural_cos)<=finite(bysmiles.loc[last,'neural_cos']) or finite(row.neural_fp)<=finite(bysmiles.loc[last,'neural_fp']):continue\n                ev=support(payload,mid,s)\n                supports=int(bool(ev['main']))+int(bool(ev['engine2']))+int(bool(ev['pubchem']))\n                eligible.append((fm,s,ev,supports))\n            if eligible:\n                margin,s,ev,supports=max(eligible,key=lambda q:q[0]+.015*q[3])\n                result[-1]=s\n                diag.update(tail_changed=True,new_candidate_source='+' .join(k for k in ('main','engine2','pubchem','fusion') if ev[k]),reason='tail_supported' if not diag['top1_changed'] and not swaps else diag['reason'])\n    assert len(result)==len(control) and len(set(result))==len(result)\n    assert set(result[:-1]).issubset(set(control)) and not (diag['top1_changed'] and result[0]==control[0])\n    return result,diag\n\ndef apply(root,comp,max_seconds=1500,allow_tail=True,allow_top=True):\n    import frag_rescore as frag\n    root=Path(root);comp=Path(comp)\n    control=root/'variants/submission_V2.csv'\n    if not control.is_file():raise FileNotFoundError(control)\n    src=pd.read_csv(control,dtype=str,keep_default_na=False)\n    need=['molecule_id','smiles','metric_key','processed','neural_ok','neural_score','neural_cos','neural_fp','baseline_rank']\n    listed=pd.read_parquet(root/'neural_candidate_features.parquet')\n    if not set(need).issubset(listed.columns):raise ValueError('Neural evidence columns missing')\n    outsiders_path=root/'outsider_neural_features.parquet'\n    outsider=pd.read_parquet(outsiders_path) if outsiders_path.is_file() else pd.DataFrame()\n    q=pd.read_parquet(root/'effective_query_spectra.parquet')\n    payload=json.loads((root/'e004_candidate_inputs.json').read_text())\n    lmap={str(mid):g.sort_values('baseline_rank') for mid,g in listed.groupby('molecule_id',sort=False)}\n    omap={str(mid):g for mid,g in outsider.groupby('molecule_id',sort=False)} if len(outsider) else {}\n    qmap={str(mid):g for mid,g in q.groupby('molecule_id',sort=False)}\n    start=time.monotonic()\n    cache={}; rows=[];audits=[];expired=False\n    frag.warmup()\n    for i,(mid,smiles) in enumerate(src.itertuples(index=False)):\n        mid=str(mid);old=smiles.split(';')\n        if time.monotonic()-start>max_seconds:\n            expired=True\n            rows.append((mid,smiles));audits.append(dict(molecule_id=mid,reason='time_budget',top1_changed=False,tail_changed=False,local_swaps=0))\n            continue\n        if mid not in lmap or mid not in qmap:\n            rows.append((mid,smiles));audits.append(dict(molecule_id=mid,reason='not_processed',top1_changed=False,tail_changed=False,local_swaps=0))\n            continue\n        try:\n            new,meta=decision(mid,old,lmap[mid],omap.get(mid),qmap[mid],payload,frag,cache,allow_tail=allow_tail,allow_top=allow_top)\n        except Exception as exc:\n            new,meta=old,dict(molecule_id=mid,reason='error:'+type(exc).__name__,top1_changed=False,tail_changed=False,local_swaps=0)\n        rows.append((mid,';'.join(new)));audits.append(meta)\n        if (i+1)%40==0:print('PARALLAX',i+1,'/',len(src),'elapsed',round(time.monotonic()-start,1),'s',flush=True)\n        if len(cache)>35000:cache.clear()\n    dest=pd.DataFrame(rows,columns=['molecule_id','smiles'])\n    ids=pd.read_parquet(comp/'test.parquet',columns=['molecule_id']).molecule_id.astype(str).unique().tolist()\n    if dest.molecule_id.tolist()!=src.molecule_id.tolist() or set(ids)!=set(dest.molecule_id):raise AssertionError('Molecule coverage changed')\n    for initial,new in zip(src.smiles,dest.smiles):\n        before=initial.split(';');after=new.split(';')\n        if not 1<=len(after)<=25 or len(after)!=len(set(after)) or any(not k for k in after):raise AssertionError('Submission format')\n        if after[0] not in before or not set(after[:-1]).issubset(set(before)):raise AssertionError('Unsafe candidate replacement')\n        if len(set(after)-set(before))>1:raise AssertionError('Too many outsiders')\n    out=root/'submission.csv';temp=root/'submission.parallax.tmp'\n    dest.to_csv(temp,index=False)\n    reread=pd.read_csv(temp,dtype=str,keep_default_na=False)\n    if not reread.equals(dest):raise AssertionError('CSV roundtrip failed')\n    temp.replace(out)\n    report=pd.DataFrame(audits)\n    report.to_csv(root/'spectral_parallax_audit.csv',index=False)\n    diagnostics={'control_sha256':hashlib.sha256(control.read_bytes()).hexdigest(),\n        'submission_sha256':hashlib.sha256(out.read_bytes()).hexdigest(),\n        'rows':len(dest),'changed_rows':int((src.smiles!=dest.smiles).sum()),\n        'top1_changed':int(sum(bool(r.get('top1_changed')) for r in audits)),\n        'tail_changed':int(sum(bool(r.get('tail_changed')) for r in audits)),\n        'adjacent_swaps':int(sum(int(r.get('local_swaps',0)) for r in audits)),\n        'time_budget_exceeded':expired,'elapsed_seconds':round(time.monotonic()-start,2),\n        'outcome':'experimental_unvalidated','baseline':'V2_0.468_user_reported'}\n    (root/'spectral_parallax_report.json').write_text(json.dumps(diagnostics,indent=2))\n    print('PARALLAX REPORT',json.dumps(diagnostics),flush=True)\n    return diagnostics\n"
assert __import__('hashlib').sha256(_physical.encode()).hexdigest() == 'eeacdcf0afb173a8b438054d93c30f061df7c9c69d1ae32db3530b2916f2e6a7'
assert __import__('hashlib').sha256(_post.encode()).hexdigest() == 'cd1594b3bbfa75ce30997559c5403438b7cec011879dcb9c0d48e111715dd85d'
(ROOT/'frag_rescore.py').write_text(_physical)
(CODE/'parallax_physical.py').write_text(_post)
print('Fragment source:', __import__('hashlib').sha256(_physical.encode()).hexdigest()[:12])


In [ ]:
# ── PF-syn: our PeakFormer (bryantyu/casmi26-models pf_tpu_f.pt) joins the neural term ───────────────
# Runs between the neural scorer and the consensus (cell below): every candidate the saner model scored (final lists
# + outsiders) is scored by PF-syn in its own process (CPU) from the competition test rows of its molecule; then, per
# molecule, neural_score := (1 - PF_W) * z(saner) + PF_W * z(PF f.z)  (OURS_MODE 'replace': z(PF f.z)) in
# neural_candidate_features.parquet / outsider_neural_features.parquet, so the reciprocal-rank fusion weights, the
# fragment check and Variant M read the blended order unchanged. Any failure keeps the saner-only files (warning).
# OURS_MODE 'columns': neural_score stays byte-identical; only the audit columns neural_score_saner (= neural_score)
# and pf_fz are added, read by the final layer's PF_MID_W hook (ranks 2-25) and the ISO reranker after it.
import glob, traceback

PF_SCORES_SRC = r'''"""PF-syn PeakFormer scores for the neural-fusion candidates (bryantyu/casmi26-models pf_tpu_f.pt).

usage: python pf_scores.py <pkg_parent> <checkpoint> <test.parquet> <out.parquet> <evidence.parquet> [<evidence>...]
<pkg_parent> holds casmi_pf/ (a copy of the casmi package of bryantyu/casmi26-code). Every candidate SMILES of the
evidence files is scored against all test spectra of its molecule (rows grouped by molecule_id; spectra built as
casmi/own.py query_from_rows with instrument token 0 = unknown). Only molecules the public neural scorer scored
(processed and neural_ok) are scored. Output: molecule_id, smiles, pf_fz, pf_cos, pf_nbits, pf_ok.
"""
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


def spectra_of(g):
    out = []
    for r in g.itertuples(index=False):
        mz = np.asarray(r.ms2_mzs, np.float64)
        it = np.asarray(r.ms2_normalized_intensities, np.float64)
        ce = r.collision_energy_ev
        ce = float(np.mean(ce)) if ce is not None and len(ce) else -1.0
        out.append(dict(mz=mz, it=it, prec=float(r.precursor_mz), adduct=r.adduct, ce=ce, inst=0))
    return out


def main(pkg_parent, ckpt, test_path, out_path, *evidence):
    t0 = time.time()
    sys.path.insert(0, str(pkg_parent))
    import torch
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
    from casmi_pf.peakformer import PFEnsemble
    import casmi_pf.peakformer as pfm
    assert Path(pfm.__file__).resolve().is_relative_to(Path(pkg_parent).resolve()), pfm.__file__
    ens = PFEnsemble([str(ckpt)], device='cpu')
    sc = ens.ms[0]
    cols = ['molecule_id', 'ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz', 'adduct', 'collision_energy_ev']
    test = pd.read_parquet(test_path, columns=cols)
    test['molecule_id'] = test.molecule_id.astype(str)
    query = {m: g for m, g in test.groupby('molecule_id', sort=False)}
    want, allowed = {}, set()                   # molecule -> unique SMILES, in evidence order
    for k, path in enumerate(evidence):
        f = pd.read_parquet(path)
        if f.empty:
            continue
        f['molecule_id'] = f.molecule_id.astype(str)
        ok = f.processed.astype(bool)
        if 'neural_ok' in f.columns:
            ok &= f.neural_ok.astype(bool)
        if k == 0:                                # main evidence: only molecules the public scorer used whole
            good = ok.groupby(f.molecule_id).transform('all')
            ok &= good
            allowed = set(f.molecule_id[ok])
        else:
            ok &= f.molecule_id.isin(allowed)
        for m, s in zip(f.molecule_id[ok], f.smiles[ok].astype(str)):
            want.setdefault(m, {})[s] = None
    rows, errors = [], []
    for i, (m, cand) in enumerate(want.items()):
        smiles = list(cand)
        try:
            if m not in query:
                raise KeyError('no test rows')
            spec = spectra_of(query[m])
            if not spec:
                raise ValueError('no spectra')
            nbits = np.array([len(sc.onbits(s, s)) for s in smiles], np.int64)
            cos, fz = ens.score(spec, smiles, smiles)
            cos = np.asarray(cos, np.float64); fz = np.asarray(fz, np.float64)
            assert len(cos) == len(fz) == len(smiles)
            ok = np.isfinite(cos) & np.isfinite(fz) & (nbits > 0)
            rows += list(zip([m] * len(smiles), smiles, fz, cos, nbits, ok))
        except Exception as exc:
            errors.append(dict(molecule_id=m, error=f'{type(exc).__name__}: {str(exc)[:200]}'))
        if (i + 1) % 200 == 0:
            print('PF SCORED', i + 1, '/', len(want), 'molecules', f'{time.time() - t0:.0f}s', flush=True)
    out = pd.DataFrame(rows, columns=['molecule_id', 'smiles', 'pf_fz', 'pf_cos', 'pf_nbits', 'pf_ok'])
    out.to_parquet(out_path, index=False)
    Path(str(out_path) + '.errors.json').write_text(json.dumps(errors, indent=2))
    print('PF SCORES molecules', out.molecule_id.nunique(), 'candidates', len(out), 'not_ok', int((~out.pf_ok).sum()),
          'failures', len(errors), f'{time.time() - t0:.0f}s', flush=True)


if __name__ == '__main__':
    main(*sys.argv[1:])
'''


def pf_blend_frames(main, outs, pf, w=0.5, mode='blend'):
    """Neural term -> per-molecule blend of z(saner neural_score) and z(PF-syn f.z).

    main: neural_candidate_features (final lists); outs: outsider_neural_features or None; pf: molecule_id, smiles,
    pf_fz, pf_ok. A molecule is blended only when the public scorer used it whole (processed, neural_ok) and every
    listed candidate has a PF score; otherwise all its rows keep the saner score. The z-scores use the mean / std of
    the listed candidates; outsiders of a blended molecule get the same affine maps (so the tail rescue and the
    fragment check compare them on the listed scale); an outsider without a PF score keeps its saner z.
    mode 'columns': neural_score is left exactly as it is; only neural_score_saner and pf_fz are added.
    Returns (new_main, new_outs, stats); neural_score_saner and pf_fz are added for the audit."""
    assert mode in ('blend', 'replace', 'columns'), mode
    w = float(w)
    assert 0.0 <= w <= 1.0, w
    pfk = {(str(m), str(s)): float(v) for m, s, v, ok in zip(pf.molecule_id, pf.smiles, pf.pf_fz, pf.pf_ok)
           if bool(ok) and np.isfinite(float(v))}
    stats = dict(molecules_total=0, blended=0, skipped_no_pf=0, inactive=0, candidates=0, outsiders=0, top1_changed=0,
                 rho=[])

    def mix(zs, zp):
        return zp if mode == 'replace' else (1.0 - w) * zs + w * zp

    def zmap(v):
        mu, sd = float(np.mean(v)), float(np.std(v))
        return (lambda x: (np.asarray(x, float) - mu) / sd) if sd > 1e-12 else (lambda x: np.zeros(len(x)))

    main = main.copy()
    main['neural_score_saner'] = main.neural_score.astype(float)
    if mode != 'columns':     # float64 before the blend: the public scorer writes float32 when every molecule is
        main['neural_score'] = main.neural_score.astype(float)   # processed, and pandas >= 3 refuses float64 in it
    main['pf_fz'] = [pfk.get((str(m), str(s)), np.nan) for m, s in zip(main.molecule_id, main.smiles)]
    maps = {}
    for mid, g in main.groupby('molecule_id', sort=False):
        stats['molecules_total'] += 1
        if not (bool(g.processed.astype(bool).all()) and bool(g.neural_ok.astype(bool).all())):
            stats['inactive'] += 1
            continue
        p = g.pf_fz.to_numpy(float)
        if not np.isfinite(p).all():
            stats['skipped_no_pf'] += 1
            continue
        s = g.neural_score_saner.to_numpy(float)
        stats['blended'] += 1
        stats['candidates'] += len(g)
        if mode != 'columns':                     # 'columns': neural_score untouched, audit columns only
            zs_of, zp_of = zmap(s), zmap(p)
            new = mix(zs_of(s), zp_of(p))
            assert np.isfinite(new).all(), mid
            main.loc[g.index, 'neural_score'] = new
            maps[str(mid)] = (zs_of, zp_of)
            stats['top1_changed'] += int(np.argmax(new) != np.argmax(s))
        if len(g) > 2:
            rho = pd.Series(s).rank().corr(pd.Series(p).rank())
            if np.isfinite(rho):
                stats['rho'].append(float(rho))
    if outs is not None and len(outs):
        outs = outs.copy()
        outs['neural_score_saner'] = outs.neural_score.astype(float)
        if mode != 'columns':
            outs['neural_score'] = outs.neural_score.astype(float)
        outs['pf_fz'] = [pfk.get((str(m), str(s)), np.nan) for m, s in zip(outs.molecule_id, outs.smiles)]
        for mid, g in outs.groupby('molecule_id', sort=False):
            if str(mid) not in maps:
                continue
            g = g[g.neural_ok.astype(bool)]
            if not len(g):
                continue
            zs_of, zp_of = maps[str(mid)]
            zs = zs_of(g.neural_score_saner.to_numpy(float))
            p = g.pf_fz.to_numpy(float)
            zp = np.where(np.isfinite(p), zp_of(np.nan_to_num(p)), zs)
            new = mix(zs, zp)
            assert np.isfinite(new).all(), mid
            outs.loc[g.index, 'neural_score'] = new
            stats['outsiders'] += len(g)
    stats['rho'] = round(float(np.mean(stats['rho'])), 3) if stats['rho'] else None
    stats['pf_rows'] = sum(int(np.isfinite(f.pf_fz.to_numpy(float)).sum()) for f in (main, outs)
                           if f is not None and 'pf_fz' in f.columns)
    return main, outs, stats


def _first_input(pattern, prefer, ok=lambda p: True):
    """First /kaggle/input match of pattern that passes ok, paths containing prefer first, then shortest."""
    hits = sorted(glob.glob(str(INPUT / '**' / pattern), recursive=True), key=lambda p: (prefer not in p, len(p)))
    for h in hits:
        try:
            if ok(Path(h)):
                return Path(h)
        except Exception:
            pass
    raise FileNotFoundError(f'{pattern} ({prefer})')


def _is_pf_code(p):
    return (p.parent / 'richfp.py').is_file() and 'class PFEnsemble' in p.read_text()


def ours_pf_blend():
    """Score the neural-fusion candidates with PF-syn and blend into neural_score in place (OURS_MODE 'columns': add
    the audit columns only, neural_score byte-identical); never fatal."""
    t0 = time.time()
    rep = dict(status='off', mode=OURS_MODE, w=PF_W, molecules=0, candidates=0, failures=0, pandas=pd.__version__)
    if not OURS:
        print(f'PF-SYN off (OURS = False): saner-only neural term | pandas {pd.__version__}', flush=True)
        return rep
    main_p = ROOT / 'neural_candidate_features.parquet'
    out_p = ROOT / 'outsider_neural_features.parquet'
    paths = [main_p] + ([out_p] if OUTSIDERS_OK and out_p.is_file() else [])
    replaced, bak = [], ROOT / 'pf_backup'          # bak: the saner-only evidence files, kept for the audit
    try:
        assert OURS_MODE in ('blend', 'replace', 'columns'), f'OURS_MODE {OURS_MODE!r}'
        assert 0.0 <= float(PF_W) <= 1.0, f'PF_W {PF_W!r}'
        budget = min(float(PF_MAX_SECONDS), 8.0 * 3600 - (time.time() - NB_T0))
        if budget < 120:
            raise TimeoutError(f'only {budget:.0f} s left before the 8.0 h guard')
        src = _first_input('peakformer.py', 'casmi26-code', _is_pf_code).parent      # the casmi package
        ckpt = _first_input('pf_tpu_f.pt', 'casmi26-models')
        pkg = ROOT / 'pf_code'
        if (pkg / 'casmi_pf').exists():
            shutil.rmtree(pkg / 'casmi_pf')
        shutil.copytree(src, pkg / 'casmi_pf', ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
        (pkg / 'pf_scores.py').write_text(PF_SCORES_SRC)
        out = ROOT / 'pf_scores.parquet'
        if out.exists():
            out.unlink()
        rep.update(code=str(src), checkpoint=str(ckpt),
                   checkpoint_sha256=hashlib.sha256(ckpt.read_bytes()).hexdigest()[:16],
                   peakformer_sha256=hashlib.sha256((src / 'peakformer.py').read_bytes()).hexdigest()[:16])
        with open(ROOT / 'pf_scores.log', 'w') as log:
            proc = subprocess.run([sys.executable, pkg / 'pf_scores.py', pkg, ckpt, COMP / 'test.parquet', out, *paths],
                                  cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, timeout=budget,
                                  env={**os.environ, 'PYTHONHASHSEED': '0', 'PYTHONUNBUFFERED': '1'})
        tail = (ROOT / 'pf_scores.log').read_text()[-1500:]
        if proc.returncode:
            raise RuntimeError(f'pf_scores.py exit {proc.returncode}: {tail}')
        print('      ' + tail.strip().splitlines()[-1][:180], flush=True)
        pf = pd.read_parquet(out)
        rep['failures'] = len(json.loads(Path(str(out) + '.errors.json').read_text()))
        main = pd.read_parquet(main_p)
        outs = pd.read_parquet(out_p) if len(paths) > 1 else None
        new_main, new_outs, st = pf_blend_frames(main, outs, pf, PF_W, OURS_MODE)
        for old, new in ((main, new_main), (outs, new_outs)):
            if old is None:
                continue
            assert len(new) == len(old) and new.molecule_id.tolist() == old.molecule_id.tolist() \
                and new.smiles.tolist() == old.smiles.tolist(), 'row order changed'
            assert np.isfinite(new.neural_score.astype(float)).all(), 'non-finite neural score'
            if OURS_MODE == 'columns':            # every column read, neural_score included, exactly as it was
                assert all(new[c].equals(old[c]) for c in old.columns) and \
                    new.neural_score.to_numpy().tobytes() == old.neural_score.to_numpy().tobytes(), 'a column changed'
        if st['blended'] == 0:
            raise RuntimeError(f'no molecule blended: {st}')
        summary = dict(st, status='applied', molecules=st['blended'], candidates=st['candidates'] + st['outsiders'])
        writes = [(main_p, new_main)] + ([(out_p, new_outs)] if outs is not None else [])
        bak.mkdir(exist_ok=True)
        for path, frame in writes:
            shutil.copyfile(path, bak / path.name)
            frame.to_parquet(path.with_name(path.name + '.pf.tmp'), index=False)
        if OURS_MODE == 'columns':                # the written neural_score reads back byte for byte
            for path, _ in writes:
                a = pd.read_parquet(bak / path.name, columns=['neural_score']).neural_score
                b = pd.read_parquet(path.with_name(path.name + '.pf.tmp'), columns=['neural_score']).neural_score
                assert a.dtype == b.dtype and a.to_numpy().tobytes() == b.to_numpy().tobytes(), path.name
        for path, _ in writes:
            os.replace(path.with_name(path.name + '.pf.tmp'), path)
            replaced.append(path)
        rep.update(summary)
    except Exception as exc:
        for path in replaced:                      # all-or-nothing: put the saner files back
            try:
                shutil.copyfile(bak / path.name, path)
            except Exception:
                traceback.print_exc()
        rep.update(status='fallback_saner', error=f'{type(exc).__name__}: {str(exc)[:400]}')
        print(f'WARNING PF-SYN failed, saner-only neural term kept: {rep["error"]}', flush=True)
    rep['seconds'] = round(time.time() - t0, 1)
    try:
        (ROOT / 'pf_blend_report.json').write_text(json.dumps(rep, indent=2, default=str))
    except Exception:
        pass
    print(f"PF-SYN {rep['status']} | mode {OURS_MODE} w {PF_W} | molecules scored {rep['molecules']} | "
          f"candidates scored {rep['candidates']} | outsiders {rep.get('outsiders', 0)} | "
          f"neural top1 changed {rep.get('top1_changed', 0)} | rank corr {rep.get('rho')} | "
          f"skipped {rep.get('skipped_no_pf', 0)} | failures {rep['failures']} | {rep['seconds']} s | "
          f"pandas {pd.__version__}", flush=True)
    return rep


In [ ]:
BUNDLE = materialise_sources()
PIPELINE_DONE = run_public_pipeline()
if not PIPELINE_DONE and not IS_RERUN and COMMIT_MODE == 'fast':
    shutil.copyfile(COMP / 'sample_submission.csv', ROOT / 'submission.csv')
    raise SystemExit('fast commit: sample file written, the scoring rerun runs every stage')
if not PIPELINE_DONE:
    raise RuntimeError('A real baseline run is required; no placeholder-only submission')
NEURAL_OK = score_final_lists()
if not NEURAL_OK:
    raise RuntimeError('The 0.468 V2 neural checkpoint is required; refusing silent 0.465 fallback')
OUTSIDERS_OK = score_outside_candidates(needed=True)
try:
    PF_REPORT = ours_pf_blend()          # PF-syn joins the neural term; saner-only on any failure
except Exception as _pf_exc:
    print('WARNING PF-SYN wrapper error, saner-only neural term kept:', repr(_pf_exc)[:300], flush=True)
    PF_REPORT = {'status': 'wrapper_error', 'error': repr(_pf_exc)[:300]}
CONSENSUS_OK, REPORT, MOVES = build_consensus()
if not CONSENSUS_OK:
    raise RuntimeError('Unable to reproduce the V2 control')
SUBMITTED = choose_and_submit()
if SUBMITTED != VARIANT:
    raise RuntimeError(f'Expected {VARIANT}, got {SUBMITTED}')
shutil.copy2(ROOT/'submission.csv', ROOT/'submission_control_V2.csv')
print('Exact V2 control:', hashlib.sha256((ROOT/'submission_control_V2.csv').read_bytes()).hexdigest()[:16])


In [ ]:
import sys, traceback
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(CODE))
from parallax_physical import apply
try:
    _remaining = max(0, int(8.55 * 3600 - (time.time() - NB_T0)))
    if not PHYSICAL:
        PARALLAX_DIAGNOSTICS = {'status': 'disabled in this experiment'}
    elif _remaining < 180:
        PARALLAX_DIAGNOSTICS = {'status':'skipped_due_to_runtime_budget','remaining_seconds':_remaining,'control_retained':True}
    else:
        PARALLAX_DIAGNOSTICS = apply(ROOT, COMP, max_seconds=min(PHYSICAL_POST_SECONDS, _remaining - 120), allow_tail=True, allow_top=True)
except Exception:
    traceback.print_exc()
    shutil.copy2(ROOT/'submission_control_V2.csv',ROOT/'submission.csv')
    PARALLAX_DIAGNOSTICS = {'error':'physical_overlay_failed', 'control_restored':True}
    (ROOT/'spectral_parallax_report.json').write_text(json.dumps(PARALLAX_DIAGNOSTICS,indent=2))
assert (ROOT/'submission.csv').is_file()
final = pd.read_csv(ROOT/'submission.csv',dtype=str,keep_default_na=False)
assert final.molecule_id.tolist() == SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.notna().all() and final.smiles.str.split(';').map(len).between(1,25).all()
assert final.smiles.str.split(';').map(lambda a: all(x.strip() for x in a)).all()
print('SUBMISSION READY:',ROOT/'submission.csv','rows',len(final),'control',SUBMITTED)
print('EXPERIMENT:', json.dumps(PARALLAX_DIAGNOSTICS))


In [ ]:
from pathlib import Path
import hashlib
_PUBLIC_RERANKER = "from __future__ import annotations\nimport json\nimport math\nimport time\nimport hashlib\nimport shutil\nfrom pathlib import Path\nfrom functools import lru_cache\n\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem\nfrom rdkit.Chem import rdMolDescriptors\nimport frag_rescore as frag\n\n@lru_cache(maxsize=40000)\ndef formula(s):\n    m = Chem.MolFromSmiles(str(s))\n    return rdMolDescriptors.CalcMolFormula(m) if m is not None else None\n\ndef number(x, default=float('nan')):\n    try:\n        v=float(x)\n        return v if math.isfinite(v) else default\n    except Exception:\n        return default\n\ndef member_keys(comp):\n    frame=pd.read_parquet(Path(comp)/'train.parquet',columns=['inchikey14'])\n    return set(frame.inchikey14.dropna().astype(str))\n\ndef balanced_spectra(query, max_groups=12):\n    groups={}\n    for r in query.itertuples(index=False):\n        mz=np.asarray(r.ms2_mzs, dtype=np.float64).ravel()\n        it=np.asarray(r.ms2_normalized_intensities, dtype=np.float64).ravel()\n        n=min(len(mz),len(it));mz,it=mz[:n],it[:n]\n        ok=np.isfinite(mz)&np.isfinite(it)&(it>0)\n        if ok.sum()<6:continue\n        order=np.argsort(it[ok])[-120:]\n        mz,it=mz[ok][order],it[ok][order]\n        energy=number(getattr(r,'collision_energy_ev',30.),30.)\n        adduct=str(getattr(r,'adduct','[M+H]+'))\n        mode=1 if str(getattr(r,'ionization_mode','positive')).lower()=='positive' else -1\n        spec={'mz':mz,'it':it,'prec':number(getattr(r,'precursor_mz',None),float(mz.max())),\n              'adduct':adduct,'mode':mode}\n        bucket=(adduct,int(round(energy/10.)))\n        groups.setdefault(bucket,[]).append((int(ok.sum()), spec))\n    chosen=[]\n    for group in sorted(groups):\n        chosen.extend(s for _,s in sorted(groups[group],key=lambda x:-x[0])[:2])\n    return chosen[:max_groups]\n\ndef evidence_rows(df, order):\n    data=df.drop_duplicates('smiles').set_index('smiles',drop=False)\n    if any(s not in data.index for s in order):\n        return None\n    if ('neural_ok' in data and not data.loc[order,'neural_ok'].astype(bool).all()):\n        return None\n    if ('processed' in data and not data.loc[order,'processed'].astype(bool).all()):\n        return None\n    return data\n\ndef source_votes(candidate, leader):\n    good=0;bad=0\n    for label in ('engine2_rank','pubchem_rank','main_rank'):\n        c=number(candidate.get(label),0.);l=number(leader.get(label),0.)\n        if 0<c<=6:good+=1\n        if 0<l<=4 and (c<=0 or c>12):bad+=1\n    sim=0\n    for label in ('ice','gl'):\n        c=number(candidate.get(label));l=number(leader.get(label))\n        if math.isfinite(c) and math.isfinite(l):\n            if c>=l-.025:sim+=1\n            elif c<l-.30:sim-=1\n    return good,bad,sim\n\ndef paired_scores(spectra, smiles, cache):\n    matrix=[];matrix_old=[]\n    for spectrum in spectra:\n        vals=frag.score_molecule([spectrum],smiles,cache=cache)\n        old=frag.score_molecule([spectrum],smiles,cfg=frag.BASELINE_CFG,cache=cache)\n        if any(x is None or not math.isfinite(float(x)) for x in vals+old):return None\n        matrix.append(vals)\n        matrix_old.append(old)\n    return np.asarray(matrix,dtype=float),np.asarray(matrix_old,dtype=float)\n\ndef propose(order, original, data, query, train, cache):\n    record={'action':'none','top1_changed':False,'swaps':0,'n_spectra':0,\n            'protected_train':False,'physical_gain':None,'neural_gain_sd':None}\n    if len(order)<4 or not order or original[0]!=order[0]:\n        record['action']='protected_previous_top1';return order,record\n    evid=evidence_rows(data,order)\n    if evid is None:\n        record['action']='incomplete_evidence';return order,record\n    keys=[str(evid.loc[s,'metric_key']) if 'metric_key' in evid.columns else '' for s in order]\n    trained=np.asarray([k in train for k in keys],dtype=bool)\n    record['protected_train']=bool(trained.any())\n    leader=order[0]\n    if formula(leader) is None:\n        return order,record\n    lib=number(evid.loc[leader].get('library_max'),0.)\n    if lib>=.90:\n        record['action']='library_protected';return order,record\n    selected=balanced_spectra(query)\n    record['n_spectra']=len(selected)\n    if len(selected)<3:\n        record['action']='insufficient_spectra';return order,record\n    head=order[:min(10,len(order))]\n    p=paired_scores(selected,head,cache)\n    if p is None:\n        record['action']='fragment_unavailable';return order,record\n    a,b=p\n    neu=np.asarray([number(evid.loc[s].get('neural_score')) for s in order],float)\n    if not np.isfinite(neu).all():\n        record['action']='neural_unavailable';return order,record\n    ns=max(float(neu.std()),.00001)\n    updated=list(order)\n    candidates=[]\n    if not trained[0]:\n        for i in range(1,min(6,len(head))):\n            if trained[i] or formula(head[i])!=formula(leader):continue\n            new_margin=a[:,i]-a[:,0]\n            old_margin=b[:,i]-b[:,0]\n            mean=float(np.mean(new_margin));med=float(np.median(new_margin))\n            low=float(np.quantile(new_margin,.25));votes=float((new_margin>.020).mean())\n            old=float(np.mean(old_margin))\n            nz=float((neu[i]-neu[0])/ns)\n            if mean<.095 or med<.070 or low<-.010 or votes<.75 or old<.022 or nz<.18:continue\n            ev=evid.loc[head[i]];lv=evid.loc[leader]\n            if number(ev.get('neural_cos'),0.)<number(lv.get('neural_cos'),0.)+.002:continue\n            if number(ev.get('neural_fp'),0.)<number(lv.get('neural_fp'),0.)-.010:continue\n            votes_source,bad,sim=source_votes(ev,lv)\n            if votes_source<1 and sim<1:continue\n            if bad>=2 or sim<0:continue\n            confidence=mean+.045*nz+.020*sim+.010*votes_source+.018*max(0,low)\n            candidates.append((confidence,i,mean,nz))\n        if candidates:\n            _,idx,gain,nz=max(candidates)\n            updated.insert(0,updated.pop(idx))\n            record.update(action='provenance_spectral_top1',top1_changed=True,\n                          physical_gain=round(gain,5),neural_gain_sd=round(nz,4))\n    swaps=0\n    for pos in range(1,min(8,len(updated)-1)):\n        before=updated[pos];after=updated[pos+1]\n        if formula(before)!=formula(after):continue\n        i,j=head.index(before) if before in head else -1, head.index(after) if after in head else -1\n        if i<0 or j<0:continue\n        if trained[i] or trained[j]:continue\n        d=a[:,j]-a[:,i];do=b[:,j]-b[:,i]\n        if float(np.mean(d))<.068 or float(np.median(d))<.045 or float((d>.010).mean())<.67:continue\n        if float(np.mean(do))<.015:continue\n        if neu[j]-neu[i]<-.04*ns:continue\n        c=source_votes(evid.loc[after],evid.loc[before])[0]\n        if c<1 and neu[j]<=neu[i]:continue\n        updated[pos],updated[pos+1]=updated[pos+1],updated[pos]\n        swaps+=1\n        if swaps>=2:break\n    record['swaps']=swaps\n    if swaps and not record['top1_changed']:record['action']='paired_energy_swap'\n    # The private 0.475 model freezes known training candidates. This public analogue\n    # only allows permutations among untrained structures, preserving every training slot.\n    for i,s in enumerate(order):\n        if trained[i] and updated[i]!=s:\n            return order,dict(record,action='training_slot_guard',top1_changed=False,swaps=0)\n    return updated,record\n\ndef run(root,comp,time_budget=900):\n    root=Path(root);comp=Path(comp)\n    path=root/'submission.csv'\n    control=root/'submission_control_V2.csv'\n    for req in [path,control,root/'neural_candidate_features.parquet',root/'effective_query_spectra.parquet',comp/'train.parquet']:\n        if not req.exists():raise FileNotFoundError(req)\n    existing=pd.read_csv(path,dtype=str,keep_default_na=False)\n    first=pd.read_csv(control,dtype=str,keep_default_na=False)\n    assert existing.molecule_id.tolist()==first.molecule_id.tolist()\n    assert existing.molecule_id.is_unique\n    shutil.copy2(path,root/'submission_physical0472_control.csv')\n    nfeatures=pd.read_parquet(root/'neural_candidate_features.parquet')\n    spectra=pd.read_parquet(root/'effective_query_spectra.parquet')\n    groups={str(m):df for m,df in nfeatures.groupby('molecule_id',sort=False)}\n    specgroups={str(m):df for m,df in spectra.groupby('molecule_id',sort=False)}\n    known=member_keys(comp)\n    frag.warmup()\n    t0=time.monotonic();cache={};newrows=[];audits=[]\n    for idx,(record,ctrl) in enumerate(zip(existing.itertuples(index=False),first.itertuples(index=False))):\n        mid=str(record.molecule_id)\n        arr=str(record.smiles).split(';');original=str(ctrl.smiles).split(';')\n        if time.monotonic()-t0>time_budget:\n            new=arr;meta={'action':'runtime_budget','top1_changed':False,'swaps':0}\n        elif mid not in groups or mid not in specgroups:\n            new=arr;meta={'action':'no_source_spectra','top1_changed':False,'swaps':0}\n        else:\n            try:\n                new,meta=propose(arr,original,groups[mid],specgroups[mid],known,cache)\n            except Exception as e:\n                new=arr;meta={'action':'failed_'+type(e).__name__,'top1_changed':False,'swaps':0}\n        if set(new)!=set(arr) or len(new)!=len(arr):\n            raise AssertionError('Candidate identities may not change')\n        newrows.append((mid,';'.join(new)))\n        audits.append({'molecule_id':mid,**meta})\n        if len(cache)>25000:cache.clear()\n        if (idx+1)%50==0:\n            print('PROVENANCE',idx+1,'/',len(existing),'elapsed',round(time.monotonic()-t0,1),'s',flush=True)\n    result=pd.DataFrame(newrows,columns=['molecule_id','smiles'])\n    assert result.molecule_id.tolist()==existing.molecule_id.tolist()\n    assert result.molecule_id.is_unique\n    assert result.smiles.str.split(';').map(len).between(1,25).all()\n    assert result.smiles.str.split(';').map(lambda a:len(set(a))==len(a) and all(a)).all()\n    temp=root/'submission.provenance.tmp.csv'\n    result.to_csv(temp,index=False)\n    assert pd.read_csv(temp,dtype=str,keep_default_na=False).equals(result)\n    temp.replace(path)\n    report=pd.DataFrame(audits)\n    report.to_csv(root/'provenance_tournament_audit.csv',index=False)\n    summary={'source':'0.472 physical consensus public pipeline',\n             'private_rr3_weights_used':False,'train_structure_keys':len(known),\n             'rows':len(result),'changed':int((existing.smiles!=result.smiles).sum()),\n             'top1_changed':int(report.top1_changed.fillna(False).astype(bool).sum()),\n             'adjacent_swaps':int(report.swaps.fillna(0).sum()),\n             'elapsed_seconds':round(time.monotonic()-t0,2),\n             'control_hash':hashlib.sha256((root/'submission_physical0472_control.csv').read_bytes()).hexdigest(),\n             'final_hash':hashlib.sha256(path.read_bytes()).hexdigest(),\n             'note':'Leaderboard improvement not validated'}\n    (root/'provenance_tournament_report.json').write_text(json.dumps(summary,indent=2))\n    print('PROVENANCE_REPORT',json.dumps(summary),flush=True)\n    return summary\n"
PROVENANCE_CODE = ROOT/'parallax_public_provenance.py'
PROVENANCE_CODE.write_text(_PUBLIC_RERANKER)
assert hashlib.sha256(PROVENANCE_CODE.read_bytes()).hexdigest() == hashlib.sha256(_PUBLIC_RERANKER.encode()).hexdigest()
print('Public provenance-aware reranker:',PROVENANCE_CODE)

In [ ]:
import traceback
try:
    import importlib.util
    spec=importlib.util.spec_from_file_location('parallax_public_provenance',PROVENANCE_CODE)
    module=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    _left=max(0,int(8.65*3600-(time.time()-NB_T0)))
    if not PROVENANCE:
        PROVENANCE_REPORT={'status':'disabled in this experiment'}
    elif _left<240:
        PROVENANCE_REPORT={'status':'skipped_because_of_kaggle_runtime','seconds_remaining':_left}
    else:
        PROVENANCE_REPORT=module.run(ROOT,COMP,time_budget=min(960,_left-180))
except Exception as exc:
    traceback.print_exc()
    shutil.copy2(ROOT/'submission_physical0472_control.csv' if (ROOT/'submission_physical0472_control.csv').is_file() else ROOT/'submission_control_V2.csv',ROOT/'submission.csv')
    PROVENANCE_REPORT={'status':'restored_control_after_error','exception':repr(exc)}

final=pd.read_csv(ROOT/'submission.csv',dtype=str,keep_default_na=False)
assert list(final.columns)==['molecule_id','smiles']
assert final.molecule_id.tolist()==SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.str.split(';').map(len).between(1,25).all()
assert final.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a))==len(a)).all()
print('SUBMISSION READY:',ROOT/'submission.csv','molecules',len(final),'provenance',PROVENANCE_REPORT)


In [ ]:
LIB_MAX = {}
# ── Variant M: rank 1 from fragment check on V2, ranks 2-25 from CUSTOM + DocPrior + In-Pool Dedup ────────
# Cell 15 of baidalinadilzhan/casmi26-midx-exact-dedup-fill (team LB 0.479) unchanged, its configuration lines in
# cell 1, plus one hook of ours: _mid_neural_score (PF_MID_W > 0: PF-syn joins the ranks 2-25 neural rank).
import os, json, shutil, hashlib, traceback, glob
from pathlib import Path
from functools import lru_cache
import pandas as pd
import numpy as np


def _load_pop_prior(root, input_root):
    """Load PubChem documentation prior and spectral annotations for InChIKey14 lookups."""
    pop_map, spec_map, np_map = {}, {}, {}
    try:
        hits = sorted(glob.glob(str(input_root / '**/pool_popularity.csv'), recursive=True))
        if hits:
            # elig0 (ours, F3): the NIST-including depositor flag src_spectral_library is loaded only when SPEC_W > 0
            cols = ['inchikey14', 'pop'] + (['src_spectral_library'] if SPEC_W else []) + \
                   ['np_coconut', 'np_lotus', 'src_natural_product']
            p_df = pd.read_csv(hits[0], usecols=cols)
            pop_map = dict(zip(p_df['inchikey14'], p_df['pop']))
            spec_map = dict(zip(p_df['inchikey14'], p_df['src_spectral_library'])) if SPEC_W else {}
            if not SPEC_W:
                print('elig0 F3: src_spectral_library not loaded (SPEC_W 0)', flush=True)
            np_map = dict(zip(p_df['inchikey14'], (p_df['np_coconut'] | p_df['np_lotus'] | p_df['src_natural_product']).astype(int)))
            print(f'Documentation Prior: loaded {len(p_df)} pool records from {Path(hits[0]).name}')
    except Exception as e:
        print(f'Note: pool_popularity lookup exception ({e}); proceeding with neutral prior.')
    return pop_map, spec_map, np_map


def _mid_neural_score(g):
    """Score that sets the neural rank nr of ranks 2-25. PF_MID_W = 0 (default): the column neural_score, as midx.
    PF_MID_W > 0: (1 - w) z(saner) + w z(PF-syn fz) per molecule, from the columns neural_score_saner (else
    neural_score) and pf_fz; any missing / non-finite PF score keeps the plain neural_score for that molecule."""
    base = g['neural_score'].to_numpy(float)
    w = float(globals().get('PF_MID_W', 0.0) or 0.0)
    if w <= 0 or 'pf_fz' not in g.columns or len(g) < 2:
        return base
    s = (g['neural_score_saner'] if 'neural_score_saner' in g.columns else g['neural_score']).to_numpy(float)
    p = g['pf_fz'].to_numpy(float)
    if not (np.isfinite(s).all() and np.isfinite(p).all()):
        return base
    zs = (s - s.mean()) / s.std() if s.std() > 1e-12 else np.zeros_like(s)
    zp = (p - p.mean()) / p.std() if p.std() > 1e-12 else np.zeros_like(p)
    return (1.0 - w) * zs + w * zp


def pxm_merge(physical, v2, custom, pop_map=None, spec_map=None, np_map=None, nf_evidence=None):
    """physical, v2, custom: molecule_id, smiles frames in the same row order. Returns (frame, counts)."""
    assert list(physical.columns) == list(v2.columns) == list(custom.columns) == ['molecule_id', 'smiles']
    assert physical.molecule_id.tolist() == v2.molecule_id.tolist() == custom.molecule_id.tolist(), 'row order differs'
    rows = []
    counts = dict(rows=0, merged=0, leader_moves=0, ranks_reordered=0, skipped={}, skipped_leader_moves=0, moved_ids=[])
    
    pop_map = pop_map or {}
    spec_map = spec_map or {}
    np_map = np_map or {}
    evidence_by_mid = nf_evidence or {}

    for mid, ps, vs, cs in zip(custom.molecule_id, physical.smiles, v2.smiles, custom.smiles):
        p, v, c = ps.split(';'), vs.split(';'), cs.split(';')
        counts['rows'] += 1
        
        # Variant M Leader Rule (Strictly Locked Rank 1):
        # If the fragment check on V2 identified a superior physical bond cleavage leader (p[0] != v[0]):
        # Place p[0] at Rank 1, followed by the high-resolution CUSTOM order.
        if p[0] != v[0]:
            leader = p[0]
            rest = [s for s in c if s != p[0]]
            counts['merged'] += 1
            counts['leader_moves'] += 1
            counts['moved_ids'].append(mid)
        else:
            leader = c[0]
            rest = c[1:]
            counts['merged'] += 1

        # Re-rank Ranks 2–25 by Documentation Prior + Spectral Library + Natural Product Bonus
        cinfo = evidence_by_mid.get(str(mid), {})
        lib_hit = float(LIB_MAX.get(str(mid), 0.0)) >= LIB_HIT_GATE
        if rest and cinfo:
            def _score_cand(s):
                if s in cinfo:
                    br, nr, k14 = cinfo[s]
                    p_val = float(pop_map.get(k14, 0.0))
                    s_val = float(spec_map.get(k14, 0.0))
                    np_val = float(np_map.get(k14, 0.0))
                    return 1.0 / (3.0 + br) + 0.75 / (3.0 + nr) + POP_W * (p_val / 20.0) + (SPEC_W if lib_hit else 0.0) * s_val + NP_W * np_val
                idx = c.index(s) if s in c else 25
                return 1.0 / (3.0 + idx + 1)
            
            old_rest = list(rest)
            rest.sort(key=_score_cand, reverse=True)
            if rest != old_rest:
                counts['ranks_reordered'] += 1
            
        if PIN_OLD_LEADER and p[0] != v[0] and v[0] in rest and rest[0] != v[0]:
            rest = [v[0]] + [x for x in rest if x != v[0]]
            counts['old_leader_pinned'] = counts.get('old_leader_pinned', 0) + 1
        rows.append((mid, [leader] + rest))
    return rows, counts


# ---- run (everything above is definitions) ----
def _pxm_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()


def _pxm_main():
    rep = dict(status='disabled')
    if not globals().get('FRAG_ON_V2', False):
        return rep
    assert not globals().get('PROVENANCE', False), 'variant M is defined without the provenance reranker'
    sub_path = ROOT / 'submission.csv'
    v_path, c_path = ROOT / 'variants/submission_V2.csv', ROOT / 'variants/submission_CUSTOM.csv'
    rep['input_sha256'] = dict(physical_v2=_pxm_sha(sub_path), v2=_pxm_sha(v_path), custom=_pxm_sha(c_path))
    diag = globals().get('PARALLAX_DIAGNOSTICS') or {}
    rep['overlay'] = {k: diag[k] for k in ('status', 'error', 'control_sha256', 'submission_sha256', 'changed_rows',
                                           'top1_changed', 'tail_changed', 'adjacent_swaps') if k in diag}
    if diag.get('control_sha256'):
        assert diag['control_sha256'] == rep['input_sha256']['v2'], 'the fragment check did not read the V2 file'
    if diag.get('submission_sha256'):
        assert diag['submission_sha256'] == rep['input_sha256']['physical_v2'], \
            'submission.csv is not the file the fragment check wrote'
    frames = [pd.read_csv(p, dtype=str, keep_default_na=False) for p in (sub_path, v_path, c_path)]
    for f in frames:
        assert list(f.columns) == ['molecule_id', 'smiles'] and f.molecule_id.tolist() == SAMPLE_IDS, \
            'an input file differs from sample_submission'

    from rdkit import Chem

    from rdkit.Chem.MolStandardize import rdMolStandardize
    _TAUT = rdMolStandardize.TautomerEnumerator()
    _IDMAP = identity_map()

    @lru_cache(maxsize=100000)
    def _to_k14(s):
        k = _IDMAP.get(s)
        if isinstance(k, str) and k:
            return k
        try:
            m = Chem.MolFromSmiles(str(s))
            if m is None:
                return ''
            Chem.RemoveStereochemistry(m)
            return Chem.MolToInchiKey(_TAUT.Canonicalize(m))[:14]
        except Exception:
            return ''

    global LIB_MAX
    LIB_MAX = {}
    out_by_mid = {}
    try:
        _nf = pd.read_parquet(ROOT / 'neural_candidate_features.parquet', columns=['molecule_id', 'library_max'])
        LIB_MAX = {str(m): float(v) for m, v in _nf.groupby('molecule_id', sort=False).library_max.max().items()}
        _op = ROOT / 'outsider_neural_features.parquet'
        if OUTSIDER_FILL and _op.is_file():
            _of = pd.read_parquet(_op)
            _sc = pd.to_numeric(_of.neural_score, errors='coerce')
            _of = _of[_of.neural_ok.astype(bool) & np.isfinite(_sc)].assign(_s=_sc)
            for _m, _g in _of.groupby('molecule_id', sort=False):
                out_by_mid[str(_m)] = _g.sort_values('_s', ascending=False, kind='stable').smiles.astype(str).tolist()
    except Exception as e:
        print(f'Note: final-layer inputs exception ({e}); gate off, no outsider fill')
    rep['lib_hit_molecules'] = int(sum(v >= LIB_HIT_GATE for v in LIB_MAX.values()))

    # Load documentation prior and candidate evidence
    pop_map, spec_map, np_map = _load_pop_prior(ROOT, INPUT)
    evidence_by_mid = {}
    nf_path = ROOT / 'neural_candidate_features.parquet'
    if nf_path.is_file():
        try:
            nf = pd.read_parquet(nf_path)
            for mid, g in nf.groupby('molecule_id', sort=False):
                order = np.argsort(-_mid_neural_score(g), kind='stable')
                nranks = np.empty(len(g), dtype=int)
                nranks[order] = np.arange(1, len(g) + 1)
                cand_info = {}
                for s, br, nr, mk in zip(g['smiles'], g['baseline_rank'], nranks, g['metric_key']):
                    k14 = mk if (isinstance(mk, str) and len(mk) == 14) else _to_k14(s)
                    cand_info[s] = (int(br), int(nr), k14)
                evidence_by_mid[str(mid)] = cand_info
        except Exception as e:
            print(f'Note: neural evidence indexing exception ({e})')

    merged_tuples, counts = pxm_merge(*frames, pop_map=pop_map, spec_map=spec_map, np_map=np_map, nf_evidence=evidence_by_mid)

    # In-Pool InChIKey14 Skeletal Deduplication
    reclaimed_smiles = []
    dedup_swaps = 0
    outsider_fills = 0
    for (mid, prim_list), c_s, v_s in zip(merged_tuples, frames[2].smiles, frames[1].smiles):
        seen_k = set()
        res = []
        # 1. Primary order (Variant M merged + DocPrior reordered ranks 2-25)
        for s in prim_list:
            if not s: continue
            k = _to_k14(s)
            if not k or k not in seen_k:
                if k: seen_k.add(k)
                res.append(s)
                if len(res) >= 25: break

        # 2. In-pool backfill strictly from c and v (no outside distractors)
        if len(res) < 25:
            for extra in (c_s, v_s):
                for s in extra.split(';'):
                    if not s or s in res: continue
                    k = _to_k14(s)
                    if not k or k not in seen_k:
                        if k: seen_k.add(k)
                        res.append(s)
                        dedup_swaps += 1
                        if len(res) >= 25: break
                if len(res) >= 25: break

        # 2b. Empty slots: scored outside candidates with a new metric identity, best neural score first
        if len(res) < 25:
            for s in out_by_mid.get(str(mid), []):
                if s in res:
                    continue
                k = _to_k14(s)
                if not k or k in seen_k:
                    continue
                seen_k.add(k)
                res.append(s)
                outsider_fills += 1
                if len(res) >= 25:
                    break

        # 3. Fallback fill if still < 25
        if len(res) < 25:
            for extra in (prim_list, c_s.split(';'), v_s.split(';')):
                for s in extra:
                    if s and s not in res:
                        res.append(s)
                        if len(res) >= 25: break
                if len(res) >= 25: break

        reclaimed_smiles.append(';'.join(res[:25]))

    new = pd.DataFrame({'molecule_id': [m for m, _ in merged_tuples], 'smiles': reclaimed_smiles})
    rep['dedup_swaps'] = dedup_swaps
    rep['outsider_fills'] = outsider_fills
    rep['old_leader_pinned'] = counts.get('old_leader_pinned', 0)
    print(f'Final layer: outsider fills {outsider_fills}, old leader pinned {rep["old_leader_pinned"]}, '
          f'library-hit molecules {rep.get("lib_hit_molecules")}, POP_W {POP_W}, NP_W {NP_W}')
    rep['ranks_reordered'] = counts['ranks_reordered']
    print(f'InChIKey14 Deduplication: reclaimed {dedup_swaps} redundant isomer slots!')
    print(f'Documentation Prior: reordered secondary ranks in {counts["ranks_reordered"]} / {counts["rows"]} rows.')

    assert new.molecule_id.tolist() == SAMPLE_IDS
    assert new.smiles.str.split(';').map(len).between(1, 25).all()
    assert new.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a)) == len(a)).all()
    tmp = ROOT / 'submission.merge.tmp'
    new.to_csv(tmp, index=False)
    out_sha = _pxm_sha(tmp)
    os.replace(tmp, sub_path)
    rep.update(status='applied', **counts, output_sha256=out_sha,
               output_equals_custom=out_sha == rep['input_sha256']['custom'])
    return rep


try:
    if not PIPELINE_DONE and not IS_RERUN and COMMIT_MODE == 'fast':
        MERGE_REPORT = {'status': 'skipped_fast_commit'}
    else:
        MERGE_REPORT = _pxm_main()
except Exception as exc:
    traceback.print_exc()
    MERGE_REPORT = dict(status='restored_custom_after_error', exception=repr(exc)[:600])
    try:
        _tmp = ROOT / 'submission.merge.tmp'
        shutil.copyfile(ROOT / 'variants/submission_CUSTOM.csv', _tmp)
        os.replace(_tmp, ROOT / 'submission.csv')
        MERGE_REPORT['output_sha256'] = _pxm_sha(ROOT / 'submission.csv')
    except Exception as exc2:
        traceback.print_exc()
        MERGE_REPORT['restore_error'] = repr(exc2)[:300]
try:
    _tmp = ROOT / 'merge_report.json.tmp'
    _tmp.write_text(json.dumps(MERGE_REPORT, indent=2, default=str))
    os.replace(_tmp, ROOT / 'merge_report.json')
except Exception:
    traceback.print_exc()

final = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
assert list(final.columns) == ['molecule_id', 'smiles']
assert final.molecule_id.tolist() == SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.str.split(';').map(len).between(1, 25).all()
assert final.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a)) == len(a)).all()
print('SUBMISSION READY:', ROOT / 'submission.csv', 'molecules', len(final), 'MERGE', json.dumps(
    {k: MERGE_REPORT.get(k) for k in ('status', 'rows', 'merged', 'leader_moves', 'ranks_reordered', 'dedup_swaps',
                                      'output_equals_custom', 'outsider_fills', 'old_leader_pinned', 'lib_hit_molecules', 'output_sha256', 'exception') if k in MERGE_REPORT},
    default=str))


In [ ]:
# ── ISO: isomer reranker on the final answer lists (ours; runs right after the final layer) ──────────────────────
# heliosli-style (team LB 0.486) with our PF-syn f.z as the model score. Per molecule, over its submitted list:
# score = z(-log rank) + ISO_ALPHA * (z(pf_fz) + ISO_COS_W * z(pf_cos))   (z over the list, population std; std 0 ->
# zeros; pf_cos = the PF-syn cosine of pf_scores.parquet, not read at all when ISO_COS_W is 0).
# ISO_SCOPE 'formula': only the candidates with the rank-1 candidate's formula are re-sorted by score, inside the positions
# they already hold; 'all': every candidate. Protected candidates never move: training structures (ISO_KEEP_TRAIN: RDKit
# InChIKey14 in the inchikey14 column of train.parquet) and SMILES RDKit cannot read. ISO_FREEZE_LEADER: rank 1 is held
# in place as well (ranks 2-25 only; scope 'formula' still re-sorts the leader's formula). Skipped molecules: best library
# match >= ISO_LIB_SKIP, a listed candidate without a finite PF-syn f.z (neural_candidate_features, then
# outsider_neural_features, then pf_scores.parquet) or, when ISO_COS_W > 0, without a finite pf_cos (pf_scores.parquet,
# pf_ok rows; counted as skipped pf), fewer than 2 candidates. ISO_ALPHA 0: submission.csv is not touched.
# Any failure restores the pre-ISO submission.csv byte for byte (WARNING). Report: iso_report.json, one 'ISO <status> | ...'
# line ('ISO applied' only when ISO ran; a fallback reports zero changes, the aborted attempt as attempted_*).
# Commit run (not IS_RERUN, ISO_ALPHA > 0): an 'ISO dryrun' line too, the same rerank with the library skip off, written to
# a scratch file that is read back and deleted (the visible molecules all have library match 1.0, so the real path skips
# them). PF_MID_W > 0: a 'PF_MID trace' line, how many molecules the final layer's hook re-ranked (pf_mid_trace.json).
import io, os, json, math, time, shutil, hashlib, traceback
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
try:
    from rdkit import Chem, rdBase
    from rdkit.Chem import rdMolDescriptors
except Exception as _iso_import_error:                          # ISO then falls back (submission.csv untouched)
    Chem = rdBase = rdMolDescriptors = None
    print('WARNING ISO: RDKit import failed:', repr(_iso_import_error)[:200], flush=True)


def iso_z(v):
    """z-scores over one list (population std); all zeros when the std is 0."""
    v = np.asarray(v, float)
    sd = float(v.std())
    return (v - float(v.mean())) / sd if sd > 1e-12 else np.zeros(len(v))


@lru_cache(maxsize=200000)
def iso_formula(s):
    """RDKit molecular formula of a SMILES; None when RDKit cannot read it."""
    try:
        m = Chem.MolFromSmiles(str(s))
        return rdMolDescriptors.CalcMolFormula(m) if m is not None else None
    except Exception:
        return None


@lru_cache(maxsize=200000)
def iso_key14(s):
    """First block of the RDKit InChIKey (the inchikey14 of train.parquet); None when RDKit cannot make one."""
    try:
        m = Chem.MolFromSmiles(str(s))
        k = Chem.MolToInchiKey(m) if m is not None else ''
        return k[:14] if len(k) >= 14 else None
    except Exception:
        return None


def iso_order(cands, pf, alpha, scope='formula', train_keys=frozenset(), keep_train=True,
              formula_of=iso_formula, key_of=iso_key14, cos=None, cos_w=0.0, freeze_leader=False):
    """New order of one answer list (pure). cands: SMILES in submitted order (>= 2); pf: finite PF-syn f.z per candidate;
    cos: finite PF-syn cosine per candidate, read only when cos_w > 0 (score = z(-log rank) + alpha * (z(pf) +
    cos_w * z(cos))). freeze_leader: position 1 is held in place like a protected candidate (scope 'formula' still uses
    its formula). Returns (new list, protected, moved): protected = in-scope candidates held in place (training structure
    or unreadable; a frozen leader is not counted), moved = size of the re-sorted group (0 when fewer than 2 candidates
    are movable)."""
    n = len(cands)
    assert n >= 2 and len(pf) == n and scope in ('formula', 'all'), (n, len(pf), scope)
    assert float(cos_w) >= 0 and isinstance(freeze_leader, bool), (cos_w, freeze_leader)
    model = iso_z(pf)
    if cos_w:                                                           # ISO_COS_W 0: cos is not looked at
        assert cos is not None and len(cos) == n, 'pf_cos per candidate'
        model = model + float(cos_w) * iso_z(cos)
    score = iso_z(-np.log(np.arange(1, n + 1, dtype=float))) + float(alpha) * model
    lead = formula_of(cands[0])
    group, protected = [], 0
    for i, s in enumerate(cands):
        f = formula_of(s)
        if scope == 'formula' and (lead is None or f != lead):
            continue
        held = f is None                                                # unreadable SMILES never move
        if not held and keep_train:
            k = key_of(s)
            held = k is None or k in train_keys
        if held:
            protected += 1
            continue
        if i == 0 and freeze_leader:                                    # ISO_FREEZE_LEADER: rank 1 stays
            continue
        group.append(i)
    new = list(cands)
    if len(group) < 2:
        return new, protected, 0
    for slot, i in zip(group, sorted(group, key=lambda j: -score[j])):      # stable: ties keep the submitted order
        new[slot] = cands[i]
    return new, protected, len(group)


def iso_rerank(lists, pf_of, lib_max, alpha, lib_skip=0.9, scope='formula', train_keys=frozenset(), keep_train=True,
               considered=None, formula_of=iso_formula, key_of=iso_key14, cos_of=None, cos_w=0.0, freeze_leader=False):
    """All answer lists (pure). lists: [(molecule_id, [SMILES, ...]), ...]; pf_of: {(molecule_id, SMILES): f.z};
    lib_max: {molecule_id: best library match}; considered: molecule ids to rerank (None = every row); cos_of:
    {(molecule_id, SMILES): pf_cos}, read only when cos_w > 0 (a listed candidate without a finite pf_cos then skips the
    molecule: skipped_pf, of which skipped_cos had every f.z). Returns (new lists in the same order, stats)."""
    assert not cos_w or cos_of is not None, 'cos_w > 0 needs cos_of'
    st = dict(considered=0, skipped_lib=0, skipped_pf=0, skipped_cos=0, skipped_short=0, reranked=0, changed_rows=0,
              top1_changed=0, protected=0, moved_candidates=0)
    finite = lambda v: all(x is not None and math.isfinite(float(x)) for x in v)
    out = []
    for mid, cands in lists:
        mid, cands, new = str(mid), list(cands), list(cands)
        if considered is None or mid in considered:
            st['considered'] += 1
            lm = lib_max.get(mid)
            pf = [pf_of.get((mid, s)) for s in cands]
            cos = [cos_of.get((mid, s)) for s in cands] if cos_w else None
            if lm is not None and math.isfinite(float(lm)) and float(lm) >= lib_skip:
                st['skipped_lib'] += 1
            elif not finite(pf) or (cos_w and not finite(cos)):
                st['skipped_pf'] += 1
                st['skipped_cos'] += int(finite(pf))
            elif len(cands) < 2:
                st['skipped_short'] += 1
            else:
                new, prot, moved = iso_order(cands, pf, alpha, scope, train_keys, keep_train, formula_of, key_of,
                                             cos, cos_w, freeze_leader)
                st['reranked'] += 1
                st['protected'] += prot
                st['moved_candidates'] += sum(a != b for a, b in zip(new, cands))
                st['changed_rows'] += int(new != cands)
                st['top1_changed'] += int(new[0] != cands[0])
        out.append((mid, new))
    return out, st


def iso_pf_lookup(frames, col='pf_fz'):
    """{(molecule_id, SMILES): PF-syn score col} (pf_fz = f.z, pf_cos = cosine) from frames with molecule_id, smiles, col
    (and pf_ok, if present); the first finite score in frame order wins (frames that are None, empty or without col are
    passed over)."""
    out = {}
    for f in frames:
        if f is None or not len(f) or col not in f.columns:
            continue
        v = pd.to_numeric(f[col], errors='coerce').to_numpy(float)
        ok = np.isfinite(v)
        if 'pf_ok' in f.columns:
            ok &= f.pf_ok.eq(True).to_numpy()
        for m, s, x in zip(f.molecule_id.astype(str).to_numpy()[ok], f.smiles.astype(str).to_numpy()[ok], v[ok]):
            out.setdefault((m, s), float(x))
    return out


def iso_train_keys(train_path):
    """InChIKey14 of every training structure (only the inchikey14 column of train.parquet is read)."""
    return frozenset(pd.read_parquet(train_path, columns=['inchikey14']).inchikey14.dropna().astype(str).unique())


def iso_check(before, after, sample_ids):
    """after is a valid answer file: sample_submission rows in order, 1-25 unique non-empty SMILES per row, and per row
    the candidate set of before."""
    assert list(after.columns) == ['molecule_id', 'smiles'], 'columns'
    assert after.molecule_id.tolist() == list(sample_ids) == before.molecule_id.tolist(), 'rows'
    a, b = after.smiles.str.split(';'), before.smiles.str.split(';')
    assert a.map(len).between(1, 25).all(), 'list length'
    assert a.map(lambda v: all(x.strip() for x in v) and len(set(v)) == len(v)).all(), 'SMILES'
    assert all(sorted(x) == sorted(y) for x, y in zip(a, b)), 'candidate set changed'


def iso_summary(rep):
    return (f"ISO {rep.get('status')} | alpha {rep.get('alpha')} | scope {rep.get('scope')} | cos w {rep.get('cos_w')} | "
            f"freeze leader {rep.get('freeze_leader')} | molecules considered "
            f"{rep.get('considered', 0)} | skipped lib {rep.get('skipped_lib', 0)} | skipped pf {rep.get('skipped_pf', 0)} | "
            f"changed rows {rep.get('changed_rows', 0)} | top1 changed {rep.get('top1_changed', 0)} | protected "
            f"{rep.get('protected', 0)} | {rep.get('seconds', 0)} s")


ISO_OUTCOME = ('reranked', 'changed_rows', 'top1_changed', 'protected', 'moved_candidates')


def iso_frame(lists):
    """Answer frame (molecule_id, smiles) of [(molecule_id, [SMILES, ...]), ...]."""
    return pd.DataFrame({'molecule_id': [m for m, _ in lists], 'smiles': [';'.join(v) for _, v in lists]})


def iso_write_checked(frame, tmp):
    """Write frame as CSV to tmp and check that it reads back equal."""
    frame.to_csv(tmp, index=False)
    assert pd.read_csv(tmp, dtype=str, keep_default_na=False).equals(frame), 'CSV roundtrip'


def iso_dryrun(sub, lists, sub_path, sample_ids, pf_of, lib_max, alpha, scope, train_keys, keep_train, considered,
               formula_of=iso_formula, key_of=iso_key14, cos_of=None, cos_w=0.0, freeze_leader=False):
    """Check of the rerank path the commit run cannot reach (its visible molecules all have library match 1.0): the same
    rerank with the library skip off, validated, written to a scratch CSV that is read back, moved with os.replace and
    deleted. submission.csv is not touched; never raises. Returns its counts (status 'ok' or 'failed')."""
    tmp, dst = Path(sub_path).with_name('iso_dryrun.csv.tmp'), Path(sub_path).with_name('iso_dryrun.csv')
    try:
        new_lists, st = iso_rerank(lists, pf_of, lib_max, alpha, math.inf, scope, train_keys, keep_train, considered,
                                   formula_of, key_of, cos_of, cos_w, freeze_leader)
        new = iso_frame(new_lists)
        iso_check(sub, new, sample_ids)
        iso_write_checked(new, tmp)
        data = tmp.read_bytes()
        os.replace(tmp, dst)
        assert dst.read_bytes() == data, 'dry-run file changed in the move'
        out = dict(status='ok', **{k: st[k] for k in ('considered', 'skipped_lib', 'skipped_pf', 'skipped_cos',
                                                      'skipped_short') + ISO_OUTCOME},
                   changed_ids=[m for (m, a), (_, b) in zip(lists, new_lists) if a != b][:50])
    except Exception as exc:
        traceback.print_exc()
        out = dict(status='failed', error=f'{type(exc).__name__}: {str(exc)[:400]}')
    finally:
        for p in (tmp, dst):
            try:
                p.unlink()
            except FileNotFoundError:
                pass
    if out['status'] == 'ok':
        print(f"ISO dryrun ok | library skip off | molecules considered {out['considered']} | skipped pf {out['skipped_pf']} | "
              f"reranked {out['reranked']} | changed rows {out['changed_rows']} | top1 changed {out['top1_changed']} | "
              f"protected {out['protected']} | moved candidates {out['moved_candidates']}", flush=True)
    else:
        print(f"WARNING ISO dryrun failed (submission.csv not touched): {out['error']}", flush=True)
    return out


def iso_pf_mid_trace(nf, score_fn):
    """Trace of the final layer's PF_MID_W hook (diagnostic only): per molecule of neural_candidate_features, whether
    score_fn (the hook) gave a score other than neural_score (PF used) and a different stable neural order."""
    st = dict(molecules=0, pf_used=0, order_changed=0)
    for _, g in nf.groupby('molecule_id', sort=False):
        base = g['neural_score'].to_numpy(float)
        sc = np.asarray(score_fn(g), float)
        assert sc.shape == base.shape, (sc.shape, base.shape)
        st['molecules'] += 1
        st['pf_used'] += int(not np.array_equal(sc, base, equal_nan=True))
        st['order_changed'] += int(not np.array_equal(np.argsort(-base, kind='stable'), np.argsort(-sc, kind='stable')))
    return st


def iso_apply(sub_path, nf_path, out_path, pf_path, train_path, sample_ids, alpha, lib_skip=0.9, keep_train=True,
              scope='formula', report_path=None, skip=None, processed_only=True, formula_of=iso_formula,
              key_of=iso_key14, dryrun=False, cos_w=0.0, freeze_leader=False):
    """Rerank the answer file sub_path in place (all inputs are arguments; no notebook globals). skip: a status string
    that leaves the file untouched (e.g. 'skipped_fast_commit'). processed_only: rerank only the molecules the pipeline
    processed (neural_candidate_features.processed; the 12 visible molecules in the smoke commit, all in the rerun).
    dryrun: also run iso_dryrun (library skip off, scratch file only; report key 'dryrun'). cos_w (ISO_COS_W >= 0): weight
    of z(pf_cos) next to z(pf_fz), pf_cos from pf_path (pf_ok rows) only when cos_w > 0. freeze_leader
    (ISO_FREEZE_LEADER, bool): rank 1 never moves.
    Never fatal: on any exception the input bytes are written back (WARNING) and the outcome counters are reported as 0
    (the aborted attempt's as attempted_*). Returns the report (also report_path)."""
    t0 = time.time()
    sub_path = Path(sub_path)
    rep = dict(status='off', alpha=alpha, scope=scope, cos_w=cos_w, freeze_leader=freeze_leader, lib_skip=lib_skip,
               keep_train=keep_train, considered=0, skipped_lib=0, skipped_pf=0, skipped_cos=0, skipped_short=0,
               reranked=0, changed_rows=0, top1_changed=0, protected=0)
    before = sub_path.read_bytes()
    rep['input_sha256'] = hashlib.sha256(before).hexdigest()
    block = None
    try:
        if skip:
            rep['status'] = str(skip)
        elif float(alpha) != 0.0:
            alpha = float(alpha)
            block = rdBase.BlockLogs()                          # no RDKit parse warnings in the log
            assert math.isfinite(alpha) and alpha > 0, f'ISO_ALPHA {alpha!r}'
            assert scope in ('formula', 'all'), f'ISO_SCOPE {scope!r}'
            assert not isinstance(cos_w, bool) and math.isfinite(float(cos_w)) and float(cos_w) >= 0, f'ISO_COS_W {cos_w!r}'
            assert isinstance(freeze_leader, bool), f'ISO_FREEZE_LEADER {freeze_leader!r}'
            cos_w = float(cos_w)
            sub = pd.read_csv(io.BytesIO(before), dtype=str, keep_default_na=False)
            iso_check(sub, sub, sample_ids)
            nf = pd.read_parquet(nf_path)
            mids = nf.molecule_id.astype(str)
            lib = pd.to_numeric(nf.library_max, errors='coerce').groupby(mids.to_numpy()).max()
            lib_max = {str(m): float(v) for m, v in lib.items()}
            considered = set(mids[nf.processed.eq(True).to_numpy()]) if processed_only else None
            frames = [nf] + [pd.read_parquet(p) if p is not None and Path(p).is_file() else None for p in (out_path, pf_path)]
            pf_of = iso_pf_lookup(frames)
            cos_of = iso_pf_lookup(frames[2:], 'pf_cos') if cos_w else None     # pf_scores.parquet only
            train = iso_train_keys(train_path) if keep_train else frozenset()
            lists = [(m, s.split(';')) for m, s in zip(sub.molecule_id, sub.smiles)]
            new_lists, st = iso_rerank(lists, pf_of, lib_max, alpha, lib_skip, scope, train, keep_train, considered,
                                       formula_of, key_of, cos_of, cos_w, freeze_leader)
            new = iso_frame(new_lists)
            iso_check(sub, new, sample_ids)
            rep.update(st, status='applied', train_keys=len(train), pf_keys=len(pf_of))
            if cos_w:
                rep['cos_keys'] = len(cos_of)
            if dryrun:
                rep['dryrun'] = iso_dryrun(sub, lists, sub_path, sample_ids, pf_of, lib_max, alpha, scope, train, keep_train,
                                           considered, formula_of, key_of, cos_of, cos_w, freeze_leader)
            if st['changed_rows']:
                shutil.copyfile(sub_path, sub_path.with_name(sub_path.stem + '_pre_iso.csv'))
                tmp = sub_path.with_name(sub_path.name + '.iso.tmp')
                iso_write_checked(new, tmp)
                os.replace(tmp, sub_path)
    except Exception as exc:
        traceback.print_exc()
        rep.update({'attempted_' + k: rep.get(k, 0) for k in ISO_OUTCOME})   # nothing was applied: outcome counters 0
        rep.update(dict.fromkeys(ISO_OUTCOME, 0), status='fallback', error=f'{type(exc).__name__}: {str(exc)[:400]}')
        try:
            if sub_path.read_bytes() != before:
                tmp = sub_path.with_name(sub_path.name + '.iso.tmp')
                tmp.write_bytes(before)
                os.replace(tmp, sub_path)
            rep['restored'] = sub_path.read_bytes() == before
        except Exception as exc2:
            rep['restore_error'] = repr(exc2)[:300]
        print(f'WARNING ISO failed, pre-ISO submission.csv kept: {rep["error"]}', flush=True)
    finally:
        block = None                                            # RDKit logging back on
    rep['output_sha256'] = hashlib.sha256(sub_path.read_bytes()).hexdigest()
    rep['seconds'] = round(time.time() - t0, 1)
    if report_path is not None:
        try:
            Path(report_path).write_text(json.dumps(rep, indent=2, default=str))
        except Exception:
            traceback.print_exc()
    print(iso_summary(rep), flush=True)
    return rep


# ---- run (everything above is definitions) ----
_iso_sub = ROOT / 'submission.csv'
_iso_before = _iso_sub.read_bytes()
_iso_fast = True                                                # until the guards below are read
try:
    _iso_fast = not PIPELINE_DONE and not IS_RERUN and COMMIT_MODE == 'fast'
    if _iso_fast:
        _iso_skip = 'skipped_fast_commit'
    elif float(ISO_ALPHA) != 0.0 and time.time() - NB_T0 > 8.7 * 3600:
        _iso_skip = 'skipped_time_guard'
    else:
        _iso_skip = None
    ISO_REPORT = iso_apply(_iso_sub, ROOT / 'neural_candidate_features.parquet', ROOT / 'outsider_neural_features.parquet',
                           ROOT / 'pf_scores.parquet', COMP / 'train.parquet', SAMPLE_IDS, ISO_ALPHA, ISO_LIB_SKIP,
                           ISO_KEEP_TRAIN, ISO_SCOPE, ROOT / 'iso_report.json', skip=_iso_skip, dryrun=not IS_RERUN,
                           cos_w=ISO_COS_W, freeze_leader=ISO_FREEZE_LEADER)
except Exception as _iso_exc:                                   # iso_apply is fail-safe itself; this guards the call
    traceback.print_exc()
    ISO_REPORT = dict(status='fallback', alpha=globals().get('ISO_ALPHA'), scope=globals().get('ISO_SCOPE'),
                      cos_w=globals().get('ISO_COS_W'), freeze_leader=globals().get('ISO_FREEZE_LEADER'),
                      error=f'wrapper {type(_iso_exc).__name__}: {str(_iso_exc)[:300]}')
    if _iso_sub.read_bytes() != _iso_before:
        _iso_sub.with_name('submission.csv.iso.tmp').write_bytes(_iso_before)
        os.replace(_iso_sub.with_name('submission.csv.iso.tmp'), _iso_sub)
    ISO_REPORT['restored'] = _iso_sub.read_bytes() == _iso_before
    print(f'WARNING ISO wrapper error, pre-ISO submission.csv kept: {ISO_REPORT["error"]}', flush=True)
    print(iso_summary(ISO_REPORT), flush=True)
    try:
        (ROOT / 'iso_report.json').write_text(json.dumps(ISO_REPORT, indent=2, default=str))
    except Exception:
        traceback.print_exc()

try:
    _iso_mid_on = float(globals().get('PF_MID_W', 0.0) or 0.0) > 0 and not _iso_fast
except Exception:
    _iso_mid_on = False
if _iso_mid_on:                                                 # diagnostic only; reads, never writes answers
    try:
        PF_MID_TRACE = dict(w=PF_MID_W, **iso_pf_mid_trace(pd.read_parquet(ROOT / 'neural_candidate_features.parquet'),
                                                           globals()['_mid_neural_score']))
        print(f"PF_MID trace | w {PF_MID_W} | molecules {PF_MID_TRACE['molecules']} | PF used "
              f"{PF_MID_TRACE['pf_used']} | neural order changed {PF_MID_TRACE['order_changed']}", flush=True)
    except Exception as _iso_exc:
        PF_MID_TRACE = dict(w=PF_MID_W, error=f'{type(_iso_exc).__name__}: {str(_iso_exc)[:300]}')
        print(f"WARNING PF_MID trace failed: {PF_MID_TRACE['error']}", flush=True)
    try:
        (ROOT / 'pf_mid_trace.json').write_text(json.dumps(PF_MID_TRACE, indent=2, default=str))
    except Exception:
        traceback.print_exc()

final = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
assert list(final.columns) == ['molecule_id', 'smiles']
assert final.molecule_id.tolist() == SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.str.split(';').map(len).between(1, 25).all()
assert final.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a)) == len(a)).all()
print('SUBMISSION READY:', ROOT / 'submission.csv', 'molecules', len(final), 'ISO', json.dumps(
    {k: ISO_REPORT.get(k) for k in ('status', 'alpha', 'scope', 'cos_w', 'freeze_leader', 'considered', 'reranked',
                                    'changed_rows', 'top1_changed', 'protected', 'output_sha256', 'error')
     if k in ISO_REPORT}, default=str))


## 4. What this run did

The table counts, layer by layer, how many answer lists changed against the layer before. In the commit run the
pipeline ranks only the first 12 visible molecules (the rest are placeholders), and the visible molecules are copies
of training spectra, so every leader has a library match of 1.0 and the fragment check's rank-1 rule cannot fire;
only its swaps can. The neural fusion can still move a leader (the V1 row). The last row is the merge: the 0.75 order with the fragment check's leader moves from the V2 order; with no leader move in the commit run it equals the 0.75 order. The hidden rerun that produced the
score runs every molecule and is not shown to anyone.
In this version the last row also holds the final layer's documentation prior, deduplication and outside-candidate fill
(cell 16) and, when ISO_ALPHA > 0, the ISO reranker (cell 17); the MERGE and ISO log lines (merge_report.json, iso_report.json)
give their own counts.

In [ ]:
# What each layer did in this run, read from the files the stages wrote.
if SHOW:
    rep = {}
    for name in ('spectral_parallax_report.json', 'provenance_tournament_report.json', 'merge_report.json'):
        p = ROOT / name
        if p.is_file():
            try:
                rep[name] = json.loads(p.read_text())
            except Exception as exc:  # noqa: BLE001 - a report is only shown, never used
                rep[name] = {'unreadable': repr(exc)}
    base = pd.read_csv(BASELINE, dtype=str, keep_default_na=False) if BASELINE.is_file() else None
    final = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
    layers = [('0.465 pipeline', BASELINE), ('V1, neural 0.25', VARIANTS_DIR / 'submission_V1.csv'),
              ('V2, ranks 2-25 at 0.50', VARIANTS_DIR / 'submission_V2.csv'),
              (f'{VARIANT}, ranks 2-25 at {CUSTOM["w_mid"]:.2f}', VARIANTS_DIR / f'submission_{VARIANT}.csv')]
    if globals().get('FRAG_ON_V2'):
        layers += [('submitted: CUSTOM + the fragment check\'s leader moves on the V2 order', ROOT / 'submission.csv')]
    elif PROVENANCE:
        layers += [('after the fragment check', ROOT / 'submission_physical0472_control.csv'),
                   ('after the provenance reranker (submitted)', ROOT / 'submission.csv')]
    else:
        layers += [('after the fragment check (submitted)' if PHYSICAL else 'submitted', ROOT / 'submission.csv')]
    prev, out = None, []
    for label, path in layers:
        if not path.is_file():
            continue
        f = pd.read_csv(path, dtype=str, keep_default_na=False).set_index('molecule_id').smiles
        if prev is not None:
            common = f.index.intersection(prev.index)
            moved = int((f[common] != prev[common]).sum())
            top1 = int(sum(a.split(';')[0] != b.split(';')[0] for a, b in zip(f[common], prev[common])))
            out.append({'layer': label, 'lists re-ordered vs previous': moved, 'rank 1 changed': top1})
        else:
            out.append({'layer': label, 'lists re-ordered vs previous': '-', 'rank 1 changed': '-'})
        prev = f
    table_html(pd.DataFrame(out).set_index('layer', drop=False), numeric=('lists re-ordered vs previous', 'rank 1 changed'))
    print('Commit run: the counts cover the 12 visible smoke molecules; the hidden scoring rerun is not shown to anyone.')
    for k, v in rep.items():
        print(k, json.dumps(v)[:600])

## 5. Every submission of this pipeline

Each row is a kernel on our account (busyaprime/33-casmi26-px-e1, e2, e3, e4, e5, e6, m1) running this pipeline on the hidden
test, submitted on 2026-10-27 and 2026-10-28 (our four earlier submissions, 2026-09-21, ran a different pipeline).
Besides the switches, e1-e6 carry one patch this version does not: their fragment check reads the 0.75 file
(variants/submission_CUSTOM.csv) instead of the V2 file, so it runs on the 0.75 order. This notebook is a new
version of m1, so the score on this page is m1's. e3-e5 point the neural bundle pin at Ahmed Berat Özer's long-trained `final_model.pt`; e6 adds his NX isomer reranker; m1 runs the fragment check on the V2 order.

| run | ranks 2-25 weight | fragment check | provenance reranker | neural model | other | public LB | submission |
|---|---:|---|---|---|---|---:|---|
| e1 | 0.75 | on, 0.75 order | off | Saner Turhaner v1 |  | 0.472 | 57713785 |
| e2 | 0.75 | on, 0.75 order | on | Saner Turhaner v1 |  | 0.472 | 57713786 |
| e3 | 0.75 | on, 0.75 order | off | long (Ahmed Berat Özer) |  | 0.453 | 57714284 |
| e4 | 0.75 | off | off | long (Ahmed Berat Özer) |  | 0.461 | 57715407 |
| e5 | 0.75 | on, 0.75 order | on | long (Ahmed Berat Özer) |  | 0.453 | 57715244 |
| e6 | 0.75 | on, 0.75 order | off | Saner Turhaner v1 | NX isomer reranker, weight 2 | 0.414 | 57720847 |
| m1 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only | **0.477** | 57722897 |

How to read a difference: the leaderboard page says the public board uses about 31% of the test data, and
participants estimate the public split at about 125-130 molecules ([Roman Rozen](https://www.kaggle.com/code/romanrozen/0-468-casmi-from-peaks-to-structures)), so one molecule
moved from rank 2 to rank 1 is worth about 0.004 and the board prints three decimals. This notebook's 0.477 is +0.005 against 0.472, the score Spectral Parallax shows and the score a fork running alex chilton's 0.75 weights shows (his own page shows 0.467): the size of about one net leader move from rank 2 to rank 1. The hidden lists are not shown, so how many leaders actually moved is not known.

A full scoring run takes hours. The only public timings are for the earlier E007 pipeline (0.440) that our base grew
from: about 2 h 38 min and, for a later version, 2 h 44 min, each a full run over the 400 visible molecules
([Kino Lutor](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/discussion/749251)).

## 6. Run it yourself

Join the competition, then Copy & Edit this notebook (the competition data and the datasets
attached to this build, listed in section 8, stay attached). Keep the accelerator on GPU T4 and internet off. Run All in the editor is a smoke test: it ranks 12
visible molecules and gives no score. For a score, Save Version (our commit runs took 21-30 minutes on a T4), then
submit that version; the scoring rerun on the hidden test takes hours.

| switch | this notebook | what it does |
|---|---|---|
| `VARIANT`, `CUSTOM` | `CUSTOM`, w_top 0.25, w_mid 0.75 | the leader-safe weights; `V2` is the 0.25 / 0.50 recipe |
| `PHYSICAL` | `True` | Roman Tamrazov's fragment check |
| `PROVENANCE` | `False` | his provenance-aware reranker |
| `FRAG_ON_V2` | `True` | keep the fragment check's leader moves from the V2 order on top of the 0.75 order; False leaves the check's own V2 output |
| `COMMIT_MODE` | `'smoke'` | the commit run ranks 12 visible molecules; the scoring rerun ranks all |

Our string patches to his code, each asserted by our builder to apply exactly once:

1. Configuration cell: `VARIANT = 'CUSTOM'`, the CUSTOM weights, and the `PHYSICAL`, `PROVENANCE` and `FRAG_ON_V2`
   switches.
2. Fragment-check source (hidden): unchanged. It reads the V2 file, as upstream does.
3. Run cell (`BUNDLE = materialise_sources()`): it accepts the configured variant instead of only V2, and in a fast
   commit it writes the sample file and stops.
4. Fragment-check cell and provenance cell: an `if not PHYSICAL` and an `if not PROVENANCE` gate.
5. Our merge cell, after the provenance cell: per molecule, if the fragment check moved the V2 leader, the answer is
   [its new leader, the V2 leader] followed by the 0.75 order; otherwise the 0.75 order unchanged. It applies only
   when V2 and the 0.75 order share the leader and the fragment check's output holds the same candidates as the 0.75
   order; a list where the check also put an outside candidate at rank 25 keeps the 0.75 order without its leader
   move. Any error restores the 0.75 order.

## 7. What we concluded

1. The fragment check's leader moves from the V2 order on top of the 0.75 order (m1): 0.477, against 0.472 for the 0.75 order alone (its forks) and 0.472 for both changes stacked the plain way (e1 0.472).
2. The provenance reranker on top of the plain stack: e1 0.472 -> e2 0.472 (+0.000, no change at three decimals).
3. The fragment check run on the long-trained model's order: e4 0.461 -> e3 0.453 (-0.008, a loss at three decimals). This fits a check that depends on the order it is handed, but the pair switches all three parts of the check and does not single out the rank-1 rule.
4. Ahmed Berat Özer's long-trained neural model instead of Saner Turhaner's: e1 0.472 -> e3 0.453 (-0.019, a loss at three decimals). Its offline diagnostic (MRR 0.484 to 0.718 on sanpier's random holdout) did not carry to the hidden test.
5. Ahmed Berat Özer's NX isomer reranker at weight 2 on the plain stack: e1 0.472 -> e6 0.414 (-0.058, a loss at three decimals).
6. Rank 1 is where this lineage gains and loses. The public runs that let the neural model move the leader more freely scored lower: rank-1 weight 0.50 shows 0.448 and 1.00 shows 0.359 on the 0.472 base, and a uniform weight of 0.35 at offset 3 shows 0.447 on two accounts.

## 8. Credits

The answer lists come from other people's code, run with the patches listed in section 6:

- **Roman Tamrazov**, [Spectral Parallax](https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax), version 22: the notebook this one runs, including the
  fragment check and the provenance reranker. His notebook thanks Kozykorpesh's
  [LB [0.468] balanced acquisition evidence](https://www.kaggle.com/code/kozykappa/lb-0-468-balanced-acquisition-evidence) first, for the reproducible V2 consensus pipeline, the
  neural spectrum-structure ranking, the candidate-evidence exports and the evidence-guided candidate recovery, and
  credits Heliosli's [training-structure protection](https://www.kaggle.com/code/heliosli/happy-to-make-the-dataset-public-if-it-helps) for the provenance idea.
- **Roman Rozen**, [From Peaks to Structures](https://www.kaggle.com/code/romanrozen/0-468-casmi-from-peaks-to-structures): the leader-safe consensus (`consensus.py`) and the
  outside-candidate collection (`outsiders.py`).
- **alex chilton**, [Leader-Safe, Ranks 2-25 at 0.75](https://www.kaggle.com/code/alexchilton/casmi26-lb-0-472-leader-safe-ranks-2-25-at-0-75): the 0.75 weight on ranks 2-25.
- **Saner Turhaner**, [PubChem + Neural025](https://www.kaggle.com/code/sanpier/pubchem-neural025-lb-0-466): the spectrum-candidate model, the reciprocal-rank
  fusion and the six helper modules.
- **Dmitrii Gluzdov**, [The Data Behind Spectra to Structures](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures): the data notebook that assembles
  and documents the shared 0.465 pipeline, and the datasets that carry it.
- **Huseyin Emre Aksoy** ([profile](https://www.kaggle.com/huseyinemreaksoy)): `frag_rescore.py`, the untrained
  MetFrag-lite fragment scorer that both the fragment check and the pipeline's fragment-gap score run (Apache-2.0,
  reached through Ahmed Berat Özer's v7c notebook).
- **Ahmed Berat Özer**, [the long-schedule spectrum model](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-spectrum-model-long): Saner Turhaner's model trained 40 x 2,000 steps instead of 16 x 500, same code; runs e3-e5 use it.

The embedded 0.465 pipeline keeps its own full credits and licence table (its section 6), unchanged. Its main
components: canqiang's E006 (the complete inference base); Ahmed Berat Özer's main engine, rankers, fingerprint
weights, candidate pool and the Kaggle packages of ICEBERG and GLACIER (the ms-pred models of the Coley group);
prvsiyan, megayak and seyitkaangunes (second engine, analogue propagation, engine fusion); flexonafft (the N1
challenger); hengck23's DreaMS resource; and the further authors and public spectral libraries listed there.

In this build the second engine does not run (fix F2a; its lists are the pipeline's own), hengck23's DreaMS resource and the
knowledge table are not attached, and the external reference spectra are not added (F4).

**Datasets attached to this build** (kernel-metadata.json; licence as the Kaggle page shows it, with the component
licences of docs/fork_eligibility.md):

| dataset | author | used for | licence |
|---|---|---|---|
| [ahmedberatozer/casmi26-v4b-models](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v4b-models) | Ahmed Berat Özer | Engine 1: the casmi package, the fe_v4 feature families, FPNet A / B, fe_D and ranker_v4b | Kaggle "other"; code MIT |
| [ahmedberatozer/casmi26-v3-models](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v3-models) | Ahmed Berat Özer | PubChem channel: the v3 casmi code and FPNet A / B | Kaggle "other"; code MIT |
| [ahmedberatozer/casmi26-fpnet-full1](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-fpnet-full1) | Ahmed Berat Özer | Engine 1's retrieval fingerprint model (fpnet_full1.pt) | Kaggle "other"; training code MIT |
| [ahmedberatozer/casmi26-v2-pool](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v2-pool) | Ahmed Berat Özer | the candidate pool (train + COCONUT, 710,701 structures) and its fingerprints | Kaggle "other" |
| [ahmedberatozer/casmi26-iceberg](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-iceberg) | Ahmed Berat Özer | ICEBERG weights (ms-pred, Coley group), simulator wheels and runner glue | Kaggle "other"; weights MIT; wheels BSD-3 / Apache-2.0 / MIT; glue without an OSI licence (F5a) |
| [ahmedberatozer/casmi26-glacier](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-glacier) | Ahmed Berat Özer | GLACIER weights (ms-pred, Coley group) and runner glue | Kaggle "other"; weights MIT; glue without an OSI licence (F5a) |
| [dmitriigluzdov/casmi26-pubchem-popularity-prior](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior) | Dmitrii Gluzdov | PubChem tier and popularity prior (its depositor-source flags are not used here: SRC_BITS 0, SPEC_W 0) | Other: PubChem courtesy of the U.S. NLM; LOTUS CC0; NPAtlas CC BY-NC 4.0 (NPAtlas is on the host's list) |
| [dmitriigluzdov/casmi26-fold-safe-fpnet](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet) | Dmitrii Gluzdov | fold-safe FPNet checkpoint of the PubChem channel | Other: checkpoint under competition data terms; scripts Apache 2.0 |
| [sanpier/casmi-spectrum-model-v1](https://www.kaggle.com/datasets/sanpier/casmi-spectrum-model-v1) | Saner Turhaner | the spectrum-candidate model | code Apache-2.0; trained on competition data (CC BY-NC 4.0) |
| [metric/rdkit-2026-3-3-wheel](https://www.kaggle.com/datasets/metric/rdkit-2026-3-3-wheel) | Kaggle (metric) | RDKit 2026.03.3, the metric's canonicalisation | RDKit BSD-3 (Kaggle "unknown") |

**Upstream lineage, not attached here.** The forked notebook was built on these datasets; elig0 neither mounts nor reads
them (fix F1). They are listed for the lineage's attribution only:

| dataset | author | used for (forked notebook) | licence |
|---|---|---|---|
| [dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets) | Dmitrii Gluzdov (a collection of other authors' public files, each folder with its own notices) | models, candidate sets and engine code of the pipeline | CC BY-NC-SA 4.0; each folder keeps its source licence (`ATTRIBUTION.md`) |
| [dmitriigluzdov/casmi26-natural-product-panel-benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark) | Dmitrii Gluzdov | natural-product panel | Other: competition data terms |
| [dmitriigluzdov/casmi26-natural-product-knowledge-table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table) | Dmitrii Gluzdov | natural-product knowledge table | Other: LOTUS CC BY 4.0, NPAtlas CC BY-NC 4.0, PubChem counts |
| [sanpier/catalogue-experiment-saner](https://www.kaggle.com/datasets/sanpier/catalogue-experiment-saner) | Saner Turhaner | attached as in Spectral Parallax; this run does not read it | as on its page |

The competition data is CC BY-NC 4.0, and the models above are trained on it: use them non-commercially, with
attribution. We wrote the field record, the figure and the audit cells. In this build we also wrote the eligibility code of cell 8
(mount and hash check, the null Engine 2 module).